# Data Science Case Study: Candidate Resume Search Platform

## Business Context & Objective

### Background
You are joining the Business Development team at Millennium which is a global hedge fund that manages assets across multiple investment strategies (fundamental equity, systematic trading, credit, etc.). The BD team is responsible for sourcing junior analyst talent across different:

- **Geographic Markets**: US, Europe, Asia-Pacific
- **Investment Approaches**: Fundamental vs. Systematic/Quantitative strategies
- **Sectors**: Technology, Healthcare, Financial Services, Energy, Industrials, Consumer, Credit, Macro, etc.
- **Experience Levels**: depending on the job requisitions

### Goal
Build a **searchable platform** to quickly identify candidates based on specific criteria based on job requisitions.

### Your Task
1. Parse resume data from PDF/Word documents using **LLM models via API**
2. Create parsed resume data as **JSON, CSV, etc.** for further analysis
3. Create a **Streamlit web application** where BD users can search and filter candidates using multiple criteria
   (Based on the background provided above, you will come up with relevant filters)
4. Visualize candidate distributions and insights
5. **Design for scalability** to handle large volumes of resumes

### Output
1. Code for data parsing and Streamlit in **this Jupyter notebook** for ease of review
2. Include the **link to Streamlit app** in the notebook
3. **JSON/CSV exports** of parsed resume data
4. Discussion of additional features and implementation approach if more time was available
---

## Sample Resume Data

**You have access to 10 made up resume files representing different candidate profiles**

---

## Technical Requirements & Evaluation
  
### Evaluation Criteria
1. **Technical Implementation (40%)**: Code quality, data processing, performance
2. **User Experience (30%)**: Interface design, functionality, responsiveness
3. **Business Value (20%)**: Feature completeness, search capabilities, insights
4. **Documentation (10%)**: Code comments, README, presentation

### Time Recommendation
- **Total Estimated Time**: ~9 hours (**tight but achievable with focused approach**)
- **Recommended Strategy**: Build core MVP in 6-7 hours, then enhance

### Phased Approach (Recommended)
**Phase 1 (3 hours): Core Parsing & Data**
- Set up LLM API integration (OpenAI, Anthropic, or alternatives)
- Parse 2-3 sample resumes using LLM
- Create structured data format with JSON, etc.
- Build basic data validation and quality checks
 
**Phase 2 (3 hours): Streamlit**  
- Basic filter interface
- Simple results display
- Core visualizations

**Phase 3 (2-3 hours): Enhancement**
- Parse remaining resumes
- Add advanced filters
- Polish UI/UX

### Success Criteria (MVP)
- LLM-powered parsing of at least 5-7 resume files with high accuracy
- Working Streamlit app with basic search functionality
- At least 3-4 meaningful filters working
- A few visualizations showing candidate insights
- JSON/CSV output of parsed resume data
- Clean, documented code


---

# Candidate resume search platform: solution

## How to read this notebook

The code does not live in this notebook. It lives in `src/resume_search/` (the package) and `app/` (the Streamlit app). The cells in the "Code" sections below write those files out with `%%writefile`, one cell per file, generated from the repository by `scripts/build_notebook.py`. Run the notebook top to bottom and the package is recreated on disk.

The sections run in this order.

1. Architecture overview and a table of modules.
2. Code, one cell per file: taxonomy, project configuration, package, Streamlit app, evaluation requisitions.
3. Install the package.
4. Demos that run without an API key: extraction, a stored record, the candidate table, searches, insights, eval metrics.
5. Two demos that call the Anthropic API. They are off by default and run only when `RUN_LLM=1`.
6. Results, then scalability and additional features, then the repository layout and how to run everything.

**Streamlit app:** run locally with `uv run streamlit run app/Home.py`; public URL: _to be added at submission_

**Exports:** `exports/candidates.json` (full records) and `exports/candidates.csv` (the `candidates` table plus facet columns). Both are rewritten at the end of every ingest.

**Requirements.** Python 3.12 and [uv](https://docs.astral.sh/uv/). Parsing, search, and the evals call the Anthropic API and read `ANTHROPIC_API_KEY` from the environment or from a `.env` file. The demos in this notebook need no key.

## Architecture overview

A resume goes through five stages. Each stage reads the output of the one before it and nothing else, so a stage can be re-run alone.

```
 data/*.docx, *.pdf
        |
        v   extract    python-docx and PyMuPDF; ligature repair; unknown-word ratio;
        |              vision trigger (not hit by any of the 10 samples)
        v
 ExtractedText
        |
        v   parse      3 Sonnet calls: profile || experience, then assessment
        |              cross-check against deterministic text parsing, targeted regeneration
        |              retry ladder: Sonnet -> Sonnet + validation errors -> Opus
        v
 ParsedResume          (literal, evidence-quoted fields)
        |
        v   derive     pure code: months of experience, seniority band, countries,
        |              regions, normalized skills, CFA status. No API call.
        v
 SQLite                candidates + facet tables + FTS5 body + float32 embeddings
        |              + parse_jobs (ingest state) + taxonomy mirrors
        v
 search                requisition -> prune (must criteria, SQL)
        |                          -> rule score (preferred criteria)
        |                          -> BM25 (FTS5) and embedding cosine
        |                          -> final = 0.5 rule + 0.2 BM25 + 0.3 embedding
        v
 Streamlit             Search, Candidate, Insights, Ingest, Evals
```

Three layers describe every record. `extracted` holds what the resume says, with a verbatim evidence quote for every judgement. `derived` holds values computed in code from `extracted` and the taxonomy. The `search` layer is the index: the FTS5 body and the embeddings. The full design record, with the decisions and their reasons, is in `docs/design.md`.

### Modules

| Module | What it does |
|---|---|
| `cli.py` | The `resume-search` command line: extract, parse, ingest, rederive, show, search, and eval. |
| `derive/__init__.py` | `derive()` builds the `derived` layer from a parsed resume and the taxonomy, in pure code. |
| `derive/experience.py` | Month arithmetic over roles: merges overlapping ranges, falls back to stated durations, and maps years to a seniority band. |
| `derive/location.py` | Derives the current country, the countries worked in, and the markets covered. |
| `derive/models.py` | The `DerivedFields` and `CandidateRecord` data shapes and the schema version. |
| `derive/skills.py` | Normalizes skills to canonical tokens and decides CFA status. |
| `eval/__init__.py` | Package marker for the evaluation harness. |
| `eval/judge.py` | The Opus judge prompts and calls for the parsing and ranking evals, and their cost accounting. |
| `eval/metrics.py` | Pure scoring functions: per-field accuracy, nDCG@k, precision@k, and judge-human agreement. |
| `eval/models.py` | Data shapes for judge output and evaluation requisitions. |
| `eval/parsing_eval.py` | Runs the parsing suite: judges every stored record against its resume text and compares with the hand grades. |
| `eval/ranking_eval.py` | Runs the ranking suite: ranks each requisition with four score variants and grades the pool once. |
| `eval/report.py` | Formats suite results as plain-text tables. |
| `eval/results.py` | Reads and writes the timestamped result files in `evals/results/`. |
| `extract/__init__.py` | `extract(path)` turns a DOCX or PDF into an `ExtractedText` with repaired text, quality numbers, and the extraction path. |
| `extract/docx.py` | Reads a DOCX body in document order and writes each table row as one line with cells joined by ` | `. |
| `extract/pdf.py` | Reads PDF text per page with PyMuPDF, measures image coverage, and renders pages to PNG for the vision path. |
| `extract/quality.py` | Computes the share of unknown words in a text and fixes close misspellings caused by bad glyphs. |
| `extract/repair.py` | Replaces ligature glyphs that PDF fonts emit (such as `Ɵ` for `ti`) and drops private-use characters. |
| `extract/types.py` | The `ExtractedText` and `TextQuality` data shapes, including the rule that decides when the vision path is needed. |
| `ingest/__init__.py` | Package marker for the ingest pipeline. |
| `ingest/jobs.py` | The `parse_jobs` row and its state machine, and the idempotency check on file hash and prompt version. |
| `ingest/pipeline.py` | `ingest()` runs extract, parse, derive, and index for many files with worker threads, and `rederive()` rebuilds derived data without the API. |
| `parse/__init__.py` | Public surface of the parser: `parse_resume`, `ParseOk`, `ParseFailed`, and `ParsedResume`. |
| `parse/client.py` | Runs the profile, experience, and assessment calls with constrained decoding, cross-verification, targeted regeneration, and the Sonnet-Sonnet-Opus retry ladder. |
| `parse/models.py` | The Pydantic models for LLM output, with validators for dates, ISO-2 countries, and verbatim evidence quotes. |
| `parse/prompt.py` | The versioned system prompts and the user-message builders, rendered from the taxonomy files. |
| `parse/verify.py` | A registry of checks that compare the LLM's fields against deterministic text parsing and name the field to regenerate on disagreement. |
| `search/__init__.py` | Public surface of the search package. |
| `search/keyword.py` | BM25 scoring over the FTS5 index and the list of query terms each candidate matched. |
| `search/llm.py` | A shared structured-output call with one retry, used by the requisition parser and the reranker. |
| `search/rank.py` | `search()` combines the rule, BM25, and embedding scores into the final order, and `rerank_llm` optionally reorders the top 10. |
| `search/requisition.py` | The `Requisition` object with must and preferred criteria, and the LLM step that turns a pasted JD into one. |
| `search/rules.py` | Prunes candidates by must criteria in SQL and scores preferred criteria, producing the explanation chips. |
| `search/vectors.py` | Embeds text with `BAAI/bge-small-en-v1.5` and scores candidates by cosine similarity over summary and role vectors. |
| `search/weights.py` | Loads rule weights and the blend weights from `taxonomy/weights.yaml`. |
| `store/db.py` | Opens the database, loads taxonomy tables, and writes or reads one candidate atomically. |
| `store/export.py` | Writes `candidates.json` and `candidates.csv` from the database. |
| `store/fts.py` | Builds the keyword-index body by removing email, URL, and phone lines from the resume text. |
| `store/schema.sql` | The SQLite DDL: candidate and facet tables, the FTS5 index, embeddings, `parse_jobs`, and taxonomy mirrors. |
| `taxonomy.py` | Loads the YAML files in `taxonomy/` into one plain-data view of countries, sectors, firm types, asset classes, and skill synonyms. |
| `app/Home.py` | Landing page with database counts and links to the other pages. |
| `app/common.py` | Shared Streamlit helpers: cached resources, the widget-to-`Requisition` mapping, and chip rendering. |
| `app/pages/1_Search.py` | Filter sidebar, JD paste box, must and preferred toggles, ranked result cards with chips, and optional LLM rerank. |
| `app/pages/2_Candidate.py` | One candidate in full: derived facets, role timeline, evidence quotes, skills, and parse metadata. |
| `app/pages/3_Insights.py` | Distributions of region, country, approach, sector, seniority, degree, skills, and firm type. |
| `app/pages/4_Ingest.py` | Multi-file upload that calls `ingest()` and shows live job status, throughput, and cost. |
| `app/pages/5_Evals.py` | Shows the latest parsing and ranking results and the judge-human disagreements. |

## Code

The cells in this part write the repository's source files. Each header names the file, and the sentence under it says what the file does.

In [1]:
from pathlib import Path

# %%writefile does not create directories, so make them first; empty package markers are touched.
for d in ['app', 'app/pages', 'evals', 'src/resume_search', 'src/resume_search/derive', 'src/resume_search/eval', 'src/resume_search/extract', 'src/resume_search/ingest', 'src/resume_search/parse', 'src/resume_search/search', 'src/resume_search/store', 'taxonomy']:
    Path(d).mkdir(parents=True, exist_ok=True)
for f in ['src/resume_search/__init__.py', 'src/resume_search/eval/__init__.py', 'src/resume_search/ingest/__init__.py', 'src/resume_search/store/__init__.py']:
    Path(f).touch()
print(f"ready: {len(['app', 'app/pages', 'evals', 'src/resume_search', 'src/resume_search/derive', 'src/resume_search/eval', 'src/resume_search/extract', 'src/resume_search/ingest', 'src/resume_search/parse', 'src/resume_search/search', 'src/resume_search/store', 'taxonomy'])} directories, {len(['src/resume_search/__init__.py', 'src/resume_search/eval/__init__.py', 'src/resume_search/ingest/__init__.py', 'src/resume_search/store/__init__.py'])} empty package markers")

ready: 12 directories, 4 empty package markers


### Taxonomy

#### `taxonomy/asset_classes.yaml`

In [2]:
%%writefile taxonomy/asset_classes.yaml
classes: [Equities, Fixed Income, Credit, FX, Commodities, Derivatives, Multi-asset]
synonyms:
  equity: Equities
  equities: Equities
  stocks: Equities
  long/short: Equities
  long short: Equities
  l/s: Equities
  rates: Fixed Income
  bonds: Fixed Income
  treasuries: Fixed Income
  fixed income: Fixed Income
  high yield: Credit
  hy: Credit
  ig: Credit
  leveraged loans: Credit
  structured credit: Credit
  distressed: Credit
  foreign exchange: FX
  currencies: FX
  oil: Commodities
  metals: Commodities
  options: Derivatives
  swaps: Derivatives
  exotics: Derivatives
  structured products: Derivatives
  multi-asset: Multi-asset
  macro: Multi-asset

Overwriting taxonomy/asset_classes.yaml


#### `taxonomy/countries.yaml`

In [3]:
%%writefile taxonomy/countries.yaml
# ISO-2 code -> name, region, phone prefix, and location aliases (cities, states, demonyms)
# region is one of: US, Europe, APAC, Other
# The LLM outputs the ISO-2 code only; region is always derived from this table.
regions: [US, Europe, APAC, Other]
countries:
  US: {name: United States, region: US, phone: ["+1"], aliases: [USA, U.S., New York, NY, Greenwich, CT, Connecticut, Boston, MA, Chicago, IL, San Francisco, CA, California, Los Angeles, Houston, TX, Texas, Ann Arbor, MI, Michigan, Brooklyn, Philadelphia, PA, Washington DC, Seattle, Miami, Stamford]}
  CA: {name: Canada, region: Other, phone: ["+1"], aliases: [Toronto, Montreal, Vancouver]}
  GB: {name: United Kingdom, region: Europe, phone: ["+44"], aliases: [UK, U.K., England, London, Edinburgh, Manchester, Oxford, Cambridge]}
  FR: {name: France, region: Europe, phone: ["+33"], aliases: [Paris, Lyon]}
  DE: {name: Germany, region: Europe, phone: ["+49"], aliases: [Frankfurt, Berlin, Munich]}
  CH: {name: Switzerland, region: Europe, phone: ["+41"], aliases: [Zurich, Geneva]}
  NL: {name: Netherlands, region: Europe, phone: ["+31"], aliases: [Amsterdam]}
  IE: {name: Ireland, region: Europe, phone: ["+353"], aliases: [Dublin]}
  ES: {name: Spain, region: Europe, phone: ["+34"], aliases: [Madrid, Barcelona]}
  IT: {name: Italy, region: Europe, phone: ["+39"], aliases: [Milan, Rome]}
  SE: {name: Sweden, region: Europe, phone: ["+46"], aliases: [Stockholm]}
  LU: {name: Luxembourg, region: Europe, phone: ["+352"], aliases: []}
  PT: {name: Portugal, region: Europe, phone: ["+351"], aliases: [Lisbon]}
  PL: {name: Poland, region: Europe, phone: ["+48"], aliases: [Warsaw]}
  HK: {name: Hong Kong, region: APAC, phone: ["+852"], aliases: [Hong Kong SAR, HKSAR]}
  CN: {name: China, region: APAC, phone: ["+86"], aliases: [Shanghai, Beijing, Shenzhen, Guangzhou, Jiangxi, PRC, Mainland China]}
  SG: {name: Singapore, region: APAC, phone: ["+65"], aliases: []}
  JP: {name: Japan, region: APAC, phone: ["+81"], aliases: [Tokyo]}
  KR: {name: South Korea, region: APAC, phone: ["+82"], aliases: [Seoul, Korea]}
  TW: {name: Taiwan, region: APAC, phone: ["+886"], aliases: [Taipei]}
  IN: {name: India, region: APAC, phone: ["+91"], aliases: [Mumbai, Bombay, Delhi, New Delhi, Noida, Gurgaon, Gurugram, Bangalore, Bengaluru, Hyderabad, Chennai, Pune, Kolkata]}
  AU: {name: Australia, region: APAC, phone: ["+61"], aliases: [Sydney, Melbourne]}
  NZ: {name: New Zealand, region: APAC, phone: ["+64"], aliases: [Auckland]}
  MY: {name: Malaysia, region: APAC, phone: ["+60"], aliases: [Kuala Lumpur]}
  ID: {name: Indonesia, region: APAC, phone: ["+62"], aliases: [Jakarta]}
  TH: {name: Thailand, region: APAC, phone: ["+66"], aliases: [Bangkok]}
  VN: {name: Vietnam, region: APAC, phone: ["+84"], aliases: [Ho Chi Minh City, Hanoi]}
  PH: {name: Philippines, region: APAC, phone: ["+63"], aliases: [Manila]}
  AE: {name: United Arab Emirates, region: Other, phone: ["+971"], aliases: [UAE, Dubai, Abu Dhabi]}
  SA: {name: Saudi Arabia, region: Other, phone: ["+966"], aliases: [Riyadh]}
  QA: {name: Qatar, region: Other, phone: ["+974"], aliases: [Doha]}
  IL: {name: Israel, region: Other, phone: ["+972"], aliases: [Tel Aviv]}
  ZA: {name: South Africa, region: Other, phone: ["+27"], aliases: [Johannesburg, Cape Town]}
  MA: {name: Morocco, region: Other, phone: ["+212"], aliases: [Rabat, Casablanca]}
  EG: {name: Egypt, region: Other, phone: ["+20"], aliases: [Cairo]}
  NG: {name: Nigeria, region: Other, phone: ["+234"], aliases: [Lagos]}
  BR: {name: Brazil, region: Other, phone: ["+55"], aliases: [Sao Paulo, São Paulo, Rio de Janeiro]}
  MX: {name: Mexico, region: Other, phone: ["+52"], aliases: [Mexico City]}
  AR: {name: Argentina, region: Other, phone: ["+54"], aliases: [Buenos Aires]}
  CL: {name: Chile, region: Other, phone: ["+56"], aliases: [Santiago]}
  CO: {name: Colombia, region: Other, phone: ["+57"], aliases: [Bogota, Bogotá]}
  TR: {name: Turkey, region: Other, phone: ["+90"], aliases: [Istanbul]}
  RU: {name: Russia, region: Other, phone: ["+7"], aliases: [Moscow]}
# Multi-country market labels the LLM may use in markets_covered when the resume says so
market_labels:
  Greater China: [CN, HK, TW]
  Asia: APAC
  APAC: APAC
  Asia-Pacific: APAC
  Europe: Europe
  EMEA: Europe
  North America: [US, CA]
  LatAm: Other
  Latin America: Other
  Middle East: Other
  Global: null

Overwriting taxonomy/countries.yaml


#### `taxonomy/firm_types.yaml`

In [4]:
%%writefile taxonomy/firm_types.yaml
types: [Hedge fund, Asset manager, Sell-side bank, Sell-side broker, PE/VC, Consulting, Corporate, Academic, Other]
# Known firms the EntityRuler can tag; the LLM uses its own knowledge when a firm is absent here.
known_firms:
  Millennium Management: Hedge fund
  Millennium: Hedge fund
  Citadel: Hedge fund
  Point72: Hedge fund
  Bridgewater: Hedge fund
  Two Sigma: Hedge fund
  D.E. Shaw: Hedge fund
  Man Group: Hedge fund
  Balyasny: Hedge fund
  ExodusPoint: Hedge fund
  BlackRock: Asset manager
  Fidelity: Asset manager
  Vanguard: Asset manager
  Wellington: Asset manager
  T. Rowe Price: Asset manager
  PIMCO: Asset manager
  Goldman Sachs: Sell-side bank
  Morgan Stanley: Sell-side bank
  J.P. Morgan: Sell-side bank
  JPMorgan: Sell-side bank
  Bank of America: Sell-side bank
  Citi: Sell-side bank
  Barclays: Sell-side bank
  UBS: Sell-side bank
  Credit Suisse: Sell-side bank
  Deutsche Bank: Sell-side bank
  BNP Paribas: Sell-side bank
  Société Générale: Sell-side bank
  Societe Generale: Sell-side bank
  HSBC: Sell-side bank
  Bank of China: Sell-side bank
  ICICI Securities: Sell-side broker
  Kotak Securities: Sell-side broker
  Anand Rathi: Sell-side broker
  Leerink Partners: Sell-side broker
  Jefferies: Sell-side bank
  Evercore: Sell-side bank
  Lazard: Sell-side bank
  Blackstone: PE/VC
  KKR: PE/VC
  Carlyle: PE/VC
  Apollo: PE/VC
  TPG: PE/VC
  Sequoia: PE/VC
  McKinsey: Consulting
  Bain: Consulting
  BCG: Consulting
  Boston Consulting Group: Consulting
  Deloitte: Consulting
  Accenture: Consulting

Overwriting taxonomy/firm_types.yaml


#### `taxonomy/sectors.yaml`

In [5]:
%%writefile taxonomy/sectors.yaml
# Core sector enum from the brief. other_sector_tags holds narrower names; this map
# lets a tag be promoted into a core sector without a re-parse.
core: [Technology, Healthcare, Financials, Energy, Industrials, Consumer, Credit, Macro, Other]
tag_to_core:
  tmt: Technology
  technology media telecom: Technology
  software: Technology
  internet: Technology
  semiconductors: Technology
  fintech: Technology
  media: Technology
  telecom: Technology
  pharmaceuticals: Healthcare
  pharma: Healthcare
  biotech: Healthcare
  biotechnology: Healthcare
  medtech: Healthcare
  medical devices: Healthcare
  life sciences tools: Healthcare
  diagnostics: Healthcare
  healthcare services: Healthcare
  hospitals: Healthcare
  therapeutics: Healthcare
  digital health: Healthcare
  banks: Financials
  insurance: Financials
  asset managers: Financials
  real estate: Financials
  reits: Financials
  oil and gas: Energy
  oil & gas: Energy
  renewables: Energy
  renewable energy: Energy
  utilities: Energy
  specialty chemicals: Industrials
  chemicals: Industrials
  infrastructure: Industrials
  transportation: Industrials
  aerospace: Industrials
  defense: Industrials
  retail: Consumer
  consumer electronics: Consumer
  consumer internet: Consumer
  consumer staples: Consumer
  consumer discretionary: Consumer
  luxury: Consumer
  high yield: Credit
  leveraged loans: Credit
  structured credit: Credit
  distressed: Credit
  investment grade: Credit
  rates: Macro
  fx: Macro
  commodities: Macro
  sovereign: Macro

Overwriting taxonomy/sectors.yaml


#### `taxonomy/skills.yaml`

In [6]:
%%writefile taxonomy/skills.yaml
# variant (lowercase) -> {canonical, category}. Categories: programming, tools, domain, cert.
# derive/skills.py lowercases the LLM's skill strings, looks them up here, then falls back to
# rapidfuzz >= 90 against the variant keys, then keeps the lowercased string as its own token.
skills:
  python: {canonical: python, category: programming}
  py: {canonical: python, category: programming}
  r: {canonical: r, category: programming}
  sql: {canonical: sql, category: programming}
  c++: {canonical: cpp, category: programming}
  cpp: {canonical: cpp, category: programming}
  c#: {canonical: csharp, category: programming}
  c# / .net: {canonical: csharp, category: programming}
  .net: {canonical: csharp, category: programming}
  java: {canonical: java, category: programming}
  matlab: {canonical: matlab, category: programming}
  vba: {canonical: vba, category: programming}
  excel vba: {canonical: vba, category: programming}
  excel with vba: {canonical: vba, category: programming}
  scala: {canonical: scala, category: programming}
  julia: {canonical: julia, category: programming}
  sas: {canonical: sas, category: programming}
  stata: {canonical: stata, category: programming}
  javascript: {canonical: javascript, category: programming}
  bash: {canonical: bash, category: programming}
  excel: {canonical: excel, category: tools}
  microsoft excel: {canonical: excel, category: tools}
  powerpoint: {canonical: powerpoint, category: tools}
  bloomberg: {canonical: bloomberg, category: tools}
  bloomberg terminal: {canonical: bloomberg, category: tools}
  factset: {canonical: factset, category: tools}
  capital iq: {canonical: capital-iq, category: tools}
  capiq: {canonical: capital-iq, category: tools}
  s&p capital iq: {canonical: capital-iq, category: tools}
  refinitiv: {canonical: refinitiv, category: tools}
  eikon: {canonical: refinitiv, category: tools}
  wind: {canonical: wind, category: tools}
  wind database: {canonical: wind, category: tools}
  tableau: {canonical: tableau, category: tools}
  power bi: {canonical: power-bi, category: tools}
  git: {canonical: git, category: tools}
  mongodb: {canonical: mongodb, category: tools}
  pandas: {canonical: pandas, category: tools}
  numpy: {canonical: numpy, category: tools}
  scikit-learn: {canonical: scikit-learn, category: tools}
  sklearn: {canonical: scikit-learn, category: tools}
  pytorch: {canonical: pytorch, category: tools}
  tensorflow: {canonical: tensorflow, category: tools}
  dcf: {canonical: dcf, category: domain}
  discounted cash flow: {canonical: dcf, category: domain}
  lbo: {canonical: lbo, category: domain}
  financial modeling: {canonical: financial-modeling, category: domain}
  financial modelling: {canonical: financial-modeling, category: domain}
  three-statement: {canonical: financial-modeling, category: domain}
  valuation: {canonical: valuation, category: domain}
  relative valuation: {canonical: valuation, category: domain}
  comps: {canonical: valuation, category: domain}
  backtesting: {canonical: backtesting, category: domain}
  backtest: {canonical: backtesting, category: domain}
  factor models: {canonical: factor-models, category: domain}
  multi-factor: {canonical: factor-models, category: domain}
  multi-factor model: {canonical: factor-models, category: domain}
  alpha research: {canonical: alpha-research, category: domain}
  signal research: {canonical: alpha-research, category: domain}
  statistical arbitrage: {canonical: stat-arb, category: domain}
  stat arb: {canonical: stat-arb, category: domain}
  machine learning: {canonical: machine-learning, category: domain}
  ml: {canonical: machine-learning, category: domain}
  deep learning: {canonical: machine-learning, category: domain}
  nlp: {canonical: nlp, category: domain}
  time series: {canonical: time-series, category: domain}
  econometrics: {canonical: econometrics, category: domain}
  stochastic calculus: {canonical: stochastic-calculus, category: domain}
  monte carlo: {canonical: monte-carlo, category: domain}
  options pricing: {canonical: derivatives-pricing, category: domain}
  derivatives pricing: {canonical: derivatives-pricing, category: domain}
  pricing library: {canonical: derivatives-pricing, category: domain}
  greeks: {canonical: derivatives-pricing, category: domain}
  risk management: {canonical: risk-management, category: domain}
  portfolio construction: {canonical: portfolio-construction, category: domain}
  performance attribution: {canonical: performance-attribution, category: domain}
  equity research: {canonical: equity-research, category: domain}
  due diligence: {canonical: due-diligence, category: domain}
  expert interviews: {canonical: primary-research, category: domain}
  expert calls: {canonical: primary-research, category: domain}
  alternative data: {canonical: alternative-data, category: domain}
  earnings models: {canonical: financial-modeling, category: domain}
  cfa: {canonical: cfa, category: cert}
  chartered financial analyst: {canonical: cfa, category: cert}
  frm: {canonical: frm, category: cert}
  caia: {canonical: caia, category: cert}
  cpa: {canonical: cpa, category: cert}

Overwriting taxonomy/skills.yaml


#### `taxonomy/weights.yaml`

In [7]:
%%writefile taxonomy/weights.yaml
# Ranking weights; see docs/design.md section 8. Editable in the Search page advanced expander.
rule:
  approach: 3
  sector: 3
  location: 2
  markets_covered: 2
  years_in_range: 2
  skill_preferred: 1
  degree: 1
  certification: 1
  asset_class: 1
location:
  current_match: 1.0
  worked_match: 0.5
  region_only_factor: 0.5
blend:
  rule: 0.5
  keyword: 0.2
  embedding: 0.3
years_decay_outside_range: 2.0
fuzzy_skill_threshold: 90

Overwriting taxonomy/weights.yaml


### Project configuration

#### `pyproject.toml`

In [8]:
%%writefile pyproject.toml
[project]
name = "resume-search"
version = "0.1.0"
description = "Candidate resume search platform: LLM parsing, SQLite storage, hybrid search, Streamlit UI"
requires-python = ">=3.12,<3.13"
dependencies = [
    "anthropic>=0.125",
    "pydantic>=2.9",
    "python-docx>=1.1",
    "pymupdf>=1.24",
    "pyyaml>=6.0",
    "rapidfuzz>=3.9",
    "wordfreq>=3.1",
    "python-dotenv>=1.0",
    "sentence-transformers>=6.2.0",
    "numpy>=2.5.4",
    "streamlit>=1.66.1",
    "pandas>=3.1.0",
    "plotly>=7.1.0",
]

[project.optional-dependencies]
dev = [
    "ipykernel>=7.4.0",
    "jupyterlab>=4.6.4",
    "nbconvert>=7.17.2",
    "nbformat>=5.11.1",
    "pytest>=8.0",
    "ruff>=0.6",
]

[project.scripts]
resume-search = "resume_search.cli:main"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.hatch.build.targets.wheel]
packages = ["src/resume_search"]

[tool.ruff]
line-length = 100
target-version = "py312"

[tool.pytest.ini_options]
testpaths = ["tests"]

Overwriting pyproject.toml


### Package: src/resume_search

#### `src/resume_search/cli.py`

The `resume-search` command line: extract, parse, ingest, rederive, show, search, and eval.

In [9]:
%%writefile src/resume_search/cli.py
from __future__ import annotations

import argparse
import json
import sys
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict
from pathlib import Path

from dotenv import load_dotenv

from resume_search.extract import ExtractedText, extract
from resume_search.ingest.pipeline import attempt_cost, build_meta, ingest, rederive
from resume_search.parse import ParseOk, parse_resume
from resume_search.parse.client import ParseResult
from resume_search.search import (
    Criterion,
    Requisition,
    Weights,
    parse_requisition,
    rerank_llm,
    search,
)
from resume_search.search.rank import SearchResult
from resume_search.store.db import connect, list_candidates

SUPPORTED = {".docx", ".pdf"}
DEFAULT_DB = "exports/candidates.sqlite"


def _supported(paths: list[str]) -> list[str]:
    """A shell glob like data/* also matches non-resume files; skip them loudly."""
    for p in paths:
        if Path(p).suffix.lower() not in SUPPORTED:
            print(f"skip\t{Path(p).name}\tunsupported suffix", file=sys.stderr)
    return [p for p in paths if Path(p).suffix.lower() in SUPPORTED]


def _quality(e: ExtractedText) -> dict:
    return asdict(e.quality)


def _cmd_extract(args: argparse.Namespace) -> int:
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    for path in _supported(args.paths):
        e = extract(path)
        (out / f"{Path(path).stem}.txt").write_text(e.text, encoding="utf-8")
        print(
            json.dumps(
                {
                    "file": e.source_file,
                    "pages": e.page_count,
                    "chars": len(e.text),
                    "extraction_path": e.extraction_path.value,
                    **_quality(e),
                }
            )
        )
    return 0


def _parse_one(path: str, out: Path) -> tuple[ExtractedText, ParseResult]:
    e = extract(path)
    result = parse_resume(e)
    doc: dict = {"meta": build_meta(e, result), "extracted": None}
    if isinstance(result, ParseOk):
        doc["extracted"] = result.record.model_dump(mode="json")
    else:
        doc["error"] = result.last_error
    (out / f"{e.candidate_id}.json").write_text(
        json.dumps(doc, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return e, result


def _cmd_parse(args: argparse.Namespace) -> int:
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    failed = 0
    total_cost = 0.0
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = [(p, pool.submit(_parse_one, p, out)) for p in _supported(args.paths)]
        for path, fut in futures:
            name = Path(path).name
            try:
                _, result = fut.result()
            except Exception as exc:  # noqa: BLE001  one bad file must not stop the batch
                failed += 1
                print(f"{name}\tERROR\t{type(exc).__name__}: {exc}")
                continue
            tokens_in = sum(
                a.input_tokens + a.cache_read_tokens + a.cache_creation_tokens
                for a in result.attempts
            )
            tokens_out = sum(a.output_tokens for a in result.attempts)
            latency = sum(a.latency_ms for a in result.attempts)
            cost = sum(attempt_cost(a) for a in result.attempts)
            total_cost += cost
            failed += result.parse_status == "failed"
            v = result.verification if isinstance(result, ParseOk) else None
            checks = (
                f"checks={v.checks_run} dis={v.disagreements} regen={v.regenerated} "
                f"unres={len(v.unresolved)}"
                if v
                else "checks=- dis=- regen=- unres=-"
            )
            print(
                f"{name}\t{result.parse_status}\t{result.attempts[-1].model}\t"
                f"in={tokens_in} out={tokens_out}\t{latency}ms\t"
                f"attempts={len(result.attempts)}\t{checks}\t${cost:.4f}"
            )
    print(f"total cost ${total_cost:.4f}")
    return 1 if failed else 0


def _cmd_ingest(args: argparse.Namespace) -> int:
    if args.reparse_derive_only:
        print(f"rederived {rederive(args.db)} candidates")
        return 0
    jobs = ingest(_supported(args.paths), args.db, force=args.force, workers=args.workers)
    total = 0.0
    for j in jobs:
        total += j.cost_usd or 0.0
        print(
            f"{j.source_file}\t{j.status.value}\tin={j.input_tokens or 0} out={j.output_tokens or 0}"
            f"\t{j.latency_ms or 0}ms\t${j.cost_usd or 0:.4f}\t{j.error or ''}"
        )
    print(f"total cost ${total:.4f}")
    return 1 if any(j.status.value == "failed" for j in jobs) else 0


def _cmd_rederive(args: argparse.Namespace) -> int:
    print(f"rederived {rederive(args.db)} candidates")
    return 0


def _cmd_show(args: argparse.Namespace) -> int:
    needle = args.candidate.lower()
    hits = [
        r
        for r in list_candidates(connect(args.db))
        if r.meta["candidate_id"].startswith(needle) or needle in r.extracted.name.lower()
    ]
    if not hits:
        print(f"no candidate matches '{args.candidate}'", file=sys.stderr)
        return 1
    for r in hits:
        print(f"{r.extracted.name} ({r.meta['candidate_id']})")
        print(json.dumps(r.derived.model_dump(mode="json"), indent=2, ensure_ascii=False))
    return 0


def _years(text: str) -> tuple[float | None, float | None]:
    lo, sep, hi = text.partition("-") if "-" in text else (text.rstrip("+"), "", "")
    if not sep and not text.endswith("+"):
        return float(lo), float(lo)
    return (float(lo) if lo else None), (float(hi) if hi else None)


def _build_requisition(args: argparse.Namespace, attempts: list) -> Requisition:
    jd = Path(args.jd_file).read_text(encoding="utf-8") if args.jd_file else args.jd
    req = parse_requisition(jd, attempts=attempts) if jd else Requisition()
    if jd:
        print("Parsed requisition:", file=sys.stderr if args.json else sys.stdout)
        print(req.model_dump_json(indent=2, exclude_none=True), file=sys.stderr if args.json else sys.stdout)
    update: dict = {}
    if args.location:
        update["location"] = Criterion[str](value=args.location)
    if args.scope:
        update["location_scope"] = args.scope
    if args.approach:
        update["approach"] = Criterion[str](value=args.approach)
    if args.sector:
        update["sectors"] = Criterion[list[str]](value=args.sector)
    if args.years:
        update["years_min"], update["years_max"] = _years(args.years)
        update["years_mode"] = "must"
    if args.skill:
        update["skills_preferred"] = [*req.skills_preferred, *args.skill]
    if args.must_skill:
        update["skills_required"] = [*req.skills_required, *args.must_skill]
    if args.cfa:
        update["certifications"] = Criterion[list[str]](value=["CFA"], mode="must")
    if jd and not update:
        return req
    return Requisition.model_validate({**req.model_dump(), **{k: v for k, v in update.items()}})


def _row(rank: int, r: SearchResult) -> str:
    b = r.breakdown
    return (
        f"{rank:>2}  {r.name:<24} final={b.final:.3f} rule={b.rule:.3f} kw={b.keyword:.3f} "
        f"emb={b.embedding:.3f}  {r.seniority_band or '-':<9} {r.current_country or '-'}"
    )


def _chips_line(r: SearchResult) -> str:
    chips = "  ".join(
        f"{'✓' if c.status == 'matched' else '✗'} {c.label}" + (f" [{c.detail}]" if c.detail else "")
        for c in r.chips
    )
    return f"    {chips}" + (f"\n    kw: {', '.join(r.keywords)}" if r.keywords else "")


def _cmd_search(args: argparse.Namespace) -> int:
    attempts: list = []
    try:
        req = _build_requisition(args, attempts)
    except ValueError as exc:
        print(f"error: {exc}", file=sys.stderr)
        return 2
    results = search(connect(args.db), req, Weights.load(), top_k=args.top)
    reasons: dict[str, str] = {}
    if args.rerank and results:
        order = rerank_llm(req, results[:10], attempts=attempts)
        reasons = dict(order)
        by_id = {r.candidate_id: r for r in results}
        results = [by_id[i] for i, _ in order] + results[10:]
    if attempts:
        cost = sum(attempt_cost(a) for a in attempts)
        tokens = sum(a.input_tokens + a.output_tokens for a in attempts)
        print(f"api calls={len(attempts)} tokens={tokens} cost=${cost:.4f}", file=sys.stderr)
    if args.json:
        print(json.dumps(
            [{**asdict(r), "reason": reasons.get(r.candidate_id)} for r in results],
            indent=2, ensure_ascii=False, default=str,
        ))
        return 0
    if not results:
        print("no candidates match")
    for i, r in enumerate(results, 1):
        print(_row(i, r))
        print(_chips_line(r))
        if r.candidate_id in reasons:
            print(f"    reason: {reasons[r.candidate_id]}")
    return 0


def _cmd_eval_run(args: argparse.Namespace) -> int:
    from resume_search.eval.parsing_eval import run_parsing_eval
    from resume_search.eval.ranking_eval import run_ranking_eval
    from resume_search.eval.report import FORMATTERS

    only = [x for x in args.only.split(",") if x] if args.only else None
    suites = ["parsing", "ranking"] if args.suite == "all" else [args.suite]
    total = 0.0
    for suite in suites:
        if suite == "parsing":
            path, body = run_parsing_eval(args.db, args.out, only=only)
        else:
            path, body = run_ranking_eval(args.db, args.out)
        total += body["cost_usd"]
        print(FORMATTERS[suite](body))
        print(f"wrote {path}\n")
    print(f"total judge cost ${total:.4f}")
    return 0


def _cmd_eval_show(args: argparse.Namespace) -> int:
    from resume_search.eval.report import FORMATTERS
    from resume_search.eval.results import SUITES, latest_result

    shown = 0
    for suite in [args.suite] if args.suite else SUITES:
        if found := latest_result(args.out, suite):
            print(f"{found[0]}\n{FORMATTERS[suite](found[1])}\n")
            shown += 1
    if not shown:
        print(f"no eval results in {args.out}", file=sys.stderr)
    return 0 if shown else 1


def main(argv: list[str] | None = None) -> None:
    load_dotenv()
    parser = argparse.ArgumentParser(prog="resume-search")
    sub = parser.add_subparsers(dest="cmd", required=True)
    p_ext = sub.add_parser("extract", help="extract text and print quality metrics")
    p_ext.add_argument("paths", nargs="+")
    p_ext.add_argument("--out", default="exports/text")
    p_ext.set_defaults(func=_cmd_extract)
    p_par = sub.add_parser("parse", help="extract and parse resumes with the LLM")
    p_par.add_argument("paths", nargs="+")
    p_par.add_argument("--out", default="exports/parsed")
    p_par.add_argument("--workers", type=int, default=4)
    p_par.set_defaults(func=_cmd_parse)
    p_ing = sub.add_parser("ingest", help="extract, parse, derive and index resumes into SQLite")
    p_ing.add_argument("paths", nargs="*")
    p_ing.add_argument("--db", default=DEFAULT_DB)
    p_ing.add_argument("--force", action="store_true")
    p_ing.add_argument("--workers", type=int, default=4)
    p_ing.add_argument("--reparse-derive-only", action="store_true")
    p_ing.set_defaults(func=_cmd_ingest)
    p_red = sub.add_parser("rederive", help="rebuild derived fields and vectors without API calls")
    p_red.add_argument("--db", default=DEFAULT_DB)
    p_red.set_defaults(func=_cmd_rederive)
    p_show = sub.add_parser("show", help="print a candidate's derived fields")
    p_show.add_argument("candidate", help="candidate_id prefix or name substring")
    p_show.add_argument("--db", default=DEFAULT_DB)
    p_show.set_defaults(func=_cmd_show)
    p_s = sub.add_parser("search", help="rank ingested candidates against a requisition")
    p_s.add_argument("--jd")
    p_s.add_argument("--jd-file")
    p_s.add_argument("--location")
    p_s.add_argument("--scope", choices=["current", "worked", "either"])
    p_s.add_argument("--approach")
    p_s.add_argument("--sector", action="append")
    p_s.add_argument("--years", help="MIN-MAX, MIN+ or a single number; always a must filter")
    p_s.add_argument("--skill", action="append", help="preferred skill")
    p_s.add_argument("--must-skill", action="append")
    p_s.add_argument("--cfa", action="store_true", help="require the CFA")
    p_s.add_argument("--top", type=int, default=20)
    p_s.add_argument("--rerank", action="store_true")
    p_s.add_argument("--db", default=DEFAULT_DB)
    p_s.add_argument("--json", action="store_true")
    p_s.set_defaults(func=_cmd_search)
    p_ev = sub.add_parser("eval", help="LLM-judged parsing and ranking evals")
    ev = p_ev.add_subparsers(dest="eval_cmd", required=True)
    p_er = ev.add_parser("run", help="run a suite and write evals/results/<suite>-<timestamp>.json")
    p_er.add_argument("--suite", choices=["parsing", "ranking", "all"], default="all")
    p_er.add_argument("--db", default=DEFAULT_DB)
    p_er.add_argument("--out", default="evals/results")
    p_er.add_argument("--only", help="comma-separated candidate_ids (parsing suite)")
    p_er.set_defaults(func=_cmd_eval_run)
    p_es = ev.add_parser("show", help="print the latest metrics table per suite")
    p_es.add_argument("--suite", choices=["parsing", "ranking"])
    p_es.add_argument("--out", default="evals/results")
    p_es.set_defaults(func=_cmd_eval_show)
    args = parser.parse_args(argv)
    sys.exit(args.func(args))

Overwriting src/resume_search/cli.py


#### `src/resume_search/derive/__init__.py`

`derive()` builds the `derived` layer from a parsed resume and the taxonomy, in pure code.

In [10]:
%%writefile src/resume_search/derive/__init__.py
from __future__ import annotations

from datetime import date

from resume_search.derive import experience as exp
from resume_search.derive import location, skills
from resume_search.derive.models import (
    CandidateRecord,
    CountryMonths,
    DerivedFields,
    SectorEvidence,
)
from resume_search.parse.models import ParsedResume
from resume_search.taxonomy import Taxonomy

__all__ = ["CandidateRecord", "DerivedFields", "derive"]

_DEGREE_ORDER = {"Bachelors": 1, "Masters": 2, "MBA": 3, "PhD": 4}


def _highest_degree(parsed: ParsedResume) -> str | None:
    levels = [e.degree_level for e in parsed.education if e.degree_level]
    if not levels:
        return None
    return max(levels, key=lambda lv: _DEGREE_ORDER.get(lv, 0))


def _dedupe(values) -> list[str]:
    return list(dict.fromkeys(values))


def _sectors(parsed: ParsedResume, tax: Taxonomy) -> tuple[list[SectorEvidence], list[str]]:
    found: dict[str, str] = {}
    for e in parsed.experience:
        for s in e.sectors:
            found.setdefault(s.label, s.evidence.quote)
    tags = sorted({t.strip().lower() for e in parsed.experience for t in e.other_sector_tags if t.strip()})
    for tag in tags:
        if core := tax.tag_to_core.get(tag):
            found.setdefault(core, tag)
    return [SectorEvidence(label=k, evidence=v) for k, v in found.items()], tags


def _today(meta: dict) -> date:
    parsed_at = meta.get("parsed_at")
    return date.fromisoformat(parsed_at[:10]) if parsed_at else date.today()


def derive(
    parsed: ParsedResume, meta: dict, taxonomy: Taxonomy, *, today: date | None = None
) -> DerivedFields:
    today = today or _today(meta)
    roles = parsed.experience
    relevant = exp.total_months([e for e in roles if exp.is_relevant_full_time(e)], today)
    country = location.current_country(parsed, taxonomy)
    worked = location.countries_worked(parsed, today)
    sectors, tags = _sectors(parsed, taxonomy)
    has_cfa, cfa_level = skills.cfa_status(parsed.certifications)
    verification = meta.get("verification") or {}
    return DerivedFields(
        current_country=country,
        current_region=taxonomy.country_region.get(country) if country else None,
        countries_worked=[CountryMonths(country=c, months=m) for c, m in worked],
        regions_worked=_dedupe(
            taxonomy.country_region[c] for c, _ in worked if c in taxonomy.country_region
        ),
        markets_covered=location.markets_covered(parsed, taxonomy),
        months_full_time_relevant=relevant,
        months_full_time_core=exp.total_months([e for e in roles if exp.is_core_full_time(e)], today),
        months_internship=exp.total_months([e for e in roles if exp.is_internship(e)], today),
        seniority_band=exp.seniority_band(relevant),
        investment_approach=parsed.investment_approach.label,
        approach_evidence=parsed.investment_approach.evidence.quote,
        sectors=sectors,
        other_sector_tags=tags,
        asset_classes=_dedupe(a.value for e in roles for a in e.asset_classes),
        firm_types=_dedupe(e.firm_type.value for e in roles),
        highest_degree=_highest_degree(parsed),
        degree_fields=_dedupe(e.field for e in parsed.education if e.field),
        skills_normalized=skills.skills_normalized(parsed, taxonomy),
        has_cfa=has_cfa,
        cfa_level=cfa_level,
        search_summary=parsed.search_summary,
        quality={
            "text_quality": meta.get("text_quality"),
            "skill_disagreement_count": 0,
            "unresolved_count": len(verification.get("unresolved", [])),
        },
    )

Overwriting src/resume_search/derive/__init__.py


#### `src/resume_search/derive/experience.py`

Month arithmetic over roles: merges overlapping ranges, falls back to stated durations, and maps years to a seniority band.

In [11]:
%%writefile src/resume_search/derive/experience.py
"""Month arithmetic over roles. Ranges are merged per bucket so overlapping jobs are not double
counted; roles without a start fall back to duration_months and are summed as-is."""

from __future__ import annotations

from collections.abc import Iterable
from datetime import date

from resume_search.parse.models import EmploymentType, Experience, Relevance

# Upper bounds are inclusive so the design's ">120" Principal and "<6" Intern hold exactly.
_BANDS = ((5, "Intern/Student"), (24, "Junior"), (60, "Mid"), (120, "Senior"))


def month_index(value: str) -> int:
    """YYYY-MM -> months since year 0; a bare YYYY means June."""
    month = int(value[5:7]) if len(value) >= 7 else 6
    return int(value[:4]) * 12 + month - 1


def role_span(e: Experience, today: date) -> tuple[int, int] | None:
    if e.start is None:
        return None
    start = month_index(e.start)
    if e.is_current:
        end = today.year * 12 + today.month - 1
    elif e.end:
        end = month_index(e.end)
    elif e.duration_months is not None:
        end = start + e.duration_months
    else:
        return None
    return start, max(start, end)


def role_months(e: Experience, today: date) -> int:
    span = role_span(e, today)
    if span:
        return span[1] - span[0]
    return e.duration_months or 0


def total_months(roles: Iterable[Experience], today: date) -> int:
    spans: list[tuple[int, int]] = []
    loose = 0
    for e in roles:
        span = role_span(e, today)
        if span:
            spans.append(span)
        else:
            loose += e.duration_months or 0
    merged = 0
    cur_start = cur_end = None
    for s, t in sorted(spans):
        if cur_end is None or s > cur_end:
            if cur_end is not None:
                merged += cur_end - cur_start
            cur_start, cur_end = s, t
        else:
            cur_end = max(cur_end, t)
    if cur_end is not None:
        merged += cur_end - cur_start
    return merged + loose


def is_relevant_full_time(e: Experience) -> bool:
    return e.employment_type is EmploymentType.full_time and e.relevance in (
        Relevance.core,
        Relevance.adjacent,
    )


def is_core_full_time(e: Experience) -> bool:
    return e.employment_type is EmploymentType.full_time and e.relevance is Relevance.core


def is_internship(e: Experience) -> bool:
    return e.employment_type is EmploymentType.internship


def seniority_band(months_relevant: int) -> str:
    for upper, label in _BANDS:
        if months_relevant <= upper:
            return label
    return "Principal"

Overwriting src/resume_search/derive/experience.py


#### `src/resume_search/derive/location.py`

Derives the current country, the countries worked in, and the markets covered.

In [12]:
%%writefile src/resume_search/derive/location.py
from __future__ import annotations

import re
from datetime import date

from resume_search.derive.experience import month_index, total_months
from resume_search.parse.models import Contact, EmploymentType, ParsedResume
from resume_search.taxonomy import Taxonomy


def _gazetteer(text: str, tax: Taxonomy) -> str | None:
    """Match comma-separated parts of a free-text location against country names and aliases,
    last part first ('Mumbai, India' hits the country before the city)."""
    parts = [p.strip().lower() for p in reversed(text.split(",")) if p.strip()]
    for part in parts:
        for iso, names in tax.country_names.items():
            if part in (n.lower() for n in names):
                return iso
    return None


def _from_phone(phone: str, tax: Taxonomy) -> str | None:
    compact = re.sub(r"[^\d+]", "", phone)
    if not compact.startswith("+"):
        return None
    for prefix in sorted(tax.phone_prefix, key=len, reverse=True):
        if compact.startswith(prefix):
            isos = tax.phone_prefix[prefix]
            return isos[0] if len(isos) == 1 else None
    return None


def _contact_country(contact: Contact, tax: Taxonomy) -> str | None:
    if contact.location_text and (iso := _gazetteer(contact.location_text, tax)):
        return iso
    if contact.phone:
        return _from_phone(contact.phone, tax)
    return None


def current_country(parsed: ParsedResume, tax: Taxonomy) -> str | None:
    """Country of the current role with the latest start, else of the latest role by start
    (first listed when nothing is dated). Roles with no stated country are skipped, then the
    contact block decides."""
    roles = parsed.experience

    def start_key(i: int) -> int:
        s = roles[i].start
        return month_index(s) if s else -1

    order = sorted(range(len(roles)), key=lambda i: (start_key(i), -i), reverse=True)
    current = [i for i in order if roles[i].is_current]
    for i in current + order:
        if roles[i].country:
            return roles[i].country
    return _contact_country(parsed.contact, tax)


def countries_worked(parsed: ParsedResume, today: date) -> list[tuple[str, int]]:
    by_country: dict[str, list] = {}
    for e in parsed.experience:
        if e.employment_type is EmploymentType.full_time and e.country:
            by_country.setdefault(e.country, []).append(e)
    return [(c, total_months(roles, today)) for c, roles in by_country.items()]


def markets_covered(parsed: ParsedResume, tax: Taxonomy) -> list[str]:
    """Labels as stated, plus the countries and regions each label expands to."""
    out: set[str] = set()
    for e in parsed.experience:
        for m in e.markets_covered:
            out.add(m.label)
            countries, regions = tax.market_expansion.get(m.label, (frozenset(), frozenset()))
            out |= countries | regions
    return sorted(out)

Overwriting src/resume_search/derive/location.py


#### `src/resume_search/derive/models.py`

The `DerivedFields` and `CandidateRecord` data shapes and the schema version.

In [13]:
%%writefile src/resume_search/derive/models.py
"""Data shapes for the derived layer (design 4.2) and the stored record."""

from __future__ import annotations

from pydantic import BaseModel

from resume_search.parse.models import ParsedResume

# Bump when derive() output changes; `rederive` then rebuilds every candidate without API calls.
SCHEMA_VERSION = 1


class CountryMonths(BaseModel):
    country: str
    months: int


class SectorEvidence(BaseModel):
    label: str
    evidence: str


class DerivedFields(BaseModel):
    current_country: str | None
    current_region: str | None
    countries_worked: list[CountryMonths]
    regions_worked: list[str]
    markets_covered: list[str]
    months_full_time_relevant: int
    months_full_time_core: int
    months_internship: int
    seniority_band: str
    investment_approach: str
    approach_evidence: str
    sectors: list[SectorEvidence]
    other_sector_tags: list[str]
    asset_classes: list[str]
    firm_types: list[str]
    highest_degree: str | None
    degree_fields: list[str]
    skills_normalized: list[str]
    has_cfa: bool
    cfa_level: str | None
    search_summary: str
    quality: dict


class CandidateRecord(BaseModel):
    meta: dict
    extracted: ParsedResume
    derived: DerivedFields

Overwriting src/resume_search/derive/models.py


#### `src/resume_search/derive/skills.py`

Normalizes skills to canonical tokens and decides CFA status.

In [14]:
%%writefile src/resume_search/derive/skills.py
from __future__ import annotations

import re

from rapidfuzz import fuzz, process

from resume_search.parse.models import ParsedResume
from resume_search.taxonomy import Taxonomy

FUZZY_THRESHOLD = 90
# Short tokens ("sas", "sap", "r") are one edit from unrelated skills; only exact-match them.
FUZZY_MIN_LEN = 5

_CFA = re.compile(r"\bCFA\b|chartered financial analyst", re.IGNORECASE)
_CFA_PENDING = re.compile(r"\blevel\b|candidate|pursuing|in progress|enrolled|passed", re.IGNORECASE)
_CFA_LEVEL = re.compile(r"\blevel\s*(III|II|I|3|2|1)\b", re.IGNORECASE)
_ARABIC = {"1": "I", "2": "II", "3": "III"}


def normalize_skill(raw: str, tax: Taxonomy) -> str:
    key = raw.strip().lower()
    if key in tax.skill_canonical:
        return tax.skill_canonical[key]
    if len(key) >= FUZZY_MIN_LEN:
        hit = process.extractOne(
            key, tax.skill_canonical.keys(), scorer=fuzz.ratio, score_cutoff=FUZZY_THRESHOLD
        )
        if hit:
            return tax.skill_canonical[hit[0]]
    return key


def skills_normalized(parsed: ParsedResume, tax: Taxonomy) -> list[str]:
    s = parsed.skills
    raw = [*s.programming, *s.tools, *s.domain]
    return sorted({n for r in raw if r.strip() and (n := normalize_skill(r, tax))})


def cfa_status(certifications: list[str]) -> tuple[bool, str | None]:
    """has_cfa is true only for a charterholder or bare CFA; a level or candidacy mention
    makes the entry a candidate, whose level is reported instead."""
    has, level = False, None
    for cert in certifications:
        if not _CFA.search(cert):
            continue
        if m := _CFA_LEVEL.search(cert):
            level = _ARABIC.get(m.group(1), m.group(1).upper())
        if not _CFA_PENDING.search(cert):
            has = True
    return has, level

Overwriting src/resume_search/derive/skills.py


#### `src/resume_search/eval/judge.py`

The Opus judge prompts and calls for the parsing and ranking evals, and their cost accounting.

In [15]:
%%writefile src/resume_search/eval/judge.py
"""Opus judges for the parsing and ranking evals (design section 10). The task model is Sonnet."""

from __future__ import annotations

import json
from collections.abc import Sequence

from resume_search.eval.models import (
    EvalRequisition,
    ParsingGrades,
    ParsingJudgement,
    RankingGrades,
    RankingJudgement,
    expected_fields,
)
from resume_search.parse.client import OPUS, Attempt
from resume_search.parse.models import ParsedResume
from resume_search.search.llm import structured_call

PARSING_PROMPT_VERSION = "pj2"
RANKING_PROMPT_VERSION = "rj1"
JUDGE_MODEL = OPUS
JUDGE_MAX_TOKENS = 12000

# USD per million tokens on Opus
PRICE_IN, PRICE_OUT, PRICE_CACHE_READ, PRICE_CACHE_WRITE = 4.0, 20.0, 0.40, 5.00


def judge_cost(attempts: Sequence[Attempt]) -> float:
    return sum(
        a.input_tokens * PRICE_IN
        + a.output_tokens * PRICE_OUT
        + a.cache_read_tokens * PRICE_CACHE_READ
        + a.cache_creation_tokens * PRICE_CACHE_WRITE
        for a in attempts
    ) / 1e6


def token_totals(attempts: Sequence[Attempt]) -> dict[str, int]:
    return {
        "input": sum(a.input_tokens for a in attempts),
        "output": sum(a.output_tokens for a in attempts),
        "cache_read": sum(a.cache_read_tokens for a in attempts),
        "cache_write": sum(a.cache_creation_tokens for a in attempts),
        "calls": len(attempts),
    }


_PARSING_SYSTEM = """\
You audit a resume parser. You get the resume text and the JSON the parser extracted from it.
Grade each field of the JSON by comparing it to the resume text. The reply is constrained to
the output schema.

Grades
- correct: the JSON value is fully supported by the text and nothing material is left out.
- partial: mostly right, with one wrong or unsupported detail (a wrong date, a wrong title, a
  list with one extra or one missing item, a misclassified type). Name the detail in reason.
- wrong: the JSON contradicts the text, or the value is mostly unsupported.
- hallucinated: the JSON holds a value the text does not support at all, where the text has
  nothing on the subject.
- missing: the text has the information and the JSON leaves it out or null.
When a field has both an unsupported item and an omission, pick the worse grade and name both.
A field the text is silent on, with an empty or null value in the JSON, is correct.

Fields to grade, each exactly once, with the field name spelled as given
- name, contact (email, phone, location_text), education (every entry, dates, degree, level,
  field), experience (the list as a whole: every role present, none invented, none merged or
  split wrongly), skills (programming, tools, domain), certifications, languages,
  investment_approach, search_summary.
- experience[i] for each role in order, i from 0. Judge employer, title, start and end dates,
  is_current, employment_type, relevance and country together. A role whose dates are wrong
  is not correct.
- Your list of field names is given in the user message. Use exactly those.

Checks you must do
- Every date: compare each start and end in education and experience to the text, month and
  year. Check is_current against "present", "till now" or "since".
- Every evidence quote: investment_approach.evidence.quote and every sectors and
  markets_covered evidence quote must appear in the text and must support the label it is
  attached to. A quote that does not support its label makes that role or the approach
  partial or wrong.
- search_summary: every factual claim (employer, title, tools, numbers, degrees) must be in
  the text or in the JSON. A claim in neither is unsupported. Flag a summary that links a
  tool or result to a role where the text does not. The opening sentence carries a seniority
  band (Intern/Student, Junior, Mid, Senior, Principal) and an "N.Ny relevant" figure that
  code computed from the JSON dates and roles, and they are not in the text by design. Do not
  mark the summary down for either one being absent from the text. Do mark it down when the
  figure rests on a date or role you found wrong, and say which.
- skills: items must be skills the text lists or clearly uses. Course names, awards and
  interests are not skills.
- A field the text states ambiguously or inconsistently is graded on the most reasonable
  reading. Say so in reason.

reason is one specific sentence. Name the value you checked and what the text says. For a
correct grade, say what you verified. Do not use facts that are in neither input.
"""


def _parsing_user(text: str, parsed: ParsedResume, fields: list[str]) -> str:
    return (
        f"<resume_text>\n{text}\n</resume_text>\n"
        f"<parsed_json>\n{parsed.model_dump_json(indent=1)}\n</parsed_json>\n"
        f"<fields_to_grade>\n{chr(10).join(fields)}\n</fields_to_grade>"
    )


def judge_parsing(
    text: str,
    parsed_record: ParsedResume,
    client,
    candidate_id: str = "",
    attempts: list[Attempt] | None = None,
) -> ParsingJudgement:
    fields = expected_fields(len(parsed_record.experience))

    def validate(out: ParsingGrades) -> list[str]:
        got = [r.field for r in out.grades]
        errors = [f"missing field {f}" for f in fields if f not in got]
        errors += [f"unknown field {f}" for f in dict.fromkeys(got) if f not in fields]
        errors += [f"duplicate field {f}" for f in dict.fromkeys(got) if got.count(f) > 1]
        return errors

    out = structured_call(
        client, _PARSING_SYSTEM, _parsing_user(text, parsed_record, fields), ParsingGrades,
        validate=validate, attempts=attempts, call="judge_parsing", model=JUDGE_MODEL,
        max_tokens=JUDGE_MAX_TOKENS,
    )
    order = {f: i for i, f in enumerate(fields)}
    return ParsingJudgement(
        candidate_id=candidate_id, grades=sorted(out.grades, key=lambda r: order[r.field])
    )


_RANKING_SYSTEM = """\
You judge how well each shortlisted hedge fund candidate fits a search requisition. The reply
is constrained to the output schema.

Grades
- 2: strong fit. Meets the must criteria and most of the preferred ones, with experience that
  matches the role.
- 1: partially meets. Misses a preferred criterion or two, or is a near fit in seniority,
  sector or approach, and the gap could be closed in an interview.
- 0: not relevant. Misses a must criterion, or the experience does not match the role.

Rules
- Must criteria are hard requirements. Preferred criteria and the free text are soft.
- Judge the candidate against the requisition on its own. Do not compare candidates to each
  other, and do not try to spread the grades. All candidates may get 0, or all 2.
- Use only the facts in the candidate block. Do not infer facts about a firm or person.
- Return every candidate exactly once. Use only the candidate_id values given.
- reason is one sentence naming the facts that decided the grade.
"""


def _facets(c: dict) -> str:
    skip = {"candidate_id", "name", "search_summary"}
    facets = {k: v for k, v in c.items() if k not in skip and v not in (None, [], "")}
    return json.dumps(facets, ensure_ascii=False)


def ranking_candidate_payload(c: dict) -> str:
    return (
        f'<candidate id="{c["candidate_id"]}" name="{c["name"]}">\n'
        f"{c['search_summary']}\nFacets: {_facets(c)}\n</candidate>"
    )


def judge_ranking(
    requisition: EvalRequisition,
    candidates: list[dict],
    client,
    attempts: list[Attempt] | None = None,
) -> RankingJudgement:
    """One call per requisition covering every candidate in the pool."""
    user = (
        f"<requisition title=\"{requisition.title}\">\n"
        f"{requisition.requisition.model_dump_json(exclude_none=True, indent=1)}\n</requisition>\n"
        + "\n".join(ranking_candidate_payload(c) for c in candidates)
    )
    want = [c["candidate_id"] for c in candidates]

    def validate(out: RankingGrades) -> list[str]:
        got = [r.candidate_id for r in out.grades]
        errors = [f"unknown candidate_id {i}" for i in dict.fromkeys(got) if i not in want]
        errors += [f"missing candidate_id {i}" for i in want if i not in got]
        errors += [f"duplicate candidate_id {i}" for i in dict.fromkeys(got) if got.count(i) > 1]
        return errors

    out = structured_call(
        client, _RANKING_SYSTEM, user, RankingGrades, validate=validate, attempts=attempts,
        call="judge_ranking", model=JUDGE_MODEL, max_tokens=JUDGE_MAX_TOKENS,
    )
    return RankingJudgement(requisition_id=requisition.id, grades=out.grades)

Overwriting src/resume_search/eval/judge.py


#### `src/resume_search/eval/metrics.py`

Pure scoring functions: per-field accuracy, nDCG@k, precision@k, and judge-human agreement.

In [16]:
%%writefile src/resume_search/eval/metrics.py
"""Pure scoring functions for the eval suites. No I/O, no model calls."""

from __future__ import annotations

import math
from collections.abc import Iterable, Mapping, Sequence

GRADES = ("correct", "partial", "wrong", "hallucinated", "missing")
# correct > partial > wrong, missing and hallucinated
LEVEL = {"correct": 2, "partial": 1, "wrong": 0, "missing": 0, "hallucinated": 0}


def field_group(field: str) -> str:
    """experience[3] belongs to the experience_role group; everything else is its own group."""
    return "experience_role" if field.startswith("experience[") else field


def _pair(row) -> tuple[str, str]:
    if isinstance(row, Mapping):
        return row["field"], row["grade"]
    return row.field, row.grade


def field_accuracy(grades: Iterable) -> dict[str, dict[str, float]]:
    """Per field group: a count per grade, `total`, `accuracy` (correct / total) and
    `soft_accuracy` (partial counts 0.5). The `overall` key covers every row."""
    out: dict[str, dict[str, float]] = {}
    for row in grades:
        field, grade = _pair(row)
        for key in (field_group(field), "overall"):
            g = out.setdefault(key, {**dict.fromkeys(GRADES, 0), "total": 0})
            g[grade] += 1
            g["total"] += 1
    for g in out.values():
        g["accuracy"] = g["correct"] / g["total"]
        g["soft_accuracy"] = (g["correct"] + 0.5 * g["partial"]) / g["total"]
    return out


def ndcg_at_k(ranked_ids: Sequence[str], grades: Mapping[str, int], k: int) -> float:
    """Linear-gain nDCG. The ideal order sorts every graded candidate; an id with no grade
    counts as 0. A pool with no relevant candidate has nothing to rank and scores 0.0."""

    def dcg(rels: Sequence[int]) -> float:
        return sum(r / math.log2(i + 2) for i, r in enumerate(rels[:k]))

    ideal = dcg(sorted(grades.values(), reverse=True))
    if ideal == 0:
        return 0.0
    return dcg([grades.get(i, 0) for i in ranked_ids]) / ideal


def precision_at_k(
    ranked_ids: Sequence[str], grades: Mapping[str, int], k: int, threshold: int = 1
) -> float:
    """Share of the top k with grade >= threshold. A pool smaller than k divides by its size."""
    top = list(ranked_ids[:k])
    if not top:
        return 0.0
    return sum(grades.get(i, 0) >= threshold for i in top) / len(top)


def agreement(human: Iterable, judge: Iterable) -> dict:
    """Compare two gradings of the same fields. `within_one_level` counts a pair as agreeing
    unless one side says correct and the other wrong, missing or hallucinated."""
    h = dict(_pair(r) for r in human)
    j = dict(_pair(r) for r in judge)
    shared = [f for f in h if f in j]
    if not shared:
        return {"exact": 0.0, "within_one_level": 0.0, "n": 0, "disagreements": []}
    exact = sum(h[f] == j[f] for f in shared)
    near = sum(abs(LEVEL[h[f]] - LEVEL[j[f]]) <= 1 for f in shared)
    return {
        "exact": exact / len(shared),
        "within_one_level": near / len(shared),
        "n": len(shared),
        "disagreements": [{"field": f, "human": h[f], "judge": j[f]} for f in shared if h[f] != j[f]],
    }

Overwriting src/resume_search/eval/metrics.py


#### `src/resume_search/eval/models.py`

Data shapes for judge output and evaluation requisitions.

In [17]:
%%writefile src/resume_search/eval/models.py
"""Data shapes shared by the judge, the metrics and the runners (design section 10)."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Literal

from pydantic import BaseModel

from resume_search.search.requisition import Requisition

FieldGrade = Literal["correct", "partial", "wrong", "hallucinated", "missing"]
RelevanceGrade = Literal[0, 1, 2]

FIELDS = (
    "name", "contact", "education", "experience", "skills", "certifications", "languages",
    "investment_approach", "search_summary",
)


def expected_fields(n_roles: int) -> list[str]:
    """Every field a parsing judgement must grade, in order: top-level fields, then each role."""
    return [*FIELDS, *(f"experience[{i}]" for i in range(n_roles))]


class FieldRow(BaseModel):
    field: str
    grade: FieldGrade
    reason: str


class ParsingGrades(BaseModel):
    """What the judge fills. candidate_id is attached by code, not asked of the model."""

    grades: list[FieldRow]


class ParsingJudgement(BaseModel):
    candidate_id: str
    grades: list[FieldRow]


class RankRow(BaseModel):
    candidate_id: str
    grade: RelevanceGrade
    reason: str


class RankingGrades(BaseModel):
    grades: list[RankRow]


class RankingJudgement(BaseModel):
    requisition_id: str
    grades: list[RankRow]


class EvalRequisition(BaseModel):
    id: str
    title: str
    requisition: Requisition


def load_requisitions(path: str | Path) -> list[EvalRequisition]:
    """One JSON object per line: id, title, then the Requisition fields flat."""
    out = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        raw = json.loads(line)
        out.append(
            EvalRequisition(
                id=raw.pop("id"), title=raw.pop("title"), requisition=Requisition.model_validate(raw)
            )
        )
    return out

Overwriting src/resume_search/eval/models.py


#### `src/resume_search/eval/parsing_eval.py`

Runs the parsing suite: judges every stored record against its resume text and compares with the hand grades.

In [18]:
%%writefile src/resume_search/eval/parsing_eval.py
from __future__ import annotations

import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from resume_search.eval.judge import (
    JUDGE_MODEL,
    PARSING_PROMPT_VERSION,
    judge_cost,
    judge_parsing,
    token_totals,
)
from resume_search.eval.metrics import agreement, field_accuracy
from resume_search.eval.results import now_utc, write_results
from resume_search.parse.client import Attempt, default_client
from resume_search.store.db import connect, get_candidate

DEFAULT_GOLD = "evals/gold/parsing_human.jsonl"


def load_gold(path: str | Path) -> dict[str, dict]:
    p = Path(path)
    if not p.is_file():
        return {}
    rows = [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()]
    return {r["candidate_id"]: r for r in rows}


def load_text(conn, text_dir: Path, candidate_id: str, source_file: str | None) -> tuple[str, str]:
    """The full extracted text when exports/text has it, else the FTS body (contact stripped)."""
    if source_file:
        f = text_dir / f"{Path(source_file).stem}.txt"
        if f.is_file():
            return f.read_text(encoding="utf-8"), "text_file"
    row = conn.execute(
        "SELECT body FROM candidate_fts WHERE candidate_id = ?", (candidate_id,)
    ).fetchone()
    return (row["body"] if row else ""), "fts_body"


def _summary(grades: list[dict]) -> dict:
    acc = field_accuracy(grades)["overall"]
    return {
        "accuracy": round(acc["accuracy"], 4), "soft_accuracy": round(acc["soft_accuracy"], 4),
        "fields": acc["total"], "hallucinated": acc["hallucinated"], "missing": acc["missing"],
    }


def run_parsing_eval(
    db_path: str | Path,
    out_dir: str | Path,
    client=None,
    only: list[str] | None = None,
    gold_path: str | Path = DEFAULT_GOLD,
    text_dir: str | Path | None = None,
    workers: int = 4,
) -> tuple[Path, dict]:
    client = client or default_client()
    conn = connect(db_path)
    text_dir = Path(text_dir) if text_dir else Path(db_path).resolve().parent / "text"
    rows = conn.execute(
        "SELECT candidate_id, name, source_file FROM candidates ORDER BY name"
    ).fetchall()
    rows = [r for r in rows if not only or r["candidate_id"] in only]
    gold = load_gold(gold_path)

    inputs = []
    for r in rows:
        record = get_candidate(conn, r["candidate_id"])
        text, source = load_text(conn, text_dir, r["candidate_id"], r["source_file"])
        inputs.append((r["candidate_id"], r["name"], record.extracted, text, source))

    def one(args) -> tuple[dict, list[Attempt]]:
        cid, name, parsed, text, source = args
        attempts: list[Attempt] = []
        grades = [g.model_dump() for g in judge_parsing(text, parsed, client, cid, attempts).grades]
        item = {"candidate_id": cid, "name": name, "text_source": source, **_summary(grades),
                "grades": grades}
        if cid in gold:
            item["human_agreement"] = agreement(gold[cid]["grades"], grades)
        return item, attempts

    with ThreadPoolExecutor(max_workers=workers) as pool:
        results = list(pool.map(one, inputs))
    items = [i for i, _ in results]
    attempts = [a for _, atts in results for a in atts]

    all_grades = [g for i in items for g in i["grades"]]
    per_field = {
        k: {n: round(v, 4) if isinstance(v, float) else v for n, v in d.items()}
        for k, d in field_accuracy(all_grades).items()
    }
    gold_items = [i for i in items if "human_agreement" in i]

    def keyed(cid: str, grades: list[dict]) -> list[dict]:
        return [{**g, "field": f"{cid}:{g['field']}"} for g in grades]

    overall = agreement(
        [g for i in gold_items for g in keyed(i["candidate_id"], gold[i["candidate_id"]]["grades"])],
        [g for i in gold_items for g in keyed(i["candidate_id"], i["grades"])],
    ) if gold_items else None

    run_at = now_utc()
    metrics = {
        "candidates": len(items),
        **_summary(all_grades),
        "per_field": per_field,
        "agreement": {
            "gold_status": sorted({gold[i["candidate_id"]].get("status", "final") for i in gold_items}),
            "candidates": [i["name"] for i in gold_items],
            "overall": overall,
            "per_candidate": {i["name"]: i["human_agreement"] for i in gold_items},
        },
    }
    body = {
        "task_model": "claude-sonnet-5-5", "judge_model": JUDGE_MODEL,
        "prompt_version": PARSING_PROMPT_VERSION, "metrics": metrics, "items": items,
        "cost_usd": round(judge_cost(attempts), 4), "tokens": token_totals(attempts),
    }
    return write_results(out_dir, "parsing", run_at, body), body

Overwriting src/resume_search/eval/parsing_eval.py


#### `src/resume_search/eval/ranking_eval.py`

Runs the ranking suite: ranks each requisition with four score variants and grades the pool once.

In [19]:
%%writefile src/resume_search/eval/ranking_eval.py
from __future__ import annotations

import hashlib
import json
from dataclasses import replace
from pathlib import Path

from resume_search.eval.judge import (
    JUDGE_MODEL,
    RANKING_PROMPT_VERSION,
    judge_cost,
    judge_ranking,
    ranking_candidate_payload,
    token_totals,
)
from resume_search.eval.metrics import ndcg_at_k, precision_at_k
from resume_search.eval.models import EvalRequisition, load_requisitions
from resume_search.eval.results import now_utc, write_results
from resume_search.parse.client import Attempt, default_client
from resume_search.search import Weights, search
from resume_search.store.db import connect, get_candidate

DEFAULT_REQUISITIONS = "evals/requisitions.jsonl"
CACHE_NAME = "ranking-grades-cache.json"
FULL = Weights.load()
# name -> (rule, keyword, embedding) blend; "full" keeps the weights.yaml blend
VARIANTS = {
    "full": None,
    "rule_only": (1.0, 0.0, 0.0),
    "keyword_only": (0.0, 1.0, 0.0),
    "embedding_only": (0.0, 0.0, 1.0),
}


def _weights(variant: str) -> Weights:
    blend = VARIANTS[variant]
    if blend is None:
        return FULL
    return replace(FULL, blend_rule=blend[0], blend_keyword=blend[1], blend_embedding=blend[2])


def _candidate_payload(record) -> dict:
    d = record.derived
    return {
        "candidate_id": record.meta["candidate_id"],
        "name": record.extracted.name,
        "search_summary": d.search_summary,
        "seniority_band": d.seniority_band,
        "investment_approach": d.investment_approach,
        "years_relevant": round(d.months_full_time_relevant / 12, 1),
        "sectors": [s.label for s in d.sectors],
        "asset_classes": d.asset_classes,
        "current_country": d.current_country,
        "countries_worked": [c.country for c in d.countries_worked],
        "markets_covered": d.markets_covered,
        "skills": d.skills_normalized,
        "highest_degree": d.highest_degree,
        "cfa": d.has_cfa,
    }


def _input_hash(req: EvalRequisition, cand: dict) -> str:
    blob = req.requisition.model_dump_json(exclude_none=True) + ranking_candidate_payload(cand)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


def _cache_key(req_id: str, cid: str) -> str:
    return f"{req_id}|{cid}|{RANKING_PROMPT_VERSION}"


def _load_cache(path: Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}


def _mean(vals: list[float]) -> float:
    return round(sum(vals) / len(vals), 4) if vals else 0.0


def run_ranking_eval(
    db_path: str | Path,
    out_dir: str | Path,
    client=None,
    requisitions_path: str | Path = DEFAULT_REQUISITIONS,
    embedder=None,
) -> tuple[Path, dict]:
    conn = connect(db_path)
    reqs = load_requisitions(requisitions_path)
    cache_path = Path(out_dir) / CACHE_NAME
    cache = _load_cache(cache_path)
    n_candidates = conn.execute("SELECT COUNT(*) FROM candidates").fetchone()[0]
    attempts: list[Attempt] = []
    items, judged_now = [], 0

    for req in reqs:
        pool = {}
        ranked = {}
        for variant in VARIANTS:
            res = search(
                conn, req.requisition, _weights(variant), embedder=embedder, top_k=max(n_candidates, 10)
            )
            ranked[variant] = [r.candidate_id for r in res]
            pool = {r.candidate_id: r for r in res}
        payloads = [
            _candidate_payload(get_candidate(conn, cid)) for cid in sorted(ranked["full"])
        ]
        hashes = {p["candidate_id"]: _input_hash(req, p) for p in payloads}
        cached = {
            p["candidate_id"]: cache.get(_cache_key(req.id, p["candidate_id"]))
            for p in payloads
        }
        hit = all(c and c["input_hash"] == hashes[cid] for cid, c in cached.items())
        if payloads and not hit:
            if client is None:
                client = default_client()
            judgement = judge_ranking(req, payloads, client, attempts)
            judged_now += 1
            for g in judgement.grades:
                cache[_cache_key(req.id, g.candidate_id)] = {
                    "grade": g.grade, "reason": g.reason, "input_hash": hashes[g.candidate_id],
                }
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            cache_path.write_text(json.dumps(cache, indent=1, ensure_ascii=False), encoding="utf-8")
        grades = {cid: cache[_cache_key(req.id, cid)]["grade"] for cid in ranked["full"]}
        row = {
            "requisition_id": req.id, "title": req.title, "pool_size": len(grades),
            "relevant_in_pool": sum(g >= 1 for g in grades.values()),
            "from_cache": hit,
        }
        for variant, ids in ranked.items():
            row[f"{variant}_ndcg5"] = round(ndcg_at_k(ids, grades, 5), 4)
            row[f"{variant}_p3"] = round(precision_at_k(ids, grades, 3), 4)
        row["candidates"] = [
            {
                "candidate_id": cid, "name": pool[cid].name,
                "grade": grades[cid], "reason": cache[_cache_key(req.id, cid)]["reason"],
                **{f"rank_{v}": ids.index(cid) + 1 for v, ids in ranked.items()},
            }
            for cid in ranked["full"]
        ]
        items.append(row)

    mean = {
        v: {
            "ndcg5": _mean([i[f"{v}_ndcg5"] for i in items]),
            "p3": _mean([i[f"{v}_p3"] for i in items]),
        }
        for v in VARIANTS
    }
    metrics = {
        "requisitions": len(items),
        "pool_sizes": {i["requisition_id"]: i["pool_size"] for i in items},
        "mean": mean,
        "judge_calls_this_run": judged_now,
        "tie_break": "score ties sort by candidate name, so a variant that scores many ties is "
        "ranked alphabetically within the tie",
    }
    body = {
        "task_model": "claude-sonnet-5-5", "judge_model": JUDGE_MODEL,
        "prompt_version": RANKING_PROMPT_VERSION, "metrics": metrics, "items": items,
        "cost_usd": round(judge_cost(attempts), 4), "tokens": token_totals(attempts),
    }
    return write_results(out_dir, "ranking", now_utc(), body), body

Overwriting src/resume_search/eval/ranking_eval.py


#### `src/resume_search/eval/report.py`

Formats suite results as plain-text tables.

In [20]:
%%writefile src/resume_search/eval/report.py
"""Plain-text tables for `eval run` and `eval show`."""

from __future__ import annotations

from resume_search.eval.ranking_eval import VARIANTS


def _table(header: list[str], rows: list[list[str]]) -> str:
    widths = [max(len(str(r[i])) for r in [header, *rows]) for i in range(len(header))]
    line = lambda r: "  ".join(str(c).ljust(w) for c, w in zip(r, widths, strict=True)).rstrip()
    return "\n".join([line(header), line(["-" * w for w in widths]), *(line(r) for r in rows)])


def _pct(x: float) -> str:
    return f"{x:.0%}"


def format_parsing(body: dict) -> str:
    m = body["metrics"]
    rows = [
        [k, int(v["total"]), int(v["correct"]), int(v["partial"]), int(v["wrong"]),
         int(v["hallucinated"]), int(v["missing"]), _pct(v["accuracy"]), _pct(v["soft_accuracy"])]
        for k, v in sorted(m["per_field"].items(), key=lambda kv: kv[0] == "overall")
    ]
    out = [
        (
            f"Parsing eval, judge {body['judge_model']}, prompt {body['prompt_version']}, "
            f"{m['candidates']} candidates, cost ${body['cost_usd']:.4f}"
        ),
        _table(["field", "n", "correct", "partial", "wrong", "halluc", "missing", "acc", "soft"], rows),
    ]
    ag = m["agreement"]
    if ag["overall"]:
        o = ag["overall"]
        out.append(
            f"\nJudge vs human ({'/'.join(ag['gold_status'])} gold), {o['n']} fields: "
            f"exact {_pct(o['exact'])}, within one level {_pct(o['within_one_level'])}"
        )
        out.append(_table(
            ["candidate", "n", "exact", "within_one"],
            [[n, a["n"], _pct(a["exact"]), _pct(a["within_one_level"])]
             for n, a in ag["per_candidate"].items()],
        ))
        for n, a in ag["per_candidate"].items():
            for d in a["disagreements"]:
                out.append(f"  {n} {d['field']}: human {d['human']}, judge {d['judge']}")
    return "\n".join(out)


def format_ranking(body: dict) -> str:
    m = body["metrics"]
    cols = [(v, v) for v in VARIANTS]
    header = ["requisition", "pool"] + [f"{v} {k}" for v, _ in cols for k in ("ndcg5", "p3")]
    rows = [
        [i["requisition_id"], i["pool_size"]]
        + [f"{i[f'{v}_{k}']:.3f}" for v, _ in cols for k in ("ndcg5", "p3")]
        for i in body["items"]
    ]
    rows.append(
        ["MEAN", ""] + [f"{m['mean'][v][k]:.3f}" for v, _ in cols for k in ("ndcg5", "p3")]
    )
    return (
        f"Ranking eval, judge {body['judge_model']}, prompt {body['prompt_version']}, "
        f"judge calls this run {m['judge_calls_this_run']}, cost ${body['cost_usd']:.4f}\n"
        + _table(header, rows)
    )


FORMATTERS = {"parsing": format_parsing, "ranking": format_ranking}

Overwriting src/resume_search/eval/report.py


#### `src/resume_search/eval/results.py`

Reads and writes the timestamped result files in `evals/results/`.

In [21]:
%%writefile src/resume_search/eval/results.py
"""Results files: evals/results/<suite>-<UTC timestamp>.json."""

from __future__ import annotations

import json
import re
from datetime import UTC, datetime
from pathlib import Path

SUITES = ("parsing", "ranking")
_NAME = re.compile(r"^(parsing|ranking)-\d{8}T\d{6}Z\.json$")


def now_utc() -> datetime:
    return datetime.now(UTC)


def write_results(out_dir: str | Path, suite: str, run_at: datetime, body: dict) -> Path:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{suite}-{run_at.strftime('%Y%m%dT%H%M%SZ')}.json"
    doc = {"suite": suite, "run_at": run_at.isoformat(timespec="seconds"), **body}
    path.write_text(json.dumps(doc, indent=2, ensure_ascii=False), encoding="utf-8")
    return path


def result_files(out_dir: str | Path, suite: str) -> list[Path]:
    out = Path(out_dir)
    if not out.is_dir():
        return []
    return sorted(
        p for p in out.iterdir() if (m := _NAME.match(p.name)) and m.group(1) == suite
    )


def latest_result(out_dir: str | Path, suite: str) -> tuple[Path, dict] | None:
    for path in reversed(result_files(out_dir, suite)):
        try:
            return path, json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
    return None

Overwriting src/resume_search/eval/results.py


#### `src/resume_search/extract/__init__.py`

`extract(path)` turns a DOCX or PDF into an `ExtractedText` with repaired text, quality numbers, and the extraction path.

In [22]:
%%writefile src/resume_search/extract/__init__.py
from __future__ import annotations

import hashlib
from pathlib import Path

from resume_search.extract.docx import extract_docx
from resume_search.extract.pdf import extract_pdf, render_pages
from resume_search.extract.quality import fuzzy_fix_tokens, taxonomy_terms, unknown_ratio
from resume_search.extract.repair import repair_text
from resume_search.extract.types import (
    ExtractedText,
    ExtractionPath,
    SourceFormat,
    TextQuality,
)

__all__ = ["ExtractedText", "TextQuality", "extract"]


def extract(path: str | Path) -> ExtractedText:
    path = Path(path)
    suffix = path.suffix.lower().lstrip(".")
    if suffix not in ("docx", "pdf"):
        raise ValueError(f"unsupported file type '.{suffix}' for {path.name}; expected .docx or .pdf")
    data = path.read_bytes()
    extra = taxonomy_terms()

    coverage, pngs = 0.0, []
    if suffix == "docx":
        raw, pages = extract_docx(path)
    else:
        raw, pages, coverage, pngs = extract_pdf(path)

    before = unknown_ratio(raw, extra)
    text, repairs = repair_text(raw)
    text, fixes = fuzzy_fix_tokens(text, extra)
    quality = TextQuality(
        unknown_ratio_before=before,
        unknown_ratio_after=unknown_ratio(text, extra),
        repairs=repairs + fixes,
        chars_per_page=len(text) / max(pages, 1),
        max_image_page_coverage=coverage,
    )
    vision = suffix == "pdf" and quality.needs_vision()
    if vision and not pngs:
        pngs = render_pages(path)
    return ExtractedText(
        source_file=path.name,
        file_sha256=hashlib.sha256(data).hexdigest(),
        file_bytes=len(data),
        format=SourceFormat(suffix),
        page_count=pages,
        text=text,
        quality=quality,
        extraction_path=ExtractionPath.vision if vision else ExtractionPath.text,
        page_images_png=tuple(pngs) if vision else (),
    )

Overwriting src/resume_search/extract/__init__.py


#### `src/resume_search/extract/docx.py`

Reads a DOCX body in document order and writes each table row as one line with cells joined by ` | `.

In [23]:
%%writefile src/resume_search/extract/docx.py
from __future__ import annotations

from pathlib import Path

from docx import Document
from docx.oxml.ns import qn
from docx.table import Table
from docx.text.paragraph import Paragraph

_MC_FALLBACK = "{http://schemas.openxmlformats.org/markup-compatibility/2006}Fallback"


def _row_line(row) -> str:
    cells: list[str] = []
    prev_tc = None
    for cell in row.cells:
        # python-docx yields the same cell object for every grid column a merged cell spans
        if cell._tc is prev_tc:
            continue
        prev_tc = cell._tc
        text = " ".join(p.text.strip() for p in cell.paragraphs if p.text.strip())
        if text and (not cells or cells[-1] != text):
            cells.append(text)
    return " | ".join(cells)


def _textbox_lines(doc) -> list[str]:
    """Floating text boxes (page furniture such as the name banner) come first.

    Word stores each box twice, as mc:Choice and a VML mc:Fallback, so the fallback copy
    is skipped. python-docx's paragraph.text never descends into them.
    """
    lines: list[str] = []
    for box in doc.element.body.iter(qn("w:txbxContent")):
        if any(a.tag == _MC_FALLBACK for a in box.iterancestors()):
            continue
        for p in box.iterchildren(qn("w:p")):
            text = Paragraph(p, doc).text.strip()
            if text and text != "-":
                lines.append(text)
    return lines


def extract_docx(path: str | Path) -> tuple[str, int]:
    doc = Document(str(path))
    lines = _textbox_lines(doc)
    for child in doc.element.body.iterchildren():
        if child.tag == qn("w:p"):
            lines.append(Paragraph(child, doc).text)
        elif child.tag == qn("w:tbl"):
            lines.extend(line for line in map(_row_line, Table(child, doc).rows) if line)
    return "\n".join(lines), 1

Overwriting src/resume_search/extract/docx.py


#### `src/resume_search/extract/pdf.py`

Reads PDF text per page with PyMuPDF, measures image coverage, and renders pages to PNG for the vision path.

In [24]:
%%writefile src/resume_search/extract/pdf.py
from __future__ import annotations

from pathlib import Path

import pymupdf

from resume_search.extract.types import TextQuality

RENDER_DPI = 150
PAGE_SEP = "\n\f\n"


def _image_coverage(page: pymupdf.Page) -> float:
    area = page.rect.width * page.rect.height
    if area <= 0:
        return 0.0
    covered = sum(
        (r & page.rect).get_area()
        for info in page.get_image_info()
        if not (r := pymupdf.Rect(info["bbox"])).is_empty
    )
    return min(covered / area, 1.0)


def render_pages(path: str | Path, dpi: int = RENDER_DPI) -> list[bytes]:
    with pymupdf.open(str(path)) as doc:
        return [page.get_pixmap(dpi=dpi).tobytes("png") for page in doc]


def extract_pdf(path: str | Path) -> tuple[str, int, float, list[bytes]]:
    """Pages are rendered here only for the triggers knowable from the raw PDF. The
    unknown-word trigger needs repaired text, so the caller renders via render_pages."""
    with pymupdf.open(str(path)) as doc:
        texts = [page.get_text("text") for page in doc]
        coverage = max((_image_coverage(p) for p in doc), default=0.0)
        n = doc.page_count
    text = PAGE_SEP.join(texts)
    sparse = TextQuality(0.0, 0.0, 0, len(text) / max(n, 1), coverage).needs_vision()
    return text, n, coverage, render_pages(path) if sparse else []

Overwriting src/resume_search/extract/pdf.py


#### `src/resume_search/extract/quality.py`

Computes the share of unknown words in a text and fixes close misspellings caused by bad glyphs.

In [25]:
%%writefile src/resume_search/extract/quality.py
from __future__ import annotations

import re

from rapidfuzz import fuzz, process
from wordfreq import zipf_frequency

from resume_search.taxonomy import taxonomy_terms, tokenize

__all__ = ["fuzzy_fix_tokens", "taxonomy_terms", "unknown_ratio"]

_DIGIT = re.compile(r"\d")
_FUZZY_CUTOFF = 90
_MIN_LEN = 3


def _is_known(token: str, extra_known: set[str] | frozenset[str]) -> bool:
    low = token.lower()
    return (
        bool(_DIGIT.search(token))
        or low in extra_known
        or zipf_frequency(low, "en") > 0
    )


def unknown_ratio(text: str, extra_known: set[str] | frozenset[str]) -> float:
    tokens = [t for t in tokenize(text) if len(t) >= _MIN_LEN]
    if not tokens:
        return 0.0
    return sum(not _is_known(t, extra_known) for t in tokens) / len(tokens)


def fuzzy_fix_tokens(text: str, extra_known: set[str] | frozenset[str]) -> tuple[str, int]:
    """Substitute unknown tokens carrying a non-ASCII char with a close known token.

    Candidates are the document's own known tokens plus extra_known. Only non-ASCII
    tokens are tried: repair already handled the known glyph maps, so what is left with
    a stray glyph is what a fuzzy match can plausibly recover.
    """
    tokens = [t for t in tokenize(text) if len(t) >= _MIN_LEN]
    known = {t.lower() for t in tokens if _is_known(t, extra_known) and t.isascii()}
    candidates = list(known | set(extra_known))
    fixes = 0

    def sub(m: re.Match[str]) -> str:
        nonlocal fixes
        tok = m.group(0)
        if len(tok) < _MIN_LEN or tok.isascii() or _is_known(tok, extra_known):
            return tok
        hit = process.extractOne(
            tok.lower(), candidates, scorer=fuzz.ratio, score_cutoff=_FUZZY_CUTOFF
        )
        if hit is None:
            return tok
        fixes += 1
        fixed = hit[0]
        return fixed.capitalize() if tok[0].isupper() else fixed

    out = re.sub(r"[^\W_]+(?:['’][^\W_]+)?", sub, text)
    return out, fixes

Overwriting src/resume_search/extract/quality.py


#### `src/resume_search/extract/repair.py`

Replaces ligature glyphs that PDF fonts emit (such as `Ɵ` for `ti`) and drops private-use characters.

In [26]:
%%writefile src/resume_search/extract/repair.py
from __future__ import annotations

import re
import unicodedata

REPAIR_MAP: dict[str, str] = {
    "Ɵ": "ti",  # U+019F, how the "ti" ligature glyph decodes in some PDF fonts
    "ﬀ": "ff",
    "ﬁ": "fi",
    "ﬂ": "fl",
    "ﬃ": "ffi",
    "ﬄ": "ffl",
    "ﬅ": "st",
    "ƞ": "tf",  # U+019E, the "tf" ligature in the same fonts
}
_PRIVATE_USE = re.compile("[-]")
_SPACES = re.compile(r"[ \t]+")
_BLANKS = re.compile(r"\n{4,}")


def repair_text(text: str) -> tuple[str, int]:
    count = 0
    for bad, good in REPAIR_MAP.items():
        count += text.count(bad)
        text = text.replace(bad, good)
    text, n = _PRIVATE_USE.subn("", text)
    count += n
    # NFKC would also expand ligatures, but the map runs first so repairs are counted
    text = unicodedata.normalize("NFKC", text)
    text = _SPACES.sub(" ", text)
    text = "\n".join(line.rstrip() for line in text.split("\n"))
    return _BLANKS.sub("\n\n\n", text), count

Overwriting src/resume_search/extract/repair.py


#### `src/resume_search/extract/types.py`

The `ExtractedText` and `TextQuality` data shapes, including the rule that decides when the vision path is needed.

In [27]:
%%writefile src/resume_search/extract/types.py
"""Data shape produced by extraction and consumed by parsing.

One ExtractedText per source file. Every downstream step (LLM parse, quality
metrics, FTS body, vision fallback) reads from this object, never from the file.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum


class SourceFormat(str, Enum):
    docx = "docx"
    pdf = "pdf"
    txt = "txt"
    md = "md"
    html = "html"


class ExtractionPath(str, Enum):
    text = "text"
    vision = "vision"


@dataclass(frozen=True)
class TextQuality:
    unknown_ratio_before: float
    unknown_ratio_after: float
    repairs: int
    chars_per_page: float
    max_image_page_coverage: float

    def needs_vision(self) -> bool:
        return (
            self.chars_per_page < 400
            or self.max_image_page_coverage > 0.5
            or self.unknown_ratio_after > 0.10
        )


@dataclass(frozen=True)
class ExtractedText:
    source_file: str
    file_sha256: str
    file_bytes: int
    format: SourceFormat
    page_count: int
    text: str
    quality: TextQuality
    extraction_path: ExtractionPath
    page_images_png: tuple[bytes, ...] = field(default=())

    @property
    def candidate_id(self) -> str:
        return self.file_sha256[:12]

Overwriting src/resume_search/extract/types.py


#### `src/resume_search/ingest/jobs.py`

The `parse_jobs` row and its state machine, and the idempotency check on file hash and prompt version.

In [28]:
%%writefile src/resume_search/ingest/jobs.py
"""Per-file job row and its state machine. Every status change goes through advance()."""

from __future__ import annotations

import sqlite3
from dataclasses import dataclass
from datetime import UTC, datetime
from enum import Enum


class JobStatus(str, Enum):
    queued = "queued"
    extracting = "extracting"
    parsing = "parsing"
    deriving = "deriving"
    indexing = "indexing"
    ok = "ok"
    failed = "failed"
    skipped = "skipped"


S = JobStatus
TERMINAL = frozenset({S.ok, S.failed, S.skipped})
_NEXT = {
    S.queued: {S.extracting, S.skipped, S.failed},
    S.extracting: {S.parsing, S.failed},
    S.parsing: {S.deriving, S.failed},
    S.deriving: {S.indexing, S.failed},
    S.indexing: {S.ok, S.failed},
}

_COLUMNS = (
    "attempts_json", "input_tokens", "output_tokens", "cost_usd", "latency_ms", "error",
    "finished_at",
)


@dataclass
class Job:
    file_sha256: str
    source_file: str
    prompt_version: str
    status: JobStatus = S.queued
    job_id: int | None = None
    candidate_id: str | None = None
    attempts_json: str | None = None
    input_tokens: int | None = None
    output_tokens: int | None = None
    cost_usd: float | None = None
    latency_ms: int | None = None
    error: str | None = None
    queued_at: str | None = None
    finished_at: str | None = None


JobResult = Job


def _now() -> str:
    return datetime.now(UTC).isoformat(timespec="seconds")


def advance(conn: sqlite3.Connection, job: Job, status: JobStatus, **fields) -> Job:
    """Move the job to `status`, apply `fields`, and write the row. A job with no row yet is
    inserted. Terminal statuses stamp finished_at."""
    if job.job_id is not None and status not in _NEXT.get(job.status, set()):
        raise ValueError(f"illegal job transition {job.status.value} -> {status.value}")
    for k, v in fields.items():
        if k not in _COLUMNS and k != "candidate_id":
            raise TypeError(f"unknown job field '{k}'")
        setattr(job, k, v)
    job.status = status
    if status in TERMINAL:
        job.finished_at = _now()
    with conn:
        if job.job_id is None:
            job.queued_at = _now()
            cur = conn.execute(
                "INSERT INTO parse_jobs (file_sha256, source_file, prompt_version, status, queued_at)"
                " VALUES (?,?,?,?,?)",
                (job.file_sha256, job.source_file, job.prompt_version, status.value, job.queued_at),
            )
            job.job_id = cur.lastrowid
        sets = ", ".join(f"{c} = ?" for c in ("status", *_COLUMNS))
        conn.execute(
            f"UPDATE parse_jobs SET {sets} WHERE job_id = ?",
            (status.value, *(getattr(job, c) for c in _COLUMNS), job.job_id),
        )
    return job


def already_ok(conn: sqlite3.Connection, file_sha256: str, prompt_version: str) -> bool:
    return (
        conn.execute(
            "SELECT 1 FROM parse_jobs WHERE file_sha256 = ? AND prompt_version = ? AND status = 'ok'",
            (file_sha256, prompt_version),
        ).fetchone()
        is not None
    )

Overwriting src/resume_search/ingest/jobs.py


#### `src/resume_search/ingest/pipeline.py`

`ingest()` runs extract, parse, derive, and index for many files with worker threads, and `rederive()` rebuilds derived data without the API.

In [29]:
%%writefile src/resume_search/ingest/pipeline.py
"""ingest(): sha256 -> skip check -> extract -> parse (worker threads) -> derive -> index (main
thread). SQLite connections stay on the main thread, so workers only return results."""

from __future__ import annotations

import hashlib
import json
import queue
import time
from collections.abc import Sequence
from concurrent.futures import Future, ThreadPoolExecutor, wait
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

from resume_search.derive import derive
from resume_search.derive.models import SCHEMA_VERSION, CandidateRecord
from resume_search.extract import ExtractedText, extract
from resume_search.ingest.jobs import Job, JobResult, JobStatus, advance, already_ok
from resume_search.parse import ParseOk, parse_resume
from resume_search.parse.client import OPUS, Attempt, ParseResult, verification_dict
from resume_search.parse.models import ParsedResume
from resume_search.parse.prompt import PROMPT_VERSION
from resume_search.search.vectors import Embedder, candidate_vectors, default_embedder
from resume_search.store.db import connect, list_candidates, load_taxonomy, upsert_candidate
from resume_search.store.export import export_csv, export_json
from resume_search.store.fts import fts_body
from resume_search.taxonomy import Taxonomy, load_taxonomy_data

S = JobStatus

# USD per million tokens: (input, output). Cache reads are $0.20 for both models; cache writes
# are billed at 1.25x the input price.
PRICES = {OPUS: (4.0, 20.0)}
SONNET_PRICE = (2.0, 10.0)
CACHE_READ_PRICE = 0.20


def attempt_cost(a: Attempt) -> float:
    p_in, p_out = PRICES.get(a.model, SONNET_PRICE)
    return (
        a.input_tokens * p_in
        + a.cache_creation_tokens * p_in * 1.25
        + a.cache_read_tokens * CACHE_READ_PRICE
        + a.output_tokens * p_out
    ) / 1e6


def build_meta(e: ExtractedText, result: ParseResult) -> dict:
    return {
        "candidate_id": e.candidate_id,
        "source_file": e.source_file,
        "file_sha256": e.file_sha256,
        "file_bytes": e.file_bytes,
        "page_count": e.page_count,
        "parsed_at": datetime.now(UTC).isoformat(timespec="seconds"),
        "extraction_path": e.extraction_path.value,
        "llm_model": result.attempts[-1].model,
        "prompt_version": PROMPT_VERSION,
        "schema_version": SCHEMA_VERSION,
        "parse_status": result.parse_status,
        "attempts": [asdict(a) for a in result.attempts],
        "verification": verification_dict(result.verification) if isinstance(result, ParseOk) else None,
        "text_quality": asdict(e.quality),
    }


def _accounting(result: ParseResult, latency_ms: int) -> dict:
    atts = result.attempts
    return {
        "attempts_json": json.dumps([asdict(a) for a in atts]),
        "input_tokens": sum(a.input_tokens + a.cache_read_tokens + a.cache_creation_tokens for a in atts),
        "output_tokens": sum(a.output_tokens for a in atts),
        "cost_usd": sum(attempt_cost(a) for a in atts),
        "latency_ms": latency_ms,
    }


def _work(i: int, path: str, events: queue.Queue) -> tuple[ExtractedText, ParseResult, int]:
    started = time.monotonic()
    events.put((i, S.extracting))
    e = extract(path)
    events.put((i, S.parsing))
    result = parse_resume(e)
    return e, result, round((time.monotonic() - started) * 1000)


def _index(
    conn, job: Job, e: ExtractedText, result: ParseOk, tax: Taxonomy, embedder: Embedder
) -> None:
    meta = build_meta(e, result)
    derived = derive(result.record, meta, tax)
    advance(conn, job, S.indexing)
    record = CandidateRecord(meta=meta, extracted=result.record, derived=derived)
    upsert_candidate(conn, record, fts_body(e.text), candidate_vectors(record, embedder))
    advance(conn, job, S.ok, candidate_id=e.candidate_id)


def ingest(
    paths: Sequence[str | Path],
    db_path: str | Path,
    *,
    force: bool = False,
    workers: int = 4,
    export_dir: str | Path | None = None,
    embedder: Embedder | None = None,
) -> list[JobResult]:
    conn = connect(db_path)
    load_taxonomy(conn)
    tax = load_taxonomy_data()
    embedder = embedder or default_embedder()
    events: queue.Queue = queue.Queue()
    jobs: list[Job] = []
    pending: dict[Future, int] = {}

    def drain() -> None:
        while True:
            try:
                i, status = events.get_nowait()
            except queue.Empty:
                return
            advance(conn, jobs[i], status)

    def fail(job: Job, exc: BaseException | str, **fields) -> None:
        msg = exc if isinstance(exc, str) else f"{type(exc).__name__}: {exc}"
        advance(conn, job, S.failed, error=msg, **fields)

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for i, path in enumerate(map(str, paths)):
            try:
                sha = hashlib.sha256(Path(path).read_bytes()).hexdigest()
            except OSError as exc:
                job = Job("", Path(path).name, PROMPT_VERSION)
                jobs.append(job)
                advance(conn, job, S.queued)
                fail(job, exc)
                continue
            job = Job(sha, Path(path).name, PROMPT_VERSION, candidate_id=sha[:12])
            jobs.append(job)
            advance(conn, job, S.queued)
            if not force and already_ok(conn, sha, PROMPT_VERSION):
                advance(conn, job, S.skipped)
                continue
            pending[pool.submit(_work, i, path, events)] = i

        while pending:
            done, _ = wait(pending, timeout=0.2, return_when="FIRST_COMPLETED")
            drain()
            for fut in done:
                job = jobs[pending.pop(fut)]
                try:
                    e, result, latency = fut.result()
                    acct = _accounting(result, latency)
                    if not isinstance(result, ParseOk):
                        fail(job, result.last_error, **acct)
                        continue
                    advance(conn, job, S.deriving, **acct)
                    _index(conn, job, e, result, tax, embedder)
                except Exception as exc:  # noqa: BLE001  one bad file must not stop the batch
                    fail(job, exc)

    out = Path(export_dir) if export_dir else Path(db_path).parent
    export_json(conn, out / "candidates.json")
    export_csv(conn, out / "candidates.csv")
    conn.close()
    return jobs


def rederive(
    db_path: str | Path, *, export_dir: str | Path | None = None, embedder: Embedder | None = None
) -> int:
    """Rebuild derived fields and vectors for every candidate from record_json, keeping the
    stored FTS body (the source text is not kept). No API calls."""
    conn = connect(db_path)
    load_taxonomy(conn)
    tax = load_taxonomy_data()
    embedder = embedder or default_embedder()
    records = list_candidates(conn)
    for r in records:
        cid = r.meta["candidate_id"]
        meta = {**r.meta, "schema_version": SCHEMA_VERSION}
        parsed = ParsedResume.model_validate(r.extracted.model_dump())
        new = CandidateRecord(meta=meta, extracted=parsed, derived=derive(parsed, meta, tax))
        row = conn.execute("SELECT body FROM candidate_fts WHERE candidate_id = ?", (cid,)).fetchone()
        upsert_candidate(conn, new, row["body"] if row else "", candidate_vectors(new, embedder))
    out = Path(export_dir) if export_dir else Path(db_path).parent
    export_json(conn, out / "candidates.json")
    export_csv(conn, out / "candidates.csv")
    conn.close()
    return len(records)

Overwriting src/resume_search/ingest/pipeline.py


#### `src/resume_search/parse/__init__.py`

Public surface of the parser: `parse_resume`, `ParseOk`, `ParseFailed`, and `ParsedResume`.

In [30]:
%%writefile src/resume_search/parse/__init__.py
from resume_search.parse.client import ParseFailed, ParseOk, parse_resume
from resume_search.parse.models import ParsedResume

__all__ = ["ParseFailed", "ParseOk", "ParsedResume", "parse_resume"]

Overwriting src/resume_search/parse/__init__.py


#### `src/resume_search/parse/client.py`

Runs the profile, experience, and assessment calls with constrained decoding, cross-verification, targeted regeneration, and the Sonnet-Sonnet-Opus retry ladder.

In [31]:
%%writefile src/resume_search/parse/client.py
from __future__ import annotations

import base64
import json
import os
import time
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, field
from datetime import date
from typing import ClassVar

import anthropic
import pydantic

from resume_search.extract.types import ExtractedText, ExtractionPath
from resume_search.parse import prompt, verify
from resume_search.parse.models import (
    Assessment,
    CertificationsOnly,
    ContactOnly,
    EducationItem,
    Experience,
    ExperienceItem,
    ExperienceList,
    NameOnly,
    ParsedResume,
    Profile,
    SkillsOnly,
    check_evidence,
    check_experience_evidence,
)

SONNET = "claude-sonnet-5-5"
OPUS = "claude-opus-5-5"
# (model, append the previous attempt's validation errors)
LADDER = ((SONNET, False), (SONNET, True), (OPUS, True))
MAX_TOKENS = 16000
MAIN_CALLS = ("profile", "experience", "assessment")


@dataclass(frozen=True)
class Attempt:
    model: str
    latency_ms: int
    input_tokens: int
    output_tokens: int
    error: str | None
    cache_read_tokens: int = 0
    cache_creation_tokens: int = 0
    call: str = ""


@dataclass(frozen=True)
class Verification:
    checks_run: int = 0
    disagreements: int = 0
    regenerated: int = 0
    unresolved: list[dict] = field(default_factory=list)
    details: list[dict] = field(default_factory=list)


@dataclass(frozen=True)
class ParseOk:
    record: ParsedResume
    attempts: list[Attempt]
    verification: Verification = field(default_factory=Verification)

    @property
    def parse_status(self) -> str:
        main = [a for a in self.attempts if a.call in MAIN_CALLS]
        if any(a.model == OPUS for a in main):
            return "ok_retry_opus"
        return "ok" if len(main) == len(MAIN_CALLS) else "ok_retry_sonnet"


@dataclass(frozen=True)
class ParseFailed:
    attempts: list[Attempt]
    last_error: str

    parse_status: ClassVar[str] = "failed"


ParseResult = ParseOk | ParseFailed


def _validation_lines(exc: pydantic.ValidationError) -> list[str]:
    return [f"{'.'.join(map(str, e['loc']))}: {e['msg']}" for e in exc.errors()]


def _content(extracted: ExtractedText, message: str) -> list[dict]:
    blocks: list[dict] = []
    if extracted.extraction_path is ExtractionPath.vision:
        blocks += [
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": base64.standard_b64encode(png).decode("ascii"),
                },
            }
            for png in extracted.page_images_png
        ]
    blocks.append({"type": "text", "text": message})
    return blocks


def default_client() -> anthropic.Anthropic:
    # Keys that are not workspace-scoped need the workspace id header; the SDK does not
    # read ANTHROPIC_WORKSPACE_ID itself.
    headers = {}
    if workspace := os.environ.get("ANTHROPIC_WORKSPACE_ID"):
        headers["anthropic-workspace-id"] = workspace
    return anthropic.Anthropic(max_retries=4, default_headers=headers)


def _attempt(
    client, model: str, system: str, content: list[dict], output_format: type[pydantic.BaseModel]
):
    return client.messages.parse(
        model=model,
        max_tokens=MAX_TOKENS,
        system=[{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": content}],
        output_format=output_format,
    )


def _run_ladder[T: pydantic.BaseModel](
    client,
    call: str,
    system: str,
    extracted: ExtractedText,
    message: str,
    output_format: type[T],
    validate: Callable[[T], list[str]],
    ladder=LADDER,
) -> tuple[T | None, list[Attempt]]:
    """One call on the Sonnet, Sonnet-with-errors, Opus-with-errors ladder."""
    attempts: list[Attempt] = []
    prev_errors: list[str] = []
    for model, with_errors in ladder:
        suffix = prompt.retry_suffix(prev_errors) if with_errors and prev_errors else ""
        started = time.monotonic()
        usage = None
        out: T | None = None
        errors: list[str] = []
        try:
            response = _attempt(
                client, model, system, _content(extracted, message + suffix), output_format
            )
            usage = response.usage
            out = response.parsed_output
            if out is None:
                errors = [f"no parsed output (stop_reason={response.stop_reason})"]
            else:
                errors = validate(out)
        except pydantic.ValidationError as exc:
            errors = _validation_lines(exc)
        except anthropic.RateLimitError as exc:
            errors = [f"RateLimitError: {exc}"]
        except anthropic.APIStatusError as exc:
            errors = [f"APIStatusError {exc.status_code}: {exc.message}"]
        except anthropic.APIConnectionError as exc:
            errors = [f"APIConnectionError: {exc}"]
        attempts.append(
            Attempt(
                model=model,
                latency_ms=int((time.monotonic() - started) * 1000),
                input_tokens=usage.input_tokens if usage else 0,
                output_tokens=usage.output_tokens if usage else 0,
                error="; ".join(errors) if errors else None,
                cache_read_tokens=(usage.cache_read_input_tokens or 0) if usage else 0,
                cache_creation_tokens=(usage.cache_creation_input_tokens or 0) if usage else 0,
                call=call,
            )
        )
        if not errors and out is not None:
            return out, attempts
        # An API failure carries nothing the model could fix, so the next rung gets no suffix.
        prev_errors = [] if usage is None else errors
    return None, attempts


# Regeneration ---------------------------------------------------------------------------------

_REGEN_MODEL: dict[str, type[pydantic.BaseModel]] = {
    "name": NameOnly,
    "contact": ContactOnly,
    "certifications": CertificationsOnly,
    "skills": SkillsOnly,
    "experience_item": ExperienceItem,
    "education_item": EducationItem,
    "experience": ExperienceList,
}
_REGEN_SYSTEM = {
    "experience_item": "experience",
    "experience": "experience",
}


@dataclass
class _State:
    profile: Profile
    experience: list[Experience]


def _previous_json(state: _State, target: str, index: int | None) -> str:
    match target:
        case "name":
            return json.dumps({"name": state.profile.name})
        case "contact":
            return state.profile.model_dump_json(include={"contact"})
        case "certifications":
            return state.profile.model_dump_json(include={"certifications"})
        case "skills":
            return state.profile.model_dump_json(include={"skills"})
        case "experience_item":
            return state.experience[index].model_dump_json()
        case "education_item":
            return state.profile.education[index].model_dump_json()
        case _:
            return ExperienceList(experience=state.experience).model_dump_json()


def _apply(state: _State, target: str, index: int | None, out: pydantic.BaseModel) -> _State:
    match target:
        case "name" | "contact" | "certifications" | "skills":
            update = {name: getattr(out, name) for name in type(out).model_fields}
            return _State(state.profile.model_copy(update=update), state.experience)
        case "experience_item":
            exps = list(state.experience)
            exps[index] = out
            return _State(state.profile, exps)
        case "education_item":
            edu = list(state.profile.education)
            edu[index] = out
            return _State(state.profile.model_copy(update={"education": edu}), state.experience)
        case _:
            return _State(state.profile, list(out.experience))


def _finding(d: verify.Disagreement) -> str:
    where = f' at: "{d.evidence_snippet}"' if d.evidence_snippet else ""
    return (
        f"{d.field}: the resume text deterministically yields {json.dumps(d.text_value)}"
        f"{where}; your value was {json.dumps(d.llm_value)}."
    )


def _regenerate(
    client,
    extracted: ExtractedText,
    state: _State,
    target: str,
    index: int | None,
    group: list[verify.Disagreement],
) -> tuple[_State | None, list[Attempt]]:
    system_key = _REGEN_SYSTEM.get(target, "profile")
    message = (
        prompt.profile_message(extracted)
        if system_key == "profile"
        else prompt.experience_message(extracted)
    ) + prompt.regen_suffix(
        _previous_json(state, target, index), [_finding(d) for d in group]
    )
    model_cls = _REGEN_MODEL[target]

    def validate(out):
        if target == "experience_item":
            return check_experience_evidence([out], extracted.text)
        if target == "experience":
            return check_experience_evidence(out.experience, extracted.text)
        return []

    out, attempts = _run_ladder(
        client, f"regen:{target}", prompt.SYSTEM_PROMPTS[system_key], extracted, message,
        model_cls, validate, ladder=LADDER[:1],
    )
    return (_apply(state, target, index, out) if out is not None else None), attempts


def _verify_and_repair(
    client, extracted: ExtractedText, state: _State
) -> tuple[_State, Verification, list[Attempt]]:
    text, filename = extracted.text, extracted.source_file
    report = verify.verify(state.profile, state.experience, text, filename)
    groups: dict[tuple[str, int | None], list[verify.Disagreement]] = {}
    for d in report.disagreements:
        if d.regen_target == "experience_item" and d.index is None:
            continue
        groups.setdefault((d.regen_target, d.index), []).append(d)
    attempts: list[Attempt] = []
    unresolved: list[dict] = []
    regenerated = 0
    for (target, index), group in groups.items():
        regenerated += 1
        candidate, regen_attempts = _regenerate(client, extracted, state, target, index, group)
        attempts += regen_attempts
        fields = frozenset(d.field for d in group)
        remaining = group
        if candidate is not None:
            recheck = verify.verify(
                candidate.profile, candidate.experience, text, filename, only=fields
            )
            # Keep a regeneration only when it fixed something; otherwise keep the LLM value.
            if len(recheck.disagreements) < len(group):
                state, remaining = candidate, recheck.disagreements
        unresolved += [
            {"field": d.field, "llm_value": d.llm_value, "text_value": d.text_value}
            for d in remaining
        ]
    details = [
        {"field": d.field, "llm_value": d.llm_value, "text_value": d.text_value}
        for d in report.disagreements
    ]
    summary = Verification(
        checks_run=report.checks_run,
        disagreements=len(report.disagreements),
        regenerated=regenerated,
        unresolved=unresolved,
        details=details,
    )
    return state, summary, attempts


def _assemble(state: _State, assessment: Assessment) -> ParsedResume:
    return ParsedResume.model_validate(
        {
            **state.profile.model_dump(),
            "experience": [e.model_dump() for e in state.experience],
            **assessment.model_dump(),
        }
    )


def parse_resume(
    extracted: ExtractedText, client: anthropic.Anthropic | None = None
) -> ParseResult:
    client = client or default_client()
    with ThreadPoolExecutor(max_workers=2) as pool:
        f_profile = pool.submit(
            _run_ladder, client, "profile", prompt.PROFILE_PROMPT, extracted,
            prompt.profile_message(extracted), Profile, lambda _: [],
        )
        f_exp = pool.submit(
            _run_ladder, client, "experience", prompt.EXPERIENCE_PROMPT, extracted,
            prompt.experience_message(extracted), ExperienceList,
            lambda out: check_experience_evidence(out.experience, extracted.text),
        )
        profile, profile_attempts = f_profile.result()
        exp_list, exp_attempts = f_exp.result()
    attempts = profile_attempts + exp_attempts
    if profile is None or exp_list is None:
        failed, last = ("profile", profile_attempts) if profile is None else ("experience", exp_attempts)
        return ParseFailed(attempts, f"{failed} call: {last[-1].error or 'unknown error'}")

    state, verification, regen_attempts = _verify_and_repair(
        client, extracted, _State(profile, exp_list.experience)
    )
    attempts += regen_attempts

    message = prompt.assessment_message(
        extracted,
        state.profile.model_dump_json(),
        ExperienceList(experience=state.experience).model_dump_json(),
        date.today().isoformat(),
    )

    def validate(out: Assessment) -> list[str]:
        return check_evidence(_assemble(state, out), extracted.text)

    assessment, assess_attempts = _run_ladder(
        client, "assessment", prompt.ASSESSMENT_PROMPT, extracted, message, Assessment, validate
    )
    attempts += assess_attempts
    if assessment is None:
        return ParseFailed(attempts, f"assessment call: {attempts[-1].error or 'unknown error'}")
    return ParseOk(_assemble(state, assessment), attempts, verification)


def verification_dict(v: Verification) -> dict:
    return asdict(v)

Overwriting src/resume_search/parse/client.py


#### `src/resume_search/parse/models.py`

The Pydantic models for LLM output, with validators for dates, ISO-2 countries, and verbatim evidence quotes.

In [32]:
%%writefile src/resume_search/parse/models.py
"""Pydantic models for the LLM output (design section 4.1)."""

from __future__ import annotations

import re
from enum import Enum

from pydantic import BaseModel, Field, field_validator, model_validator

from resume_search.taxonomy import iso2_codes, market_labels


class Relevance(str, Enum):
    core = "core"
    adjacent = "adjacent"
    unrelated = "unrelated"


class EmploymentType(str, Enum):
    full_time = "full_time"
    internship = "internship"
    part_time = "part_time"
    academic = "academic"


class FirmType(str, Enum):
    hedge_fund = "Hedge fund"
    asset_manager = "Asset manager"
    sell_side_bank = "Sell-side bank"
    sell_side_broker = "Sell-side broker"
    pe_vc = "PE/VC"
    consulting = "Consulting"
    corporate = "Corporate"
    academic = "Academic"
    other = "Other"


class AssetClass(str, Enum):
    equities = "Equities"
    fixed_income = "Fixed Income"
    credit = "Credit"
    fx = "FX"
    commodities = "Commodities"
    derivatives = "Derivatives"
    multi_asset = "Multi-asset"


class Sector(str, Enum):
    technology = "Technology"
    healthcare = "Healthcare"
    financials = "Financials"
    energy = "Energy"
    industrials = "Industrials"
    consumer = "Consumer"
    credit = "Credit"
    macro = "Macro"
    other = "Other"


class Approach(str, Enum):
    fundamental = "Fundamental"
    systematic = "Systematic"
    both = "Both"
    unclear = "Unclear"


_DATE = re.compile(r"^\d{4}(-(0[1-9]|1[0-2]))?$")
_LONG_DIGITS = re.compile(r"\d{7,}")
_WS = re.compile(r"\s+")
_QUOTE_FOLD = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"', "–": "-", "—": "-"})


def _check_date(v: str | None) -> str | None:
    if v is not None and not _DATE.match(v):
        raise ValueError(f"date '{v}' must be YYYY or YYYY-MM")
    return v


def _check_country(v: str | None) -> str | None:
    if v is not None and v not in iso2_codes():
        raise ValueError(f"country '{v}' is not an ISO-2 code in taxonomy/countries.yaml")
    return v


def _check_order(start: str | None, end: str | None) -> None:
    if start and end:
        n = min(len(start), len(end))  # compare at the coarser precision
        if start[:n] > end[:n]:
            raise ValueError(f"start {start} is after end {end}")


class Evidence(BaseModel):
    quote: str = Field(max_length=200, description="Verbatim span from the resume")


class Labeled(BaseModel):
    label: str
    evidence: Evidence


class Contact(BaseModel):
    email: str | None
    phone: str | None
    location_text: str | None


class Education(BaseModel):
    institution: str
    degree: str | None
    degree_level: str | None
    field: str | None
    start: str | None
    end: str | None
    country: str | None

    _dates = field_validator("start", "end")(_check_date)
    _country = field_validator("country")(_check_country)

    @model_validator(mode="after")
    def _order(self) -> Education:
        _check_order(self.start, self.end)
        return self


class Experience(BaseModel):
    employer: str
    title: str
    location_text: str | None
    country: str | None
    start: str | None
    end: str | None
    is_current: bool
    employment_type: EmploymentType
    relevance: Relevance
    firm_type: FirmType
    asset_classes: list[AssetClass] = []
    sectors: list[Labeled] = []
    other_sector_tags: list[str] = []
    markets_covered: list[Labeled] = []
    duration_months: int | None
    bullets: list[str]

    _dates = field_validator("start", "end")(_check_date)
    _country = field_validator("country")(_check_country)

    @model_validator(mode="after")
    def _consistent(self) -> Experience:
        # end may be null for a finished role whose dates the resume omits (duration-only
        # entries), so the rule is one-directional: current implies no end.
        if self.is_current and self.end is not None:
            raise ValueError("end must be null when is_current is true")
        _check_order(self.start, self.end)
        sector_names = {s.value for s in Sector}
        for s in self.sectors:
            if s.label not in sector_names:
                raise ValueError(f"sector label '{s.label}' is not one of {sorted(sector_names)}")
        allowed = iso2_codes() | market_labels()
        for m in self.markets_covered:
            if m.label not in allowed:
                raise ValueError(f"markets_covered label '{m.label}' is not an ISO-2 code or market label")
        return self


class Skills(BaseModel):
    programming: list[str] = []
    tools: list[str] = []
    domain: list[str] = []


class Language(BaseModel):
    name: str
    level: str | None


class Profile(BaseModel):
    name: str
    contact: Contact
    education: list[Education]
    skills: Skills
    certifications: list[str] = []
    languages: list[Language] = []


class ExperienceList(BaseModel):
    experience: list[Experience]


class Assessment(BaseModel):
    investment_approach: Labeled
    search_summary: str = Field(min_length=200, max_length=700)

    @field_validator("search_summary")
    @classmethod
    def _no_contact_leak(cls, v: str) -> str:
        if "@" in v or _LONG_DIGITS.search(v):
            raise ValueError("search_summary must not contain '@' or a run of 7+ digits")
        return v

    @model_validator(mode="after")
    def _approach_label(self) -> Assessment:
        valid = {a.value for a in Approach}
        if self.investment_approach.label not in valid:
            raise ValueError(f"investment_approach label must be one of {sorted(valid)}")
        return self


class ParsedResume(Assessment, ExperienceList, Profile):
    """The stored record. Bases are ordered so fields dump as profile, experience, assessment."""


# Smallest output models for targeted regeneration (design 5.5).
class NameOnly(BaseModel):
    name: str


class ContactOnly(BaseModel):
    contact: Contact


class CertificationsOnly(BaseModel):
    certifications: list[str]


class SkillsOnly(BaseModel):
    skills: Skills


ExperienceItem = Experience
EducationItem = Education


def _norm(s: str) -> str:
    return _WS.sub(" ", s.translate(_QUOTE_FOLD)).strip()


def _evidence_errors(
    text: str, approach: Labeled | None, experiences: list[Experience]
) -> list[str]:
    haystack = _norm(text)
    errors: list[str] = []

    def check(path: str, labeled: Labeled) -> None:
        if _norm(labeled.evidence.quote) not in haystack:
            errors.append(f"{path}.evidence.quote not found verbatim in resume: "
                          f"'{labeled.evidence.quote}'")

    if approach is not None:
        check("investment_approach", approach)
    for i, exp in enumerate(experiences):
        for j, s in enumerate(exp.sectors):
            check(f"experience[{i}].sectors[{j}]", s)
        for j, m in enumerate(exp.markets_covered):
            check(f"experience[{i}].markets_covered[{j}]", m)
    return errors


def check_evidence(parsed: ParsedResume, text: str) -> list[str]:
    """Every Evidence.quote must be a substring of the resume after whitespace folding."""
    return _evidence_errors(text, parsed.investment_approach, parsed.experience)


def check_experience_evidence(experiences: list[Experience], text: str) -> list[str]:
    return _evidence_errors(text, None, experiences)

Overwriting src/resume_search/parse/models.py


#### `src/resume_search/parse/prompt.py`

The versioned system prompts and the user-message builders, rendered from the taxonomy files.

In [33]:
%%writefile src/resume_search/parse/prompt.py
from __future__ import annotations

from html import escape

from resume_search import taxonomy
from resume_search.extract.types import ExtractedText

PROMPT_VERSION = "v2"

_COMMON = """\
You extract structured data from a single candidate resume for a hedge fund
business-development team. The reply is constrained to the output schema.

Common rules
- Copy facts; never infer. If a value is not stated, use null or an empty list.
- Dates: YYYY-MM when the month is given, YYYY otherwise. "Present", "till
  now", "since X" mean is_current=true and end=null. Fix obvious typos in
  month names.
- Country: ISO-2 code of where the role or school was based, from the
  location on that line or the contact block. Do not output regions.
  Allowed codes: {iso2}.
- Every evidence quote must be a verbatim span of the resume text, at most
  200 characters.
"""

_PROFILE = """\

Task: the candidate profile (name, contact, education, skills, certifications,
languages). Ignore work history.
- The user message starts with a <source filename="..."/> tag. The filename
  corroborates name, certifications and sector hints; it is never the sole
  source for anything except the name.
- name: the candidate's full name, without suffixes such as CFA.
- contact: email, phone as written, and location_text as written.
- education: one entry per degree or school programme. degree_level is one of
  Bachelors, Masters, MBA, PhD, Other, exactly as spelled. MBA is MBA. PGDM, an
  integrated M.Tech and a French "Diplôme d'Ingénieur" are Masters. MBBS,
  diplomas, high school and board exams are Other.
- skills: programming languages, tools and software, and domain methods
  ("DCF", "factor models", "options pricing") as the resume names them. Include
  every one named anywhere in the resume, in experience bullets and coursework as
  well as in a skills section.
- certifications: professional credentials such as CFA, FRM, CAIA, CPA, ACCA,
  with a level or "candidate" status when stated ("CFA Level II candidate").
  A competition or event run by an institute is not a certification.
- languages: spoken languages with level when stated.
"""

_EXPERIENCE = """\

Task: the work history, one entry per role, in the resume's order.
- markets_covered: fill ONLY when the resume states what markets or
  geographies the person covered (e.g. "Indian healthcare", "Greater China
  markets"). Do not infer from employer location. Give the quote. The label
  is an ISO-2 code or one of: {market_labels}.
- Sectors: choose from {{{sectors}}}; put narrower names
  ("Specialty Chemicals", "MedTech") in other_sector_tags. Give a quote for
  each sector label.
- relevance: core = investment research, trading, portfolio management,
  quant research/development, risk, at a financial firm; adjacent =
  consulting, corporate finance, data or software work outside finance;
  unrelated = everything else (teaching, clubs, retail).
- employment_type: internship when the title or text says intern, summer
  analyst, or the role is under 6 months while studying.
- firm_type: {{{firm_types}}}. Use your knowledge of the firm when the resume
  does not say.
- asset_classes: {{{asset_classes}}}, only those the role worked on.
- duration_months: whole months, filled only when the resume states a
  duration for the role in words. Years times 12 plus months; weeks divided
  by 4, rounded down. Examples: "8 years 10 months" is 106, "3 Months" is 3,
  "8 weeks" is 2. Null when the resume gives no duration text, even if
  dates let you compute one.
- Keep bullets verbatim, one string each, bullet characters removed.
"""

_ASSESSMENT = """\

Task: the investment approach and the search summary, written from the
verified profile and experience JSON in the user message. The resume text is
there only so approach evidence can be quoted from it.
- investment_approach: label is one of {{{approaches}}}. Systematic when the
  work is model-, factor-, or code-driven trading or research; Fundamental
  when it is company analysis, valuation, coverage; Both when clearly both;
  Unclear otherwise. The evidence quote is the deciding span, copied verbatim
  from the resume text.
- search_summary: 60-90 words, slots in this fixed order, as flowing
  sentences with no slot numbers.
  1. Seniority band and approach, with relevant years in parentheses, for
     example "Junior systematic equity analyst (2.3y relevant)". Relevant
     years are the full-time roles with relevance core or adjacent, overlapping
     ranges merged, using duration_months where a role has no dates, and "today"
     from the user message as the end of a current role. Bands: Intern/Student
     under 6 months, Junior 6 to 24, Mid 24 to 60, Senior 60 to 120, Principal
     over 120 months.
  2. Asset classes and sectors.
  3. Markets covered, only if stated in the JSON.
  4. Current or most recent employer, firm type in parentheses, and title.
  5. Two or three standout skills, tools or achievements, with numbers when
     present.
  6. Highest degree and institution, then certifications.
  No contact details. No evaluative adjectives such as "passionate" or
  "strong". Use only facts in the JSON.
"""


def _common() -> str:
    return _COMMON.format(iso2=", ".join(sorted(taxonomy.iso2_codes())))


def _render_profile() -> str:
    return _common() + _PROFILE


def _render_experience() -> str:
    return _common() + _EXPERIENCE.format(
        market_labels=", ".join(sorted(taxonomy.market_labels())),
        sectors=", ".join(taxonomy.sectors()),
        firm_types=", ".join(taxonomy.firm_types()),
        asset_classes=", ".join(taxonomy.asset_classes()),
    )


def _render_assessment() -> str:
    from resume_search.parse.models import Approach

    return _common() + _ASSESSMENT.format(approaches=", ".join(a.value for a in Approach))


PROFILE_PROMPT = _render_profile()
EXPERIENCE_PROMPT = _render_experience()
ASSESSMENT_PROMPT = _render_assessment()

# Regeneration reuses the call's own system prompt, so it shares the prompt cache.
SYSTEM_PROMPTS = {"profile": PROFILE_PROMPT, "experience": EXPERIENCE_PROMPT}


def source_tag(extracted: ExtractedText) -> str:
    return (
        f'<source filename="{escape(extracted.source_file, quote=True)}" '
        f'pages="{extracted.page_count}" extraction="{extracted.extraction_path.value}"/>'
    )


def resume_block(extracted: ExtractedText) -> str:
    return f"<resume>\n{extracted.text}\n</resume>"


def profile_message(extracted: ExtractedText) -> str:
    return f"{source_tag(extracted)}\n{resume_block(extracted)}"


def experience_message(extracted: ExtractedText) -> str:
    return resume_block(extracted)


def assessment_message(
    extracted: ExtractedText, profile_json: str, experience_json: str, today: str
) -> str:
    return (
        f"<today>{today}</today>\n"
        f"<verified_profile>\n{profile_json}\n</verified_profile>\n"
        f"<verified_experience>\n{experience_json}\n</verified_experience>\n"
        f"{resume_block(extracted)}"
    )


def retry_suffix(errors: list[str]) -> str:
    return (
        "\n<validation_errors>\n"
        + "\n".join(errors)
        + "\n</validation_errors>\n"
        + "Fix only these and return the full object again."
    )


def regen_suffix(previous_json: str, findings: list[str]) -> str:
    return (
        f"\n<previous_value>\n{previous_json}\n</previous_value>\n"
        "<verification_findings>\n"
        + "\n".join(findings)
        + "\n</verification_findings>\n"
        "A deterministic check of the resume text disagrees with the previous value. "
        "Re-read the resume, correct the value if the finding is right, and keep it if the "
        "resume supports it. Return only the requested object."
    )

Overwriting src/resume_search/parse/prompt.py


#### `src/resume_search/parse/verify.py`

A registry of checks that compare the LLM's fields against deterministic text parsing and name the field to regenerate on disagreement.

In [34]:
%%writefile src/resume_search/parse/verify.py
"""Cross-verification of LLM output against deterministic text parsing (design 5.5).

The registry CHECKS is data: adding a check is adding a row. A row whose `field` contains
`[i]` runs once per list item; `anchor_key` names the sibling key (employer, institution) whose
value lets `extract` find that item in the text.
"""

from __future__ import annotations

import re
import unicodedata
from collections.abc import Callable
from dataclasses import dataclass, field
from functools import cache
from pathlib import Path
from typing import Any

from rapidfuzz import fuzz

from resume_search import taxonomy
from resume_search.parse.models import ExperienceList, Profile

_WS = re.compile(r"\s+")


@dataclass(frozen=True)
class Found:
    value: Any
    snippet: str = ""


@dataclass(frozen=True)
class Check:
    field: str
    extract: Callable[[str, str, str | None], Found]
    compare: Callable[[Any, Any], bool]
    regen_target: str
    anchor_key: str | None = None
    applies: Callable[[dict], bool] | None = None


@dataclass(frozen=True)
class Disagreement:
    field: str
    llm_value: Any
    text_value: Any
    evidence_snippet: str
    regen_target: str
    index: int | None = None


@dataclass(frozen=True)
class VerificationReport:
    checks_run: int
    disagreements: list[Disagreement] = field(default_factory=list)


def _snip(text: str, start: int, end: int, pad: int = 50) -> str:
    return _WS.sub(" ", text[max(0, start - pad) : end + pad]).strip()


def _plain(s: str) -> str:
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower()


# name ---------------------------------------------------------------------------------------

_FILENAME_NOISE = {
    "resume", "cv", "curriculum", "vitae", "cfa", "frm", "caia", "cpa", "acca", "final", "updated",
}


def _name_candidates(text: str, filename: str) -> list[str]:
    stem = re.sub(r"[_\-.,()\[\]]+", " ", Path(filename).stem)
    tokens = [
        t for t in stem.split()
        if t.lower() not in _FILENAME_NOISE and not re.fullmatch(r"\d{4,8}", t)
    ]
    cands = [" ".join(tokens)] if tokens else []
    first = next((ln.strip() for ln in text.splitlines() if ln.strip()), "")
    if first:
        cands.append(first)
    return cands


def _extract_name(text: str, filename: str, anchor: str | None = None) -> Found:
    cands = _name_candidates(text, filename)
    return Found(cands, " | ".join(cands))


def _compare_name(llm: str | None, cands: list[str]) -> bool:
    if not llm:
        return False
    return any(fuzz.token_set_ratio(_plain(llm), _plain(c)) >= 80 for c in cands)


# email and phone ----------------------------------------------------------------------------

_EMAIL = re.compile(r"[\w.+\-]+@[\w\-]+(?:\.[\w\-]+)*")
_NOT_PHONE_LABEL = re.compile(r"(?:issn|isbn|doi|pmid)\W*$", re.IGNORECASE)
_PHONE = re.compile(r"(?<![\w.])(?:\+|00)?\d[\d ()\t.\-]{5,}\d(?![\w])")


def _extract_email(text: str, filename: str, anchor: str | None = None) -> Found:
    matches = list(_EMAIL.finditer(text))
    emails = sorted({m.group().lower() for m in matches})
    return Found(emails, _snip(text, matches[0].start(), matches[0].end()) if matches else "")


def _compare_email(llm: str | None, emails: list[str]) -> bool:
    if llm is None:
        return not emails
    return llm.strip().lower() in emails


def _digits(s: str) -> str:
    return re.sub(r"\D", "", s)


def _is_year_run(candidate: str) -> bool:
    groups = re.findall(r"\d+", candidate)
    return all(re.fullmatch(r"(?:19|20)\d{2}", g) for g in groups)


def _phone_candidates(text: str) -> list[re.Match]:
    out = []
    for m in _PHONE.finditer(text):
        n = len(_digits(m.group()))
        labelled = _NOT_PHONE_LABEL.search(text[max(0, m.start() - 8) : m.start()])
        if 7 <= n <= 15 and not _is_year_run(m.group()) and not labelled:
            out.append(m)
    return out


def _extract_phone(text: str, filename: str, anchor: str | None = None) -> Found:
    matches = _phone_candidates(text)
    nums = [m.group().strip() for m in matches]
    return Found(nums, _snip(text, matches[0].start(), matches[0].end()) if matches else "")


def _compare_phone(llm: str | None, nums: list[str]) -> bool:
    if llm is None:
        return not nums
    return _digits(llm) in {_digits(n) for n in nums}


# certifications -----------------------------------------------------------------------------

_CERTS: dict[str, re.Pattern] = {
    "CFA": re.compile(
        r"\bCFA\b(?!\s+(?:Institute|Society|Research|Challenge))|Chartered Financial Analyst",
    ),
    "FRM": re.compile(r"\bFRM\b|Financial Risk Manager"),
    "CAIA": re.compile(r"\bCAIA\b|Chartered Alternative Investment Analyst"),
    "CPA": re.compile(r"\bCPA\b|Certified Public Accountant"),
    "ACCA": re.compile(r"\bACCA\b"),
}


def _cert_tokens(s: str) -> set[str]:
    return {name for name, rx in _CERTS.items() if rx.search(s)}


def _extract_certs(text: str, filename: str, anchor: str | None = None) -> Found:
    tokens = _cert_tokens(text) | _cert_tokens(Path(filename).stem)
    snippet = ""
    for name in sorted(tokens):
        m = _CERTS[name].search(text)
        if m:
            snippet = _snip(text, m.start(), m.end())
            break
    return Found(sorted(tokens), snippet or Path(filename).stem)


def _compare_certs(llm: list[str], tokens: list[str]) -> bool:
    llm_tokens: set[str] = set()
    for c in llm:
        llm_tokens |= _cert_tokens(c)
    return llm_tokens == set(tokens)


# skills -------------------------------------------------------------------------------------


@cache
def _skill_matchers() -> tuple[re.Pattern, re.Pattern, dict[str, str]]:
    """Word-boundary regexes over every skills.yaml variant, built once. One- and two-letter
    variants and 'wind' match uppercase only, so 'R', 'ML' and 'WIND' hit but 'r' and 'wind' do not."""
    canon: dict[str, str] = {}
    loose: list[str] = []
    strict: list[str] = []
    for variant, info in taxonomy.skill_variants().items():
        if info["category"] == "cert":
            continue
        canon[variant] = info["canonical"]
        body = r"\s+".join(re.escape(w) for w in variant.split())
        (strict if len(variant) <= 2 or variant == "wind" else loose).append(body)
    wrap = r"(?<!\w)(?:{})(?!\w)"
    loose_rx = re.compile(wrap.format("|".join(sorted(loose, key=len, reverse=True))), re.IGNORECASE)
    upper = [s.upper() for s in strict]
    strict_rx = re.compile(wrap.format("|".join(sorted(upper, key=len, reverse=True))))
    return loose_rx, strict_rx, canon


def _skill_hits(s: str) -> dict[str, re.Match]:
    loose_rx, strict_rx, canon = _skill_matchers()
    hits: dict[str, re.Match] = {}
    for rx in (loose_rx, strict_rx):
        for m in rx.finditer(s):
            key = _WS.sub(" ", m.group().lower())
            if key in canon:
                hits.setdefault(canon[key], m)
    return hits


def _extract_skills(text: str, filename: str, anchor: str | None = None) -> Found:
    hits = _skill_hits(text)
    first = next(iter(hits.values()), None)
    return Found(sorted(hits), _snip(text, first.start(), first.end()) if first else "")


def _compare_skills(llm: dict, matched: list[str]) -> bool:
    _, _, canon = _skill_matchers()
    have: set[str] = set()
    for group in ("programming", "tools", "domain"):
        for item in llm.get(group, []):
            low = _WS.sub(" ", item.lower().strip())
            if low in canon:
                have.add(canon[low])
            have |= set(_skill_hits(item))
    return set(matched) <= have


# date grounding -----------------------------------------------------------------------------

_MONTH_WORDS: dict[int, tuple[str, ...]] = {
    1: ("jan", "january", "janvier", "janeiro"),
    2: ("feb", "february", "fev", "fevrier", "fevereiro"),
    3: ("mar", "march", "mars", "marco"),
    4: ("apr", "april", "avr", "avril", "abr", "abril"),
    5: ("may", "mai", "maio"),
    6: ("jun", "june", "juin", "junho"),
    7: ("jul", "july", "juil", "juillet", "julho"),
    8: ("aug", "august", "aout", "ago", "agosto"),
    9: ("sep", "sept", "september", "septembre", "set", "setembro"),
    10: ("oct", "october", "octobre", "out", "outubro"),
    11: ("nov", "november", "novembre", "novembro"),
    12: ("dec", "december", "decembre", "dez", "dezembro"),
}
_MONTH_NUM = {w: n for n, ws in _MONTH_WORDS.items() for w in ws}
_LETTERS = r"[A-Za-zÀ-ÿ]"
_MONTH_YEAR = re.compile(
    rf"(?<!{_LETTERS})(?P<mon>{_LETTERS}{{3,10}})\.?[\s,\-–/'’‘.]*(?P<yr>\d{{4}}|\d{{2}})(?!\d)"
)
_MONTH_DAY_YEAR = re.compile(
    rf"(?<!{_LETTERS})(?P<mon>{_LETTERS}{{3,10}})\.?\s*\d{{1,2}}(?:st|nd|rd|th)?,?\s*(?P<yr>\d{{4}})(?!\d)"
)
_NUM_MY = re.compile(r"(?<!\d)(?P<m>0?[1-9]|1[0-2])\s*[/.\-]\s*(?P<y>(?:19|20)\d{2})(?!\d)")
_NUM_YM = re.compile(r"(?<!\d)(?P<y>(?:19|20)\d{2})\s*[/.\-]\s*(?P<m>0?[1-9]|1[0-2])(?!\d)")
_YEAR = re.compile(r"(?<!\d)(?:19|20)\d{2}(?!\d)")


def _full_year(s: str) -> int:
    if len(s) == 4:
        return int(s)
    yy = int(s)
    return 2000 + yy if yy <= 69 else 1900 + yy


def grounded_dates(text: str) -> tuple[set[str], set[str]]:
    """(YYYY-MM tokens, YYYY tokens) the text can justify. Month names, 'Sept.2017',
    "May'22", "Oct-'20", 'Jun 2021', '03/2020' and '2020-03' ground a month; any year ground a year."""
    months: set[str] = set()
    for rx in (_MONTH_YEAR, _MONTH_DAY_YEAR):
        for m in rx.finditer(text):
            n = _MONTH_NUM.get(_plain(m["mon"]))
            if n:
                months.add(f"{_full_year(m['yr'])}-{n:02d}")
    for rx in (_NUM_MY, _NUM_YM):
        for m in rx.finditer(text):
            months.add(f"{m['y']}-{int(m['m']):02d}")
    years = {y.group() for y in _YEAR.finditer(text)} | {mo[:4] for mo in months}
    return months, years


def _extract_dates(text: str, filename: str, anchor: str | None = None) -> Found:
    months, years = grounded_dates(text)
    return Found({"months": sorted(months), "years": sorted(years)}, "")


def _compare_date(llm: str | None, grounded: dict) -> bool:
    if llm is None:
        return True
    if len(llm) == 7:
        return llm in grounded["months"]
    return llm in grounded["years"]


# duration -----------------------------------------------------------------------------------

_DURATION = re.compile(
    r"(\d+)\s*(years?|yrs?)(?:\s*(\d+)\s*months?)?|(\d+)\s*months?|(\d+)\s*weeks?", re.IGNORECASE
)
_ORG_SUFFIX = re.compile(
    r"\b(?:ltd|limited|inc|llc|llp|plc|pvt|private|corp|corporation|co|company|pte|gmbh)\b\.?",
    re.IGNORECASE,
)


def duration_months(s: str) -> int | None:
    """First duration phrase in `s` as whole months; weeks are floored to a quarter-month."""
    m = _DURATION.search(s)
    return _months_of(m) if m else None


def _months_of(m: re.Match) -> int:
    if m.group(1):
        return int(m.group(1)) * 12 + int(m.group(3) or 0)
    if m.group(4):
        return int(m.group(4))
    return int(m.group(5)) // 4


def _employer_forms(employer: str) -> list[str]:
    forms = [_WS.sub(" ", employer.lower()).strip()]
    short = _WS.sub(" ", _ORG_SUFFIX.sub("", employer.lower())).strip(" .,")
    if short and short not in forms:
        forms.append(short)
    return forms


def _extract_duration(text: str, filename: str, anchor: str | None = None) -> Found:
    if not anchor:
        return Found(None)
    for form in _employer_forms(anchor):
        for line in text.splitlines():
            flat = _WS.sub(" ", line.lower())
            pos = flat.find(form)
            if pos < 0:
                continue
            ms = list(_DURATION.finditer(line))
            if ms:
                nearest = min(ms, key=lambda m: abs(m.start() - pos))
                return Found(_months_of(nearest), _WS.sub(" ", line).strip()[:160])
    return Found(None)


def _compare_duration(llm: int | None, text_months: int | None) -> bool:
    if text_months is None:
        return True
    return llm is not None and abs(llm - text_months) <= 1


# role coverage on duration-only resumes -----------------------------------------------------

_HEADER_DURATION = re.compile(
    r"(?:\b(?:Duration|Others?)\s*[|:]\s*|\(\s*)(?:" + _DURATION.pattern + ")", re.IGNORECASE
)


def _extract_header_durations(text: str, filename: str, anchor: str | None = None) -> Found:
    """Durations written as a role header ('Duration | 7 months', 'Others: 8 months', 'Project
    (8 weeks):'). Free
    prose such as 'within 3 months of joining' is not a header, so it never counts as a role."""
    ms = list(_HEADER_DURATION.finditer(text))
    return Found([_months_of(m) for m in ms], _snip(text, ms[0].start(), ms[0].end()) if ms else "")


def _compare_role_coverage(roles: list[dict], header_months: list[int]) -> bool:
    """On a duration-only resume every header duration must belong to some LLM role. This is the
    experience-count row: employer lines have no reliable pattern, duration headers do."""
    if not header_months or all(r["start"] for r in roles):
        return True
    pool = [r["duration_months"] for r in roles if r["duration_months"] is not None]
    for m in header_months:
        hit = next((p for p in pool if abs(p - m) <= 1), None)
        if hit is None:
            return False
        pool.remove(hit)
    return True


# degree level -------------------------------------------------------------------------------

_A = r"(?<![\w.])"  # acronyms must not sit inside a longer dotted form such as 'B.M.S'
_NOT_STATE = r"(?<!,\s)"  # 'Cambridge, MA' is a state, not a Master of Arts
_DEGREE_RULES: tuple[tuple[re.Pattern, str], ...] = (
    (re.compile(rf"{_A}M\.?B\.?B\.?S\b"), "Other"),
    (re.compile(rf"{_A}MBA\b|Master of Business Administration"), "MBA"),
    (
        re.compile(
            rf"{_A}PGDM\b|{_A}PGP\b|Post[- ]?Graduate Diploma in Management", re.IGNORECASE
        ),
        "Masters",
    ),
    (re.compile(rf"{_A}Ph\.?\s?D\b|\bDPhil\b|Doctor(?:ate)? of Philosophy"), "PhD"),
    (
        re.compile(
            rf"{_A}M\.?Sc\b|{_NOT_STATE}{_A}M\.?S\b|{_NOT_STATE}{_A}M\.?A\b|\bMEng\b"
            rf"|{_A}M\.?Tech\b|\bMFE\b|\bMFin\b|\bMPhil\b|\bMSF\b|\bLLM\b|\bMasters?\b"
            r"|\bMaster['’]s\b|Dipl[oô]me d['’]Ing[eé]nieur"
        ),
        "Masters",
    ),
    (
        re.compile(
            rf"{_A}B\.?Sc\b|{_A}B\.?S\b|{_A}B\.?A\b|\bBBA\b|\bBEng\b|{_A}B\.E\b"
            rf"|{_A}B\.?Tech\b|{_A}B\.?Com\b|{_A}B\.?M\.?S\b|\bBEcon\b|\bBFin\b"
            r"|\bBachelors?\b|\bBachelor['’]s\b"
        ),
        "Bachelors",
    ),
)


def _level_key(level: str | None) -> str | None:
    if level is None:
        return None
    return level.lower().replace("'", "").replace("’", "").rstrip("s")


def _extract_degree(text: str, filename: str, anchor: str | None = None) -> Found:
    if not anchor:
        return Found(None)
    needle = _WS.sub(" ", anchor.lower()).strip()
    for line in text.splitlines():
        if needle not in _WS.sub(" ", line.lower()):
            continue
        levels = {lvl for rx, lvl in _DEGREE_RULES if rx.search(line)}
        if len(levels) == 1:
            return Found(levels.pop(), _WS.sub(" ", line).strip()[:160])
    return Found(None)


def _compare_degree(llm: str | None, text_level: str | None) -> bool:
    if text_level is None:
        return True
    return _level_key(llm) == _level_key(text_level)


# country from phone prefix ------------------------------------------------------------------


def _prefix_countries(number: str) -> list[str]:
    n = number.strip()
    if n.startswith("00"):
        n = "+" + n[2:]
    if not n.startswith("+"):
        return []
    digits = "+" + _digits(n)
    for prefix in sorted(taxonomy.phone_prefixes(), key=len, reverse=True):
        if digits.startswith(prefix):
            return taxonomy.phone_prefixes()[prefix]
    return []


def _extract_phone_country(text: str, filename: str, anchor: str | None = None) -> Found:
    for m in _phone_candidates(text):
        iso = _prefix_countries(m.group())
        if iso:
            return Found(iso, _snip(text, m.start(), m.end()))
    return Found([])


def _compare_country(llm: str | None, iso: list[str]) -> bool:
    return not iso or llm is None or llm in iso


# registry -----------------------------------------------------------------------------------

CHECKS: tuple[Check, ...] = (
    Check("name", _extract_name, _compare_name, "name"),
    Check("contact.email", _extract_email, _compare_email, "contact"),
    Check("contact.phone", _extract_phone, _compare_phone, "contact"),
    Check("certifications", _extract_certs, _compare_certs, "certifications"),
    Check("skills", _extract_skills, _compare_skills, "skills"),
    Check("experience[i].start", _extract_dates, _compare_date, "experience_item"),
    Check("experience[i].end", _extract_dates, _compare_date, "experience_item"),
    Check(
        "experience[i].duration_months", _extract_duration, _compare_duration,
        "experience_item", anchor_key="employer",
        # With start dates the derive layer ignores duration, and a duration near the employer
        # name may be a career total ("covering healthcare for 9 years").
        applies=lambda e: e["start"] is None or e["duration_months"] is not None,
    ),
    Check("experience", _extract_header_durations, _compare_role_coverage, "experience"),
    Check(
        "education[i].degree_level", _extract_degree, _compare_degree,
        "education_item", anchor_key="institution",
    ),
    Check(
        "derived.current_country", _extract_phone_country, _compare_country, "experience_item"
    ),
)


def current_role_index(experiences: list[dict]) -> int | None:
    """The is_current role with the latest start, else the role with the latest end or start."""
    if not experiences:
        return None
    current = [i for i, e in enumerate(experiences) if e["is_current"]]
    if current:
        return max(current, key=lambda i: experiences[i]["start"] or "")
    return max(range(len(experiences)), key=lambda i: (
        experiences[i]["end"] or experiences[i]["start"] or ""))


def _walk(doc: Any, path: str) -> Any:
    for part in path.replace("]", "").replace("[", ".").split("."):
        doc = doc[int(part)] if part.isdigit() else doc[part]
    return doc


def build_doc(profile: Profile, experiences: ExperienceList | list) -> dict:
    exps = experiences.experience if isinstance(experiences, ExperienceList) else experiences
    doc = profile.model_dump(mode="json")
    doc["experience"] = [e.model_dump(mode="json") for e in exps]
    idx = current_role_index(doc["experience"])
    doc["derived"] = {"current_country": doc["experience"][idx]["country"] if idx is not None else None}
    return doc


def verify(
    profile: Profile,
    experiences: ExperienceList | list,
    text: str,
    filename: str,
    only: frozenset[str] | None = None,
) -> VerificationReport:
    """Run every registry row. `only` restricts to concrete field paths such as
    'experience[2].start', for re-checking one regenerated group."""
    doc = build_doc(profile, experiences)
    cur = current_role_index(doc["experience"])
    run = 0
    found: list[Disagreement] = []
    for check in CHECKS:
        if "[i]" in check.field:
            head = check.field.split("[i]")[0]
            targets: list[tuple[str, int | None, str | None]] = []
            for i, item in enumerate(_walk(doc, head)):
                if check.applies and not check.applies(item):
                    continue
                anchor = item[check.anchor_key] if check.anchor_key else None
                targets.append((check.field.replace("[i]", f"[{i}]"), i, anchor))
        else:
            index = cur if check.regen_target == "experience_item" else None
            targets = [(check.field, index, None)]
        for path, index, anchor in targets:
            if only is not None and path not in only:
                continue
            run += 1
            llm = _walk(doc, path)
            got = check.extract(text, filename, anchor)
            if not check.compare(llm, got.value):
                shown = [(r["employer"], r["duration_months"]) for r in llm] if path == "experience" else llm
                found.append(
                    Disagreement(path, shown, got.value, got.snippet, check.regen_target, index)
                )
    return VerificationReport(run, found)

Overwriting src/resume_search/parse/verify.py


#### `src/resume_search/search/__init__.py`

Public surface of the search package.

In [35]:
%%writefile src/resume_search/search/__init__.py
from resume_search.search.keyword import keyword_matches, keyword_query, keyword_scores
from resume_search.search.rank import ScoreBreakdown, SearchResult, rerank_llm, search
from resume_search.search.requisition import (
    Criterion,
    Requisition,
    parse_requisition,
    resolve_location,
)
from resume_search.search.rules import Chip, ChipStatus, prune_sql, rule_score
from resume_search.search.weights import Weights

__all__ = [
    "Chip", "ChipStatus", "Criterion", "Requisition", "ScoreBreakdown", "SearchResult",
    "Weights", "keyword_matches", "keyword_query", "keyword_scores", "parse_requisition",
    "prune_sql", "rerank_llm", "resolve_location", "rule_score", "search",
]

Overwriting src/resume_search/search/__init__.py


#### `src/resume_search/search/keyword.py`

BM25 scoring over the FTS5 index and the list of query terms each candidate matched.

In [36]:
%%writefile src/resume_search/search/keyword.py
from __future__ import annotations

import sqlite3

from resume_search import taxonomy as tx
from resume_search.search.requisition import Requisition

STOPWORDS = frozenset(
    ["a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "have", "in", "is", "it", "its", "of", "on", "or", "our", "that", "the", "their", "this", "to", "we", "will", "with", "you", "your", "who", "they", "than", "then", "there", "these", "those", "been", "being", "into", "over", "per", "such", "can", "may", "must", "not", "also", "other", "about", "across", "within"]
)
MAX_TOKENS = 60


def keyword_query(req: Requisition) -> str:
    """Empty when there is no free text: keyword scoring is dropped, not run on skills alone."""
    text = (req.free_text or "").strip()
    return " ".join([text, *req.skills_required, *req.skills_preferred]) if text else ""


def query_tokens(text: str) -> list[str]:
    seen: dict[str, None] = {}
    for tok in tx.tokenize(text):
        low = tok.lower()
        if len(low) > 1 and low not in STOPWORDS:
            seen.setdefault(low)
    return list(seen)[:MAX_TOKENS]


def _quote(token: str) -> str:
    return '"' + token.replace('"', '""') + '"'


def keyword_scores(
    conn: sqlite3.Connection, candidate_ids: list[str], query_text: str
) -> dict[str, float]:
    """BM25 relevance (higher is better) for the candidates that match at least one token.
    FTS5's bm25() is negative with lower meaning better, so it is negated here."""
    tokens = query_tokens(query_text)
    if not tokens or not candidate_ids:
        return {}
    ph = ",".join("?" * len(candidate_ids))
    rows = conn.execute(
        f"SELECT candidate_id, -bm25(candidate_fts) AS score FROM candidate_fts "
        f"WHERE candidate_fts MATCH ? AND candidate_id IN ({ph})",
        [" OR ".join(map(_quote, tokens)), *candidate_ids],
    ).fetchall()
    return {r["candidate_id"]: r["score"] for r in rows}


def keyword_matches(
    conn: sqlite3.Connection, candidate_ids: list[str], query_text: str
) -> dict[str, list[str]]:
    """Which query tokens occur in each candidate's text (stemmed like the index)."""
    out: dict[str, list[str]] = {cid: [] for cid in candidate_ids}
    if not candidate_ids:
        return out
    ph = ",".join("?" * len(candidate_ids))
    for tok in query_tokens(query_text):
        rows = conn.execute(
            f"SELECT candidate_id FROM candidate_fts WHERE candidate_fts MATCH ? "
            f"AND candidate_id IN ({ph})",
            [_quote(tok), *candidate_ids],
        )
        for r in rows:
            out[r["candidate_id"]].append(tok)
    return out

Overwriting src/resume_search/search/keyword.py


#### `src/resume_search/search/llm.py`

A shared structured-output call with one retry, used by the requisition parser and the reranker.

In [37]:
%%writefile src/resume_search/search/llm.py
from __future__ import annotations

import time
from collections.abc import Callable

import pydantic

from resume_search.parse import prompt
from resume_search.parse.client import SONNET, Attempt, _validation_lines, default_client

MAX_TOKENS = 4000


def structured_call[T: pydantic.BaseModel](
    client,
    system: str,
    user: str,
    output_format: type[T],
    *,
    validate: Callable[[T], list[str]] = lambda _: [],
    attempts: list[Attempt] | None = None,
    call: str = "",
    model: str = SONNET,
    max_tokens: int = MAX_TOKENS,
) -> T:
    """One structured-output call (Sonnet unless `model` says otherwise), retried once with the
    validation errors appended."""
    client = client or default_client()
    suffix = ""
    errors: list[str] = []
    for _ in range(2):
        started = time.monotonic()
        usage = None
        out: T | None = None
        try:
            response = client.messages.parse(
                model=model,
                max_tokens=max_tokens,
                system=system,
                messages=[{"role": "user", "content": user + suffix}],
                output_format=output_format,
            )
            usage = response.usage
            out = response.parsed_output
            errors = (
                validate(out) if out is not None
                else [f"no parsed output (stop_reason={response.stop_reason})"]
            )
        except pydantic.ValidationError as exc:
            errors = _validation_lines(exc)
        if attempts is not None:
            attempts.append(
                Attempt(
                    model=model,
                    latency_ms=int((time.monotonic() - started) * 1000),
                    input_tokens=usage.input_tokens if usage else 0,
                    output_tokens=usage.output_tokens if usage else 0,
                    cache_read_tokens=getattr(usage, "cache_read_input_tokens", 0) or 0,
                    cache_creation_tokens=getattr(usage, "cache_creation_input_tokens", 0) or 0,
                    error="; ".join(errors) or None,
                    call=call,
                )
            )
        if not errors and out is not None:
            return out
        suffix = prompt.retry_suffix(errors)
    raise ValueError(f"{call} failed after retry: {'; '.join(errors)}")

Overwriting src/resume_search/search/llm.py


#### `src/resume_search/search/rank.py`

`search()` combines the rule, BM25, and embedding scores into the final order, and `rerank_llm` optionally reorders the top 10.

In [38]:
%%writefile src/resume_search/search/rank.py
from __future__ import annotations

import sqlite3
from dataclasses import dataclass

from pydantic import BaseModel

from resume_search.search.keyword import keyword_matches, keyword_query, keyword_scores
from resume_search.search.llm import structured_call
from resume_search.search.requisition import Requisition
from resume_search.search.rules import Chip, ChipStatus, load_candidates, prune_sql, rule_score
from resume_search.search.vectors import VectorIndex, default_embedder
from resume_search.search.weights import Weights


@dataclass(frozen=True)
class ScoreBreakdown:
    rule: float
    keyword: float
    embedding: float
    final: float
    rule_terms: list[tuple[str, float, float]]


@dataclass(frozen=True)
class SearchResult:
    candidate_id: str
    name: str
    search_summary: str
    seniority_band: str | None
    current_country: str | None
    current_region: str | None
    breakdown: ScoreBreakdown
    chips: list[Chip]
    keywords: list[str]


def _minmax(raw: dict[str, float], ids: list[str], missing: float) -> dict[str, float]:
    """Scale to 0..1 over the pruned set. When nothing separates the candidates, everyone with
    a signal gets 1.0 and everyone without gets 0.0."""
    vals = {i: raw.get(i, missing) for i in ids}
    lo, hi = min(vals.values(), default=0.0), max(vals.values(), default=0.0)
    if hi == lo:
        return {i: 1.0 if hi > 0 and i in raw else 0.0 for i in ids}
    return {i: (v - lo) / (hi - lo) for i, v in vals.items()}


def search(
    conn: sqlite3.Connection,
    req: Requisition,
    weights: Weights,
    embedder=None,
    index: VectorIndex | None = None,
    top_k: int = 20,
) -> list[SearchResult]:
    ids = [r[0] for r in conn.execute(*prune_sql(req))]
    cands = load_candidates(conn, ids)
    ids = [c.candidate_id for c in cands]
    rules = {c.candidate_id: rule_score(req, c, weights) for c in cands}
    has_rule = any(terms for _, terms, _ in rules.values())
    query = keyword_query(req)
    has_text = bool((req.free_text or "").strip())

    kw = emb = {i: 0.0 for i in ids}
    if has_text:
        kw = _minmax(keyword_scores(conn, ids, query), ids, 0.0)
        qvec = (embedder or default_embedder()).embed([req.free_text.strip()])[0]
        raw = (index or VectorIndex.load(conn)).scores(qvec)
        raw = {i: raw[i] for i in ids if i in raw}
        emb = _minmax(raw, ids, min(raw.values(), default=0.0))

    parts = []
    if has_rule:
        parts.append((weights.blend_rule, "rule"))
    if has_text:
        parts += [(weights.blend_keyword, "kw"), (weights.blend_embedding, "emb")]

    def final(cid: str) -> float:
        if not parts:
            return 1.0
        vals = {"rule": rules[cid][0], "kw": kw[cid], "emb": emb[cid]}
        return sum(w * vals[k] for w, k in parts) / sum(w for w, _ in parts)

    scored = sorted(((final(c.candidate_id), c) for c in cands), key=lambda t: (-t[0], t[1].name))
    top = scored[:top_k]
    hits = keyword_matches(conn, [c.candidate_id for _, c in top], query) if has_text else {}
    return [
        SearchResult(
            candidate_id=c.candidate_id, name=c.name, search_summary=c.search_summary,
            seniority_band=c.seniority_band, current_country=c.current_country,
            current_region=c.current_region,
            breakdown=ScoreBreakdown(
                rule=rules[c.candidate_id][0], keyword=kw[c.candidate_id],
                embedding=emb[c.candidate_id], final=f, rule_terms=rules[c.candidate_id][1],
            ),
            chips=rules[c.candidate_id][2],
            keywords=hits.get(c.candidate_id, []),
        )
        for f, c in top
    ]


class _Ranked(BaseModel):
    candidate_id: str
    reason: str


class _Reranking(BaseModel):
    order: list[_Ranked]


_RERANK_SYSTEM = """\
You re-rank shortlisted hedge fund candidates against a search requisition. The reply is
constrained to the output schema.
- Return every candidate exactly once, best fit first. Use only the candidate_id values given.
- reason is one sentence naming the facts from the summary or criteria that decided the
  position. Do not use facts that are not in the input.
- Must criteria are hard requirements. Preferred criteria and the free text are soft.
"""


def _chip_line(c: Chip) -> str:
    mark = "met" if c.status is ChipStatus.matched else "not met"
    return f"{c.label} ({mark}{': ' + c.detail if c.detail else ''})"


def rerank_llm(
    req: Requisition, results: list[SearchResult], client=None, attempts: list | None = None
) -> list[tuple[str, str]]:
    blocks = [
        f'<candidate id="{r.candidate_id}" band="{r.seniority_band}" '
        f'country="{r.current_country}">\n{r.search_summary}\n'
        f"Criteria: {'; '.join(_chip_line(c) for c in r.chips) or 'none'}\n</candidate>"
        for r in results
    ]
    user = (
        f"<requisition>\n{req.model_dump_json(exclude_none=True)}\n</requisition>\n"
        + "\n".join(blocks)
    )
    want = {r.candidate_id for r in results}

    def validate(out: _Reranking) -> list[str]:
        got = [x.candidate_id for x in out.order]
        errors = [f"unknown candidate_id {i}" for i in got if i not in want]
        errors += [f"missing candidate_id {i}" for i in want - set(got)]
        errors += [f"duplicate candidate_id {i}" for i in {i for i in got if got.count(i) > 1}]
        return errors

    out = structured_call(
        client, _RERANK_SYSTEM, user, _Reranking, validate=validate, attempts=attempts,
        call="rerank",
    )
    return [(x.candidate_id, x.reason) for x in out.order]

Overwriting src/resume_search/search/rank.py


#### `src/resume_search/search/requisition.py`

The `Requisition` object with must and preferred criteria, and the LLM step that turns a pasted JD into one.

In [39]:
%%writefile src/resume_search/search/requisition.py
from __future__ import annotations

from dataclasses import dataclass
from functools import cache
from typing import Literal

from pydantic import BaseModel, ValidationError, field_validator

from resume_search import taxonomy as tx
from resume_search.parse.client import _validation_lines
from resume_search.parse.models import Approach
from resume_search.search.llm import structured_call

Mode = Literal["must", "preferred"]
BANDS = ("Intern/Student", "Junior", "Mid", "Senior", "Principal")
DEGREES = ("Bachelors", "Masters", "MBA", "PhD")
APPROACHES = tuple(a.value for a in Approach if a is not Approach.unclear)


class Criterion[T](BaseModel):
    value: T
    mode: Mode = "preferred"


@dataclass(frozen=True)
class Place:
    kind: Literal["country", "region"]
    code: str


def resolve_location(value: str) -> Place:
    """ISO-2 code first (so "US" is the country), then a region name, then a country name or
    alias such as "United Kingdom" or "London"."""
    text = value.strip()
    if text.upper() in tx.iso2_codes():
        return Place("country", text.upper())
    for region in tx._load("countries")["regions"]:
        if region.casefold() == text.casefold():
            return Place("region", region)
    for iso, c in tx.countries().items():
        if text.casefold() in (n.casefold() for n in (c["name"], *c.get("aliases", []))):
            return Place("country", iso)
    raise ValueError(f"unknown location {value!r}")


def _pick(value: str, allowed: tuple[str, ...] | list[str], what: str) -> str:
    for a in allowed:
        if a.casefold() == value.strip().casefold():
            return a
    raise ValueError(f"unknown {what} {value!r}; allowed: {', '.join(allowed)}")


def _values(c: Criterion | None) -> list[str]:
    if c is None:
        return []
    return [c.value] if isinstance(c.value, str) else list(c.value)


class Requisition(BaseModel):
    free_text: str | None = None
    location: Criterion[str] | None = None
    location_scope: Literal["current", "worked", "either"] = "either"
    approach: Criterion[str] | None = None
    sectors: Criterion[list[str]] | None = None
    markets_covered: Criterion[list[str]] | None = None
    years_min: float | None = None
    years_max: float | None = None
    years_mode: Mode = "must"
    seniority_bands: Criterion[list[str]] | None = None
    skills_required: list[str] = []
    skills_preferred: list[str] = []
    degree_min: Criterion[str] | None = None
    certifications: Criterion[list[str]] | None = None
    languages: Criterion[list[str]] | None = None
    asset_classes: Criterion[list[str]] | None = None

    @field_validator("location")
    @classmethod
    def _location(cls, c):
        if c is not None:
            resolve_location(c.value)
        return c

    @field_validator("approach")
    @classmethod
    def _approach(cls, c):
        return c and Criterion[str](value=_pick(c.value, APPROACHES, "approach"), mode=c.mode)

    @field_validator("degree_min")
    @classmethod
    def _degree(cls, c):
        return c and Criterion[str](value=_pick(c.value, DEGREES, "degree"), mode=c.mode)

    @field_validator("sectors", "seniority_bands", "asset_classes")
    @classmethod
    def _enum_list(cls, c, info):
        if c is None:
            return c
        allowed = {
            "sectors": tx.sectors(), "seniority_bands": list(BANDS), "asset_classes": tx.asset_classes()
        }[info.field_name]
        return Criterion[list[str]](
            value=[_pick(v, allowed, info.field_name) for v in c.value], mode=c.mode
        )

    def is_empty(self) -> bool:
        return not (
            (self.free_text or "").strip()
            or self.years_min is not None
            or self.years_max is not None
            or self.skills_required
            or self.skills_preferred
            or any(
                _values(c)
                for c in (
                    self.location, self.approach, self.sectors, self.markets_covered,
                    self.seniority_bands, self.degree_min, self.certifications,
                    self.languages, self.asset_classes,
                )
            )
        )


Field = Literal[
    "location", "approach", "sectors", "markets_covered", "seniority_bands", "degree_min",
    "certifications", "languages", "asset_classes",
]
_SINGLE = ("location", "approach", "degree_min")


class DraftCriterion(BaseModel):
    field: Field
    values: list[str]
    mode: Mode


class RequisitionDraft(BaseModel):
    """What the model fills. Requisition's nested optional generics exceed the structured-output
    schema limit, so the model gets this flat list and `to_requisition` builds the real thing."""

    criteria: list[DraftCriterion] = []
    location_scope: Literal["current", "worked", "either"] = "either"
    years_min: float | None = None
    years_max: float | None = None
    years_mode: Mode = "must"
    skills_required: list[str] = []
    skills_preferred: list[str] = []

    def to_requisition(self, free_text: str) -> Requisition:
        kw: dict = {}
        for c in self.criteria:
            if not c.values:
                continue
            kw[c.field] = (
                Criterion[str](value=c.values[0], mode=c.mode)
                if c.field in _SINGLE
                else Criterion[list[str]](value=c.values, mode=c.mode)
            )
        return Requisition(
            free_text=free_text, location_scope=self.location_scope, years_min=self.years_min,
            years_max=self.years_max, years_mode=self.years_mode,
            skills_required=self.skills_required, skills_preferred=self.skills_preferred, **kw,
        )


_SYSTEM = """\
You turn a pasted job description or keyword string from a hedge fund recruiter into a
search requisition. The reply is constrained to the output schema.

Rules
- Fill only what the text states. Leave everything else null or empty. Never infer a
  criterion from the firm, the title alone, or common practice.
- Every filled criterion has mode "preferred", except skills explicitly called required or
  must-have (put those in skills_required) and an explicit years-of-experience range or
  minimum, which gets years_mode "must". A years figure hedged as "ideally" or "about" is
  "preferred". Skills described as a plus, nice to have, or preferred go in skills_preferred.
- criteria holds one entry per filled criterion: field, values, mode. location, approach and
  degree_min take exactly one value; the other fields take a list. Omit a field the text does
  not mention.
- location: an ISO-2 code or a region name. Codes: {iso2}. Regions: {regions}.
  location_scope is "current" when the text says based in, located in, or must be in a
  place, otherwise "either". "worked" only when it asks for work history in a place.
- approach: one of {approaches}.
- sectors: from {sectors}.
- asset_classes: from {asset_classes}.
- seniority_bands: from {bands}, only when the text names a level such as junior or senior.
- degree_min: one of {degrees}, the lowest acceptable.
- certifications: names as written, for example "CFA".
- markets_covered: ISO-2 codes or market labels ({market_labels}) for geographies the
  candidate must have covered.
- Certifications such as the CFA go in certifications only, never in the skill lists.
- skills_required and skills_preferred: use the canonical skill names where one fits:
  {skills}. Otherwise use the short lowercase name from the text.
- years_min and years_max: numbers of years. "2-5 years" is 2 and 5. "5+ years" is
  years_min 5 and years_max null.
"""


@cache
def system_prompt() -> str:
    skills = sorted({v["canonical"] for v in tx.skill_variants().values()})
    return _SYSTEM.format(
        iso2=", ".join(sorted(tx.iso2_codes())),
        regions=", ".join(tx._load("countries")["regions"]),
        approaches=", ".join(APPROACHES),
        sectors=", ".join(tx.sectors()),
        asset_classes=", ".join(tx.asset_classes()),
        bands=", ".join(BANDS),
        degrees=", ".join(DEGREES),
        market_labels=", ".join(sorted(tx.market_labels())),
        skills=", ".join(skills),
    )


def _convert_errors(draft: RequisitionDraft, text: str) -> list[str]:
    try:
        draft.to_requisition(text)
    except ValidationError as exc:
        return _validation_lines(exc)
    return []


def parse_requisition(text: str, client=None, attempts: list | None = None) -> Requisition:
    draft = structured_call(
        client, system_prompt(), f"<job_description>\n{text}\n</job_description>",
        RequisitionDraft, validate=lambda d: _convert_errors(d, text), attempts=attempts,
        call="requisition",
    )
    return draft.to_requisition(text)

Overwriting src/resume_search/search/requisition.py


#### `src/resume_search/search/rules.py`

Prunes candidates by must criteria in SQL and scores preferred criteria, producing the explanation chips.

In [40]:
%%writefile src/resume_search/search/rules.py
from __future__ import annotations

import sqlite3
from collections import defaultdict
from collections.abc import Iterable
from dataclasses import dataclass
from enum import StrEnum

from rapidfuzz import fuzz

from resume_search import taxonomy as tx
from resume_search.search.requisition import (
    DEGREES,
    Criterion,
    Place,
    Requisition,
    _values,
    resolve_location,
)
from resume_search.search.weights import Weights


class ChipStatus(StrEnum):
    matched = "matched"
    unmatched = "unmatched"


@dataclass(frozen=True)
class Chip:
    criterion: str
    label: str
    status: ChipStatus
    detail: str | None = None


@dataclass(frozen=True)
class Candidate:
    candidate_id: str
    name: str
    search_summary: str
    seniority_band: str | None
    approach: str | None
    months_relevant: int
    highest_degree: str | None
    has_cfa: bool
    current_country: str | None
    current_region: str | None
    sectors: dict[str, str]
    assets: frozenset[str]
    skills: frozenset[str]
    worked: dict[str, int]
    covered: frozenset[str]
    certs: tuple[str, ...]
    languages: frozenset[str]


# Prune ------------------------------------------------------------------------------------------


def canonical_skill(name: str) -> str:
    key = name.strip().lower()
    hit = tx.skill_variants().get(key)
    return hit["canonical"] if hit else key


def _in(values: list) -> str:
    return "(" + ",".join("?" * len(values)) + ")"


def _exists(table: str, cond: str) -> str:
    return f"EXISTS (SELECT 1 FROM {table} t WHERE t.candidate_id = c.candidate_id AND {cond})"


def _location_clause(place: Place, scope: str) -> tuple[str, list]:
    if place.kind == "country":
        cur, cur_p = "c.current_country = ?", [place.code]
        worked = _exists("candidate_countries", "t.kind = 'worked' AND t.country = ?")
    else:
        cur, cur_p = "c.current_region = ?", [place.code]
        worked = _exists(
            "candidate_countries",
            "t.kind = 'worked' AND t.country IN (SELECT iso2 FROM taxonomy_country WHERE region = ?)",
        )
    if scope == "current":
        return cur, cur_p
    if scope == "worked":
        return worked, [place.code]
    return f"({cur} OR {worked})", cur_p + [place.code]


def _lower_in(values: list[str]) -> tuple[str, list[str]]:
    return _in(values), [v.casefold() for v in values]


def prune_sql(req: Requisition) -> tuple[str, list]:
    """Candidate ids satisfying every must criterion. Skills match on the canonical token only;
    fuzzy matching is a scoring concern."""
    where: list[str] = []
    params: list = []

    def add(clause: str, p: list) -> None:
        where.append(clause)
        params.extend(p)

    if req.location and req.location.mode == "must":
        add(*_location_clause(resolve_location(req.location.value), req.location_scope))
    if req.approach and req.approach.mode == "must":
        add("(c.investment_approach = ? OR c.investment_approach = 'Both')", [req.approach.value])
    if req.sectors and req.sectors.mode == "must":
        ph, p = _in(req.sectors.value), list(req.sectors.value)
        add(_exists("candidate_sectors", f"t.sector IN {ph}"), p)
    if req.markets_covered and req.markets_covered.mode == "must":
        ph, p = _lower_in(req.markets_covered.value)
        add(_exists("candidate_countries", f"t.kind = 'covered' AND lower(t.country) IN {ph}"), p)
    if req.asset_classes and req.asset_classes.mode == "must":
        ph, p = _in(req.asset_classes.value), list(req.asset_classes.value)
        add(_exists("candidate_assets", f"t.asset_class IN {ph}"), p)
    if req.seniority_bands and req.seniority_bands.mode == "must":
        add(f"c.seniority_band IN {_in(req.seniority_bands.value)}", list(req.seniority_bands.value))
    if req.years_mode == "must":
        if req.years_min is not None:
            add("c.months_full_time_relevant >= ?", [req.years_min * 12])
        if req.years_max is not None:
            add("c.months_full_time_relevant <= ?", [req.years_max * 12])
    for skill in req.skills_required:
        add(_exists("candidate_skills", "t.skill = ?"), [canonical_skill(skill)])
    if req.degree_min and req.degree_min.mode == "must":
        ok = list(DEGREES[DEGREES.index(req.degree_min.value):])
        add(f"c.highest_degree IN {_in(ok)}", ok)
    if req.certifications and req.certifications.mode == "must":
        for cert in req.certifications.value:
            like = _exists("candidate_certs", "lower(t.cert) LIKE ?")
            if cert.casefold() == "cfa":
                like = f"(c.has_cfa = 1 OR {like})"
            add(like, [f"%{cert.casefold()}%"])
    if req.languages and req.languages.mode == "must":
        for lang in req.languages.value:
            add(_exists("candidate_languages", "lower(t.language) = ?"), [lang.casefold()])

    sql = "SELECT c.candidate_id FROM candidates c"
    if where:
        sql += " WHERE " + " AND ".join(where)
    return sql, params


# Loading ----------------------------------------------------------------------------------------


def _chunks(ids: list[str], n: int = 500):
    for i in range(0, len(ids), n):
        yield ids[i : i + n]


def load_candidates(conn: sqlite3.Connection, ids: Iterable[str]) -> list[Candidate]:
    """Candidate rows plus every facet in eight grouped queries, not one per candidate."""
    ids = list(ids)
    facets: dict[str, dict[str, list]] = {
        k: defaultdict(list)
        for k in ("sectors", "assets", "skills", "worked", "covered", "certs", "languages")
    }
    rows = []
    for chunk in _chunks(ids):
        ph = _in(chunk)
        rows += conn.execute(f"SELECT * FROM candidates WHERE candidate_id IN {ph}", chunk).fetchall()
        for table, cols, key, fn in (
            ("candidate_sectors", "sector, evidence", "sectors", lambda r: (r[0], r[1])),
            ("candidate_assets", "asset_class", "assets", lambda r: r[0]),
            ("candidate_skills", "skill", "skills", lambda r: r[0]),
            ("candidate_certs", "cert", "certs", lambda r: r[0]),
            ("candidate_languages", "language", "languages", lambda r: r[0]),
        ):
            for r in conn.execute(
                f"SELECT candidate_id, {cols} FROM {table} WHERE candidate_id IN {ph}", chunk
            ):
                facets[key][r[0]].append(fn(r[1:]))
        for r in conn.execute(
            f"SELECT candidate_id, country, months, kind FROM candidate_countries "
            f"WHERE candidate_id IN {ph}", chunk,
        ):
            if r["kind"] == "worked":
                facets["worked"][r[0]].append((r["country"], r["months"] or 0))
            else:
                facets["covered"][r[0]].append(r["country"])
    return [
        Candidate(
            candidate_id=r["candidate_id"], name=r["name"], search_summary=r["search_summary"],
            seniority_band=r["seniority_band"], approach=r["investment_approach"],
            months_relevant=r["months_full_time_relevant"] or 0,
            highest_degree=r["highest_degree"], has_cfa=bool(r["has_cfa"]),
            current_country=r["current_country"], current_region=r["current_region"],
            sectors=dict(facets["sectors"][r["candidate_id"]]),
            assets=frozenset(facets["assets"][r["candidate_id"]]),
            skills=frozenset(facets["skills"][r["candidate_id"]]),
            worked=dict(facets["worked"][r["candidate_id"]]),
            covered=frozenset(facets["covered"][r["candidate_id"]]),
            certs=tuple(facets["certs"][r["candidate_id"]]),
            languages=frozenset(facets["languages"][r["candidate_id"]]),
        )
        for r in rows
    ]


# Scoring ----------------------------------------------------------------------------------------


@dataclass(frozen=True)
class _Term:
    criterion: str
    mode: str
    weight: float
    fraction: float
    chips: tuple[Chip, ...]


def _term(
    criterion: str, mode: str, weight: float, fraction: float, label: str,
    detail: str | None, matched: bool | None = None,
) -> _Term:
    ok = fraction > 0 if matched is None else matched
    chip = Chip(criterion, label, ChipStatus.matched if ok else ChipStatus.unmatched, detail)
    return _Term(criterion, mode, weight, fraction, (chip,))


def _location(
    cand: Candidate, place: Place, scope: str, w: Weights
) -> tuple[float, str | None]:
    regions = {iso: c["region"] for iso, c in tx.countries().items()}
    target_region = regions[place.code] if place.kind == "country" else place.code

    def current() -> tuple[float, str | None]:
        if place.kind == "country" and cand.current_country == place.code:
            return w.location_current, f"Currently in {place.code}"
        if cand.current_region == target_region and cand.current_country:
            factor = w.location_region_factor if place.kind == "country" else 1.0
            return w.location_current * factor, f"Currently in {cand.current_country} ({target_region})"
        return 0.0, None

    def worked() -> tuple[float, str | None]:
        # An explicit "worked" scope makes work history the primary signal, so it scores in full.
        base = 1.0 if scope == "worked" else w.location_worked
        if place.kind == "country" and place.code in cand.worked:
            return base, f"Worked in {place.code} {cand.worked[place.code] / 12:.1f}y"
        in_region = {c: m for c, m in cand.worked.items() if regions.get(c) == target_region}
        if in_region:
            c, m = max(in_region.items(), key=lambda kv: kv[1])
            factor = w.location_region_factor if place.kind == "country" else 1.0
            return base * factor, f"Worked in {c} ({target_region}) {m / 12:.1f}y"
        return 0.0, None

    if scope == "current":
        return current()
    if scope == "worked":
        return worked()
    return max(current(), worked(), key=lambda t: t[0])


def _years(cand: Candidate, lo: float | None, hi: float | None, decay: float) -> float:
    y = cand.months_relevant / 12
    dist = max((lo - y) if lo is not None else 0.0, (y - hi) if hi is not None else 0.0, 0.0)
    return max(0.0, 1.0 - dist / decay) if decay > 0 else float(dist == 0)


def _skill_match(cand: Candidate, skill: str, threshold: float) -> str | None:
    token = canonical_skill(skill)
    if token in cand.skills:
        return token
    best = max(cand.skills, key=lambda s: fuzz.ratio(token, s), default=None)
    if best is not None and fuzz.ratio(token, best) >= threshold:
        return best
    return None


def _range_label(lo: float | None, hi: float | None) -> str:
    if lo is not None and hi is not None:
        return f"{lo:g}-{hi:g}y"
    return f"{lo:g}y+" if lo is not None else f"up to {hi:g}y"


def _terms(req: Requisition, cand: Candidate, w: Weights) -> list[_Term]:
    out: list[_Term] = []

    def crit(c: Criterion | None) -> tuple[list[str], str]:
        return _values(c), (c.mode if c else "preferred")

    if req.location:
        place = resolve_location(req.location.value)
        frac, detail = _location(cand, place, req.location_scope, w)
        label = f"Location {place.code}" + ("" if req.location_scope == "either" else f" ({req.location_scope})")
        out.append(_term("location", req.location.mode, w.weight("location"), frac, label,
                         detail or "No match"))

    if req.approach:
        want = req.approach.value
        ok = cand.approach == want or cand.approach == "Both"
        out.append(_term("approach", req.approach.mode, w.weight("approach"), float(ok),
                         f"Approach {want}", cand.approach))

    vals, mode = crit(req.sectors)
    if vals:
        chips = tuple(
            Chip("sector", v, ChipStatus.matched if v in cand.sectors else ChipStatus.unmatched,
                 cand.sectors.get(v))
            for v in vals
        )
        frac = sum(v in cand.sectors for v in vals) / len(vals)
        out.append(_Term("sector", mode, w.weight("sector"), frac, chips))

    vals, mode = crit(req.markets_covered)
    if vals:
        cov = {c.casefold() for c in cand.covered}
        hit = [v for v in vals if v.casefold() in cov]
        out.append(_term("markets_covered", mode, w.weight("markets_covered"), float(bool(hit)),
                         "Markets " + "/".join(vals), ", ".join(hit) or "No overlap"))

    if req.years_min is not None or req.years_max is not None:
        frac = _years(cand, req.years_min, req.years_max, w.years_decay)
        out.append(_term("years", req.years_mode, w.weight("years_in_range"), frac,
                         f"Years {_range_label(req.years_min, req.years_max)}",
                         f"{cand.months_relevant / 12:.1f}y relevant", matched=frac >= 1.0))

    vals, mode = crit(req.seniority_bands)
    if vals:
        out.append(_term("seniority", mode, w.weight("seniority_band"),
                         float(cand.seniority_band in vals), "Level " + "/".join(vals),
                         cand.seniority_band))

    for skills, smode, key in (
        (req.skills_required, "must", "skill_preferred"),
        (req.skills_preferred, "preferred", "skill_preferred"),
    ):
        for s in skills:
            found = _skill_match(cand, s, w.fuzzy_skill_threshold)
            out.append(_term("skill", smode, w.weight(key), float(found is not None), s, found))

    if req.degree_min:
        need = DEGREES.index(req.degree_min.value)
        have = DEGREES.index(cand.highest_degree) if cand.highest_degree in DEGREES else -1
        out.append(_term("degree", req.degree_min.mode, w.weight("degree"), float(have >= need),
                         f"Degree {req.degree_min.value}+", cand.highest_degree))

    vals, mode = crit(req.certifications)
    for v in vals:
        key = v.casefold()
        found = next((c for c in cand.certs if key in c.casefold()), None)
        if found is None and key == "cfa" and cand.has_cfa:
            found = "CFA"
        out.append(_term("certification", mode, w.weight("certification"), float(found is not None),
                         v, found))

    vals, mode = crit(req.languages)
    have_langs = {x.casefold() for x in cand.languages}
    for v in vals:
        out.append(_term("language", mode, w.weight("language"), float(v.casefold() in have_langs), v, None))

    vals, mode = crit(req.asset_classes)
    if vals:
        hit = [a for a in vals if a in cand.assets]
        out.append(_term("asset_class", mode, w.weight("asset_class"), float(bool(hit)),
                         "Assets " + "/".join(vals), ", ".join(hit) or "No overlap"))
    return out


def rule_score(
    req: Requisition, cand: Candidate, weights: Weights
) -> tuple[float, list[tuple[str, float, float]], list[Chip]]:
    """Weighted mean of match fractions over the preferred criteria; chips cover every criterion
    the user set. Score is 0 when there are no preferred criteria."""
    terms = _terms(req, cand, weights)
    scored = [t for t in terms if t.mode == "preferred"]
    total = sum(t.weight for t in scored)
    score = sum(t.weight * t.fraction for t in scored) / total if total else 0.0
    return score, [(t.criterion, t.weight, t.fraction) for t in scored], [c for t in terms for c in t.chips]

Overwriting src/resume_search/search/rules.py


#### `src/resume_search/search/vectors.py`

Embeds text with `BAAI/bge-small-en-v1.5` and scores candidates by cosine similarity over summary and role vectors.

In [41]:
%%writefile src/resume_search/search/vectors.py
from __future__ import annotations

import sqlite3
from dataclasses import dataclass

import numpy as np

from resume_search.derive.models import CandidateRecord

MODEL_NAME = "BAAI/bge-small-en-v1.5"
DIM = 384


class Embedder:
    """Loads the model on first use; one instance per process is enough."""

    def __init__(self, model_name: str = MODEL_NAME):
        self.model_name = model_name
        self._model = None

    def embed(self, texts: list[str]) -> np.ndarray:
        if self._model is None:
            from sentence_transformers import SentenceTransformer

            self._model = SentenceTransformer(self.model_name, device="cpu")
        if not texts:
            return np.zeros((0, DIM), dtype=np.float32)
        out = self._model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        return np.asarray(out, dtype=np.float32)


_default = Embedder()


def default_embedder() -> Embedder:
    return _default


def to_blob(vec: np.ndarray) -> bytes:
    return np.asarray(vec, dtype="<f4").tobytes()


def from_blob(blob: bytes) -> np.ndarray:
    return np.frombuffer(blob, dtype="<f4")


def candidate_vectors(
    record: CandidateRecord, embedder: Embedder | None = None
) -> list[tuple[str, int, np.ndarray]]:
    """One vector for the search summary and one per experience entry. Neither text carries
    contact data (the summary validator forbids it; bullets are job content)."""
    ex = record.extracted
    texts = [record.derived.search_summary] + [
        f"{e.title} at {e.employer}. " + " ".join(e.bullets) for e in ex.experience
    ]
    vecs = (embedder or _default).embed(texts)
    keys = [("summary", 0)] + [("role", i) for i in range(len(ex.experience))]
    return [(kind, idx, v) for (kind, idx), v in zip(keys, vecs, strict=True)]


@dataclass(frozen=True)
class VectorHit:
    candidate_id: str
    score: float
    kind: str
    idx: int


class VectorIndex:
    def __init__(self, keys: list[tuple[str, str, int]], matrix: np.ndarray):
        self.keys = keys
        self.matrix = matrix

    @classmethod
    def load(cls, conn: sqlite3.Connection) -> VectorIndex:
        rows = conn.execute(
            "SELECT candidate_id, kind, idx, vec FROM embeddings ORDER BY candidate_id, kind, idx"
        ).fetchall()
        keys = [(r["candidate_id"], r["kind"], r["idx"]) for r in rows]
        matrix = (
            np.stack([from_blob(r["vec"]) for r in rows])
            if rows
            else np.zeros((0, DIM), dtype=np.float32)
        )
        return cls(keys, matrix)

    def scores(self, query_vec: np.ndarray) -> dict[str, float]:
        """Cosine (vectors are normalized, so a dot product), best vector per candidate."""
        if not self.keys:
            return {}
        sims = self.matrix @ np.asarray(query_vec, dtype=np.float32)
        best: dict[str, float] = {}
        for (cid, _, _), score in zip(self.keys, sims, strict=True):
            best[cid] = max(best.get(cid, -1.0), float(score))
        return best

    def search(self, query_vec: np.ndarray, top_k: int = 10) -> list[VectorHit]:
        """Cosine (vectors are normalized, so a dot product), best vector per candidate."""
        if not self.keys:
            return []
        scores = self.matrix @ np.asarray(query_vec, dtype=np.float32)
        best: dict[str, VectorHit] = {}
        for (cid, kind, idx), score in zip(self.keys, scores, strict=True):
            if cid not in best or score > best[cid].score:
                best[cid] = VectorHit(cid, float(score), kind, idx)
        return sorted(best.values(), key=lambda h: -h.score)[:top_k]

Overwriting src/resume_search/search/vectors.py


#### `src/resume_search/search/weights.py`

Loads rule weights and the blend weights from `taxonomy/weights.yaml`.

In [42]:
%%writefile src/resume_search/search/weights.py
from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass
from pathlib import Path
from types import MappingProxyType

import yaml

from resume_search import taxonomy

# Criteria the design table has no weight for (seniority band, language) count as one skill.
DEFAULT_WEIGHT = 1.0


@dataclass(frozen=True)
class Weights:
    rule: Mapping[str, float]
    location_current: float = 1.0
    location_worked: float = 0.5
    location_region_factor: float = 0.5
    blend_rule: float = 0.5
    blend_keyword: float = 0.2
    blend_embedding: float = 0.3
    years_decay: float = 2.0
    fuzzy_skill_threshold: float = 90.0

    def weight(self, key: str) -> float:
        return float(self.rule.get(key, DEFAULT_WEIGHT))

    @classmethod
    def load(cls, path: str | Path | None = None) -> Weights:
        raw = yaml.safe_load(Path(path or taxonomy.TAXONOMY_DIR / "weights.yaml").read_text("utf-8"))
        loc, blend = raw["location"], raw["blend"]
        return cls(
            rule=MappingProxyType({k: float(v) for k, v in raw["rule"].items()}),
            location_current=float(loc["current_match"]),
            location_worked=float(loc["worked_match"]),
            location_region_factor=float(loc["region_only_factor"]),
            blend_rule=float(blend["rule"]),
            blend_keyword=float(blend["keyword"]),
            blend_embedding=float(blend["embedding"]),
            years_decay=float(raw["years_decay_outside_range"]),
            fuzzy_skill_threshold=float(raw["fuzzy_skill_threshold"]),
        )

Overwriting src/resume_search/search/weights.py


#### `src/resume_search/store/db.py`

Opens the database, loads taxonomy tables, and writes or reads one candidate atomically.

In [43]:
%%writefile src/resume_search/store/db.py
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import yaml

from resume_search import taxonomy as tx
from resume_search.derive import _today
from resume_search.derive.experience import role_months
from resume_search.derive.models import CandidateRecord
from resume_search.search.vectors import MODEL_NAME, to_blob

SCHEMA = (Path(__file__).with_name("schema.sql")).read_text(encoding="utf-8")

_CHILD_TABLES = (
    "experience", "education", "candidate_sectors", "candidate_tags", "candidate_assets",
    "candidate_skills", "candidate_countries", "candidate_certs", "candidate_languages",
    "candidate_fts", "embeddings",
)


def connect(path: str | Path, *, check_same_thread: bool = True) -> sqlite3.Connection:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(path, check_same_thread=check_same_thread)
    conn.row_factory = sqlite3.Row
    conn.executescript(SCHEMA)
    return conn


def _flatten(d: dict, prefix: str = "") -> dict[str, float]:
    out: dict[str, float] = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(_flatten(v, f"{key}."))
        else:
            out[key] = float(v)
    return out


def load_taxonomy(conn: sqlite3.Connection) -> None:
    sectors = tx._load("sectors")
    with conn:
        conn.executemany(
            "INSERT OR REPLACE INTO taxonomy_country VALUES (?,?,?)",
            [(iso, c["name"], c["region"]) for iso, c in tx.countries().items()],
        )
        conn.executemany(
            "INSERT OR REPLACE INTO taxonomy_sector VALUES (?,?)",
            [(s, None) for s in sectors["core"]]
            + [(str(t), core) for t, core in sectors["tag_to_core"].items()],
        )
        conn.executemany(
            "INSERT OR REPLACE INTO taxonomy_skill_synonym VALUES (?,?,?)",
            [(k, v["canonical"], v["category"]) for k, v in tx.skill_variants().items()],
        )
        weights = yaml.safe_load((tx.TAXONOMY_DIR / "weights.yaml").read_text(encoding="utf-8"))
        conn.executemany(
            "INSERT OR REPLACE INTO ranking_weights VALUES (?,?)", _flatten(weights).items()
        )


def upsert_candidate(
    conn: sqlite3.Connection,
    record: CandidateRecord,
    fts_body: str,
    vectors: list[tuple[str, int, np.ndarray]],
) -> None:
    """Replace one candidate's rows atomically: a failure leaves the previous version intact."""
    meta, ex, d = record.meta, record.extracted, record.derived
    cid = meta["candidate_id"]
    today = _today(meta)
    with conn:
        for table in _CHILD_TABLES:
            conn.execute(f"DELETE FROM {table} WHERE candidate_id = ?", (cid,))
        conn.execute(
            "INSERT OR REPLACE INTO candidates VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (
                cid, ex.name, ex.contact.email, ex.contact.phone, ex.contact.location_text,
                d.current_country, d.current_region,
                d.months_full_time_relevant, d.months_full_time_core, d.months_internship,
                d.seniority_band, d.investment_approach, d.approach_evidence,
                d.highest_degree, int(d.has_cfa), d.cfa_level, d.search_summary,
                record.model_dump_json(),
                meta.get("prompt_version"), meta.get("schema_version"), meta.get("parse_status"),
                meta.get("parsed_at"), meta.get("file_sha256"), meta.get("source_file"),
            ),
        )
        conn.executemany(
            "INSERT INTO experience VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
            [
                (cid, i, e.employer, e.title, e.country, e.start, e.end, int(e.is_current),
                 e.employment_type.value, e.relevance.value, e.firm_type.value,
                 role_months(e, today))
                for i, e in enumerate(ex.experience)
            ],
        )
        conn.executemany(
            "INSERT INTO education VALUES (?,?,?,?,?,?,?,?,?)",
            [
                (cid, i, e.institution, e.degree, e.degree_level, e.field, e.start, e.end, e.country)
                for i, e in enumerate(ex.education)
            ],
        )
        conn.executemany(
            "INSERT INTO candidate_sectors VALUES (?,?,?)",
            [(cid, s.label, s.evidence) for s in d.sectors],
        )
        conn.executemany("INSERT INTO candidate_tags VALUES (?,?)", [(cid, t) for t in d.other_sector_tags])
        conn.executemany("INSERT INTO candidate_assets VALUES (?,?)", [(cid, a) for a in d.asset_classes])
        conn.executemany(
            "INSERT INTO candidate_skills VALUES (?,?,?)", [(cid, s, "llm") for s in d.skills_normalized]
        )
        conn.executemany(
            "INSERT INTO candidate_countries VALUES (?,?,?,?)",
            [(cid, c.country, c.months, "worked") for c in d.countries_worked]
            + [(cid, m, None, "covered") for m in d.markets_covered],
        )
        conn.executemany("INSERT INTO candidate_certs VALUES (?,?)", [(cid, c) for c in ex.certifications])
        conn.executemany(
            "INSERT INTO candidate_languages VALUES (?,?,?)",
            [(cid, lang.name, lang.level) for lang in ex.languages],
        )
        conn.execute("INSERT INTO candidate_fts (candidate_id, body) VALUES (?,?)", (cid, fts_body))
        conn.executemany(
            "INSERT INTO embeddings VALUES (?,?,?,?,?,?)",
            [(cid, kind, idx, MODEL_NAME, int(vec.shape[0]), to_blob(vec)) for kind, idx, vec in vectors],
        )


def get_candidate(conn: sqlite3.Connection, candidate_id: str) -> CandidateRecord | None:
    row = conn.execute(
        "SELECT record_json FROM candidates WHERE candidate_id = ?", (candidate_id,)
    ).fetchone()
    return CandidateRecord.model_validate_json(row["record_json"]) if row else None


def list_candidates(conn: sqlite3.Connection) -> list[CandidateRecord]:
    rows = conn.execute("SELECT record_json FROM candidates ORDER BY name").fetchall()
    return [CandidateRecord.model_validate_json(r["record_json"]) for r in rows]

Overwriting src/resume_search/store/db.py


#### `src/resume_search/store/export.py`

Writes `candidates.json` and `candidates.csv` from the database.

In [44]:
%%writefile src/resume_search/store/export.py
from __future__ import annotations

import csv
import json
import sqlite3
from pathlib import Path

from resume_search.store.db import list_candidates

# facet column -> (table, value column, extra WHERE)
_FACETS = {
    "sectors": ("candidate_sectors", "sector", ""),
    "tags": ("candidate_tags", "tag", ""),
    "asset_classes": ("candidate_assets", "asset_class", ""),
    "skills": ("candidate_skills", "skill", ""),
    "countries_worked": ("candidate_countries", "country", "AND kind = 'worked'"),
    "certs": ("candidate_certs", "cert", ""),
}


def export_json(conn: sqlite3.Connection, path: str | Path) -> int:
    records = [r.model_dump(mode="json") for r in list_candidates(conn)]
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")
    return len(records)


def export_csv(conn: sqlite3.Connection, path: str | Path) -> int:
    """The candidates table without record_json (it is the JSON export), plus facet columns."""
    cols = [r["name"] for r in conn.execute("PRAGMA table_info(candidates)") if r["name"] != "record_json"]
    rows = conn.execute(f"SELECT {', '.join(cols)} FROM candidates ORDER BY name").fetchall()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([*cols, *_FACETS])
        for row in rows:
            facets = [
                ";".join(
                    r[0]
                    for r in conn.execute(
                        f"SELECT {col} FROM {table} WHERE candidate_id = ? {extra} ORDER BY rowid",
                        (row["candidate_id"],),
                    )
                    if r[0] is not None
                )
                for table, col, extra in _FACETS.values()
            ]
            w.writerow([*tuple(row), *facets])
    return len(rows)

Overwriting src/resume_search/store/export.py


#### `src/resume_search/store/fts.py`

Builds the keyword-index body by removing email, URL, and phone lines from the resume text.

In [45]:
%%writefile src/resume_search/store/fts.py
from __future__ import annotations

import re

_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
_URL = re.compile(r"https?://|www\.|linkedin\.com|github\.com", re.IGNORECASE)
# Anywhere in the text only an explicit phone shape counts; a bare digit run would also match
# date ranges like "2018-01 - 2020-03".
_PHONE_STRICT = re.compile(
    r"\+\d[\d\s().-]{7,}\d|\(\d{3}\)\s*\d{3}[-.\s]\d{4}|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"
)
_PHONE_LOOSE = re.compile(r"\d[\d\s().-]{7,}\d")
_YEAR = re.compile(r"\b(?:19|20)\d{2}\b")
HEADER_LINES = 8


def fts_body(text: str) -> str:
    """Resume text without the contact block: email, URL and phone lines are dropped. The first
    lines also get a looser phone test, unless they carry a year."""
    kept = []
    for i, line in enumerate(text.splitlines()):
        if _EMAIL.search(line) or _URL.search(line) or _PHONE_STRICT.search(line):
            continue
        if i < HEADER_LINES and _PHONE_LOOSE.search(line) and not _YEAR.search(line):
            continue
        kept.append(line)
    return "\n".join(kept)

Overwriting src/resume_search/store/fts.py


#### `src/resume_search/store/schema.sql`

The SQLite DDL: candidate and facet tables, the FTS5 index, embeddings, `parse_jobs`, and taxonomy mirrors.

In [46]:
%%writefile src/resume_search/store/schema.sql
CREATE TABLE IF NOT EXISTS candidates (
  candidate_id TEXT PRIMARY KEY, name TEXT NOT NULL,
  email TEXT, phone TEXT, location_text TEXT,
  current_country TEXT, current_region TEXT,
  months_full_time_relevant INTEGER, months_full_time_core INTEGER, months_internship INTEGER,
  seniority_band TEXT, investment_approach TEXT, approach_evidence TEXT,
  highest_degree TEXT, has_cfa INTEGER, cfa_level TEXT,
  search_summary TEXT NOT NULL,
  record_json TEXT NOT NULL,
  prompt_version TEXT, schema_version INTEGER, parse_status TEXT, parsed_at TEXT,
  file_sha256 TEXT UNIQUE, source_file TEXT
);
CREATE TABLE IF NOT EXISTS experience (
  candidate_id TEXT REFERENCES candidates, idx INTEGER,
  employer TEXT, title TEXT, country TEXT, start TEXT, end TEXT, is_current INTEGER,
  employment_type TEXT, relevance TEXT, firm_type TEXT, months INTEGER,
  PRIMARY KEY (candidate_id, idx)
);
CREATE TABLE IF NOT EXISTS education (candidate_id TEXT, idx INTEGER, institution TEXT, degree TEXT,
  degree_level TEXT, field TEXT, start TEXT, end TEXT, country TEXT, PRIMARY KEY (candidate_id, idx));
CREATE TABLE IF NOT EXISTS candidate_sectors   (candidate_id TEXT, sector TEXT, evidence TEXT);
CREATE TABLE IF NOT EXISTS candidate_tags      (candidate_id TEXT, tag TEXT);
CREATE TABLE IF NOT EXISTS candidate_assets    (candidate_id TEXT, asset_class TEXT);
CREATE TABLE IF NOT EXISTS candidate_skills    (candidate_id TEXT, skill TEXT, source TEXT);
CREATE TABLE IF NOT EXISTS candidate_countries (candidate_id TEXT, country TEXT, months INTEGER, kind TEXT);
CREATE TABLE IF NOT EXISTS candidate_certs     (candidate_id TEXT, cert TEXT);
CREATE TABLE IF NOT EXISTS candidate_languages (candidate_id TEXT, language TEXT, level TEXT);
CREATE VIRTUAL TABLE IF NOT EXISTS candidate_fts USING fts5(candidate_id UNINDEXED, body, tokenize='porter unicode61');
CREATE TABLE IF NOT EXISTS embeddings (candidate_id TEXT, kind TEXT, idx INTEGER, model TEXT, dim INTEGER,
  vec BLOB, PRIMARY KEY (candidate_id, kind, idx));
CREATE TABLE IF NOT EXISTS parse_jobs (
  job_id INTEGER PRIMARY KEY, file_sha256 TEXT, source_file TEXT, prompt_version TEXT,
  status TEXT,
  attempts_json TEXT, input_tokens INTEGER, output_tokens INTEGER, cost_usd REAL,
  latency_ms INTEGER, error TEXT, queued_at TEXT, finished_at TEXT
);
CREATE TABLE IF NOT EXISTS taxonomy_country (iso2 TEXT PRIMARY KEY, name TEXT, region TEXT);
CREATE TABLE IF NOT EXISTS taxonomy_sector (name TEXT PRIMARY KEY, parent TEXT);
CREATE TABLE IF NOT EXISTS taxonomy_skill_synonym (variant TEXT PRIMARY KEY, canonical TEXT, category TEXT);
CREATE TABLE IF NOT EXISTS ranking_weights (criterion TEXT PRIMARY KEY, weight REAL);
CREATE INDEX IF NOT EXISTS idx_cand_country ON candidates(current_country);
CREATE INDEX IF NOT EXISTS idx_cand_months ON candidates(months_full_time_relevant);
CREATE INDEX IF NOT EXISTS idx_exp_country ON experience(country);
CREATE INDEX IF NOT EXISTS idx_jobs_key ON parse_jobs(file_sha256, prompt_version);

Overwriting src/resume_search/store/schema.sql


#### `src/resume_search/taxonomy.py`

Loads the YAML files in `taxonomy/` into one plain-data view of countries, sectors, firm types, asset classes, and skill synonyms.

In [47]:
%%writefile src/resume_search/taxonomy.py
"""Loader for the YAML files in taxonomy/. The single source of truth for ISO-2 codes,
sector and firm-type lists, and the vocabulary treated as known words."""

from __future__ import annotations

import os
import re
from dataclasses import dataclass
from functools import cache
from pathlib import Path

import yaml

TAXONOMY_DIR = Path(
    os.environ.get("RESUME_SEARCH_TAXONOMY", Path(__file__).resolve().parents[2] / "taxonomy")
)

_TOKEN = re.compile(r"[^\W_]+(?:['’][^\W_]+)?")


def tokenize(text: str) -> list[str]:
    return _TOKEN.findall(text)


@cache
def _load(name: str) -> dict:
    return yaml.safe_load((TAXONOMY_DIR / f"{name}.yaml").read_text(encoding="utf-8"))


@cache
def countries() -> dict[str, dict]:
    return _load("countries")["countries"]


@cache
def iso2_codes() -> frozenset[str]:
    return frozenset(countries())


@cache
def market_labels() -> frozenset[str]:
    data = _load("countries")
    return frozenset(data["market_labels"]) | frozenset(data["regions"])


def sectors() -> list[str]:
    return _load("sectors")["core"]


def firm_types() -> list[str]:
    return _load("firm_types")["types"]


def asset_classes() -> list[str]:
    return _load("asset_classes")["classes"]


@cache
def taxonomy_terms() -> frozenset[str]:
    """Lowercased word tokens from skills, firms, countries, aliases, sectors."""
    phrases: list[str] = []
    phrases += _load("skills")["skills"]
    phrases += _load("firm_types")["known_firms"]
    for c in countries().values():
        phrases += [c["name"], *c.get("aliases", [])]
    phrases += _load("sectors")["tag_to_core"]
    phrases += _load("asset_classes")["synonyms"]
    phrases += _load("countries")["market_labels"]
    return frozenset(t.lower() for p in phrases for t in tokenize(str(p)))


@cache
def skill_variants() -> dict[str, dict]:
    """variant (lowercase) -> {canonical, category} from skills.yaml."""
    return {str(k).lower(): v for k, v in _load("skills")["skills"].items()}


@cache
def phone_prefixes() -> dict[str, list[str]]:
    """Calling prefix ('+852') -> ISO-2 codes that share it, from countries.yaml."""
    out: dict[str, list[str]] = {}
    for iso, c in countries().items():
        for prefix in c.get("phone", []):
            out.setdefault(prefix, []).append(iso)
    return out


@dataclass(frozen=True)
class Taxonomy:
    """Plain-data view of the YAML files so derive() stays pure and tests can hand-build one."""

    country_region: dict[str, str]
    country_names: dict[str, tuple[str, ...]]
    phone_prefix: dict[str, tuple[str, ...]]
    # market label -> (ISO-2 codes it expands to, regions it expands to)
    market_expansion: dict[str, tuple[frozenset[str], frozenset[str]]]
    skill_canonical: dict[str, str]
    tag_to_core: dict[str, str]


def load_taxonomy_data() -> Taxonomy:
    country_region = {iso: c["region"] for iso, c in countries().items()}
    expansion: dict[str, tuple[frozenset[str], frozenset[str]]] = {
        iso: (frozenset({iso}), frozenset({region})) for iso, region in country_region.items()
    }
    for label, target in _load("countries")["market_labels"].items():
        if target is None:
            expansion[label] = (frozenset(), frozenset())
        elif isinstance(target, str):
            expansion[label] = (frozenset(), frozenset({target}))
        else:
            expansion[label] = (
                frozenset(target),
                frozenset(country_region[c] for c in target),
            )
    for region in _load("countries")["regions"]:
        expansion.setdefault(region, (frozenset(), frozenset({region})))
    return Taxonomy(
        country_region=country_region,
        country_names={
            iso: (c["name"], *c.get("aliases", [])) for iso, c in countries().items()
        },
        phone_prefix={p: tuple(v) for p, v in phone_prefixes().items()},
        market_expansion=expansion,
        skill_canonical={k: v["canonical"] for k, v in skill_variants().items()},
        tag_to_core={str(k).lower(): v for k, v in _load("sectors")["tag_to_core"].items()},
    )

Overwriting src/resume_search/taxonomy.py


### Streamlit app: app

#### `app/Home.py`

Landing page with database counts and links to the other pages.

In [48]:
%%writefile app/Home.py
import common
import streamlit as st

st.set_page_config(page_title="Resume Search", layout="wide")
st.title("Candidate resume search")
st.write(
    "Search hedge fund candidates parsed from DOCX and PDF resumes. Each resume is read by an LLM into "
    "a structured record, normalized in code, stored in SQLite with a keyword index and "
    "embeddings, and ranked against a requisition by a rule score, BM25, and embedding "
    "similarity. Ingest new resumes, inspect any candidate, and see how the pool is distributed."
)

path = common.db_path()
st.caption(f"Database: {path} (set RESUME_DB to change)")
db = common.conn()
n_cand = db.execute("SELECT COUNT(*) FROM candidates").fetchone()[0]
n_jobs = db.execute("SELECT COUNT(*) FROM parse_jobs").fetchone()[0]
c1, c2 = st.columns(2)
c1.metric("Candidates", n_cand)
c2.metric("Parse jobs", n_jobs)

st.page_link("pages/1_Search.py", label="Search candidates")
st.page_link("pages/2_Candidate.py", label="Candidate detail")
st.page_link("pages/3_Insights.py", label="Insights")
st.page_link("pages/4_Ingest.py", label="Ingest resumes")
st.page_link("pages/5_Evals.py", label="Evals and quality")

Overwriting app/Home.py


#### `app/common.py`

Shared Streamlit helpers: cached resources, the widget-to-`Requisition` mapping, and chip rendering.

In [49]:
%%writefile app/common.py
"""Shared Streamlit helpers: cached resources, the requisition <-> widget mapping, chip rendering."""

from __future__ import annotations

import json
import os
from dataclasses import replace
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv

from resume_search import taxonomy as tx
from resume_search.search import Chip, ChipStatus, Criterion, Requisition, SearchResult, Weights
from resume_search.search import search as run_ranked_search
from resume_search.search.requisition import resolve_location
from resume_search.search.vectors import Embedder, VectorIndex
from resume_search.store.db import connect

load_dotenv()

DEFAULT_DB = "exports/candidates.sqlite"
ANY = "(any)"
YEARS_MAX = 25
# Criteria that carry a must/preferred mode on the Requisition.
MODE_FIELDS = (
    "location", "approach", "sectors", "markets_covered", "seniority_bands", "degree_min",
    "certifications", "languages", "asset_classes",
)
MODE_LABELS = {
    "location": "Location", "approach": "Approach", "sectors": "Sectors",
    "markets_covered": "Markets covered", "seniority_bands": "Seniority",
    "degree_min": "Degree", "certifications": "Certifications", "languages": "Languages",
    "asset_classes": "Asset classes",
}


def db_path() -> str:
    return os.environ.get("RESUME_DB", DEFAULT_DB)


def uploads_dir() -> Path:
    return Path(db_path()).resolve().parent / "uploads"


# Cached resources -------------------------------------------------------------------------------


@st.cache_resource
def get_conn(path: str):
    return connect(path, check_same_thread=False)


def conn():
    return get_conn(db_path())


@st.cache_resource
def get_embedder() -> Embedder:
    return Embedder()


@st.cache_resource
def get_index(path: str) -> VectorIndex:
    return VectorIndex.load(get_conn(path))


@st.cache_resource
def get_weights() -> Weights:
    return Weights.load()


@st.cache_resource
def get_taxonomy() -> dict:
    countries = tx.countries()
    return {
        "regions": list(tx._load("countries")["regions"]),
        "countries": {iso: c["name"] for iso, c in countries.items()},
        "sectors": tx.sectors(),
        "asset_classes": tx.asset_classes(),
        "markets": sorted(tx.market_labels() | tx.iso2_codes()),
    }


def weights_to_dict(w: Weights) -> dict:
    return {**{f: getattr(w, f) for f in Weights.__dataclass_fields__ if f != "rule"}, "rule": dict(w.rule)}


def weights_from_dict(d: dict) -> Weights:
    return replace(get_weights(), **{**d, "rule": dict(d["rule"])})


@st.cache_data(show_spinner="Searching")
def run_search(req_json: str, weights: dict, path: str, top_k: int = 20) -> list[SearchResult]:
    return run_ranked_search(
        get_conn(path), Requisition.model_validate_json(req_json), weights_from_dict(weights),
        embedder=get_embedder(), index=get_index(path), top_k=top_k,
    )


def clear_caches() -> None:
    """After ingest: new rows invalidate search results and the vector index. The embedder and
    the connection stay."""
    get_index.clear()
    st.cache_data.clear()


# Requisition <-> widget state -------------------------------------------------------------------


def location_options() -> dict[str, str]:
    t = get_taxonomy()
    out = {ANY: ""}
    out.update({f"Region: {r}": r for r in t["regions"]})
    out.update({f"{name} ({iso})": iso for iso, name in sorted(t["countries"].items(), key=lambda kv: kv[1])})
    return out


def _csv(text: str) -> list[str]:
    return [p.strip() for p in (text or "").split(",") if p.strip()]


def default_state() -> dict:
    return {
        "f_location": ANY, "f_scope": "either", "f_approach": ANY, "f_sectors": [],
        "f_markets": [], "f_years": (0, YEARS_MAX), "f_years_hard": True, "f_bands": [],
        "f_degree": ANY, "f_cfa": False, "f_certs": "", "f_skills_req": "", "f_skills_pref": "",
        "f_languages": "", "f_assets": [], "f_must": [], "jd_text": "",
    }


def init_state() -> None:
    for k, v in default_state().items():
        st.session_state.setdefault(k, v)


def state_from_requisition(req: Requisition) -> dict:
    """Widget values that reproduce `req`. Values outside a widget's options are dropped."""
    s = default_state()
    s["jd_text"] = req.free_text or ""
    if req.location:
        place = resolve_location(req.location.value)
        s["f_location"] = next((k for k, v in location_options().items() if v == place.code), ANY)
    s["f_scope"] = req.location_scope
    if req.approach:
        s["f_approach"] = req.approach.value
    if req.sectors:
        s["f_sectors"] = list(req.sectors.value)
    if req.markets_covered:
        s["f_markets"] = [m for m in req.markets_covered.value if m in get_taxonomy()["markets"]]
    lo, hi = req.years_min, req.years_max
    s["f_years"] = (int(lo or 0), int(min(hi, YEARS_MAX)) if hi is not None else YEARS_MAX)
    s["f_years_hard"] = req.years_mode == "must"
    if req.seniority_bands:
        s["f_bands"] = list(req.seniority_bands.value)
    if req.degree_min:
        s["f_degree"] = req.degree_min.value
    certs = list(req.certifications.value) if req.certifications else []
    s["f_cfa"] = any(c.upper() == "CFA" for c in certs)
    s["f_certs"] = ", ".join(c for c in certs if c.upper() != "CFA")
    s["f_skills_req"] = ", ".join(req.skills_required)
    s["f_skills_pref"] = ", ".join(req.skills_preferred)
    if req.languages:
        s["f_languages"] = ", ".join(req.languages.value)
    if req.asset_classes:
        s["f_assets"] = list(req.asset_classes.value)
    s["f_must"] = [f for f in MODE_FIELDS if (c := getattr(req, f)) is not None and c.mode == "must"]
    return s


def set_criteria(s: dict) -> list[str]:
    """The mode-carrying criteria the widget state sets, in MODE_FIELDS order."""
    present = {
        "location": s["f_location"] != ANY, "approach": s["f_approach"] != ANY,
        "sectors": bool(s["f_sectors"]), "markets_covered": bool(s["f_markets"]),
        "seniority_bands": bool(s["f_bands"]), "degree_min": s["f_degree"] != ANY,
        "certifications": bool(s["f_cfa"] or _csv(s["f_certs"])),
        "languages": bool(_csv(s["f_languages"])), "asset_classes": bool(s["f_assets"]),
    }
    return [f for f in MODE_FIELDS if present[f]]


def requisition_from_state(s: dict) -> Requisition:
    must = set(s["f_must"])

    def crit(field: str, value):
        mode = "must" if field in must else "preferred"
        return Criterion(value=value, mode=mode)

    lo, hi = s["f_years"]
    certs = (["CFA"] if s["f_cfa"] else []) + _csv(s["f_certs"])
    kw: dict = {}
    if s["f_location"] != ANY:
        kw["location"] = crit("location", location_options()[s["f_location"]])
    if s["f_approach"] != ANY:
        kw["approach"] = crit("approach", s["f_approach"])
    for field, key, vals in (
        ("sectors", "f_sectors", s["f_sectors"]), ("markets_covered", "f_markets", s["f_markets"]),
        ("seniority_bands", "f_bands", s["f_bands"]), ("certifications", None, certs),
        ("languages", None, _csv(s["f_languages"])), ("asset_classes", "f_assets", s["f_assets"]),
    ):
        if vals:
            kw[field] = crit(field, list(vals))
    if s["f_degree"] != ANY:
        kw["degree_min"] = crit("degree_min", s["f_degree"])
    return Requisition(
        free_text=(s["jd_text"] or "").strip() or None,
        location_scope=s["f_scope"],
        years_min=float(lo) if lo > 0 else None,
        years_max=float(hi) if hi < YEARS_MAX else None,
        years_mode="must" if s["f_years_hard"] else "preferred",
        skills_required=_csv(s["f_skills_req"]), skills_preferred=_csv(s["f_skills_pref"]), **kw,
    )


def requisition_rows(req: Requisition) -> list[dict]:
    rows = []
    for f in MODE_FIELDS:
        c = getattr(req, f)
        if c is not None:
            v = c.value if isinstance(c.value, str) else ", ".join(c.value)
            rows.append({"criterion": MODE_LABELS[f], "value": v, "mode": c.mode})
    if req.location:
        rows.append({"criterion": "Location scope", "value": req.location_scope, "mode": ""})
    if req.years_min is not None or req.years_max is not None:
        rows.append({"criterion": "Years", "value": f"{req.years_min or 0} to {req.years_max or 'any'}",
                     "mode": req.years_mode})
    rows += [{"criterion": "Skill", "value": s, "mode": "must"} for s in req.skills_required]
    rows += [{"criterion": "Skill", "value": s, "mode": "preferred"} for s in req.skills_preferred]
    return rows


# Chips ------------------------------------------------------------------------------------------


def chip_markdown(chip: Chip) -> str:
    label = chip.label.replace("[", "(").replace("]", ")")
    detail = f" ({chip.detail})" if chip.detail else ""
    if chip.status == ChipStatus.matched:
        return f":green-badge[{label}{detail}]"
    return f":orange-badge[{label}{detail}]"


def chips(result: SearchResult) -> None:
    if not result.chips:
        return
    try:
        st.markdown(" ".join(chip_markdown(c) for c in result.chips))
    except Exception:  # noqa: BLE001  badge syntax unavailable in an older Streamlit
        st.markdown("  ".join(
            f"{'✓' if c.status == ChipStatus.matched else '✗'} {c.label}"
            + (f" ({c.detail})" if c.detail else "")
            for c in result.chips
        ))


def req_cache_key(req: Requisition) -> str:
    return json.dumps(json.loads(req.model_dump_json()), sort_keys=True)


Overwriting app/common.py


#### `app/pages/1_Search.py`

Filter sidebar, JD paste box, must and preferred toggles, ranked result cards with chips, and optional LLM rerank.

In [50]:
%%writefile app/pages/1_Search.py
import common
import pandas as pd
import streamlit as st

from resume_search.search import Requisition, parse_requisition, rerank_llm
from resume_search.search.requisition import APPROACHES, BANDS, DEGREES

st.set_page_config(page_title="Search", layout="wide")
st.title("Search")

common.init_state()
tax = common.get_taxonomy()

# A parse request set by the button is applied here, before any widget is instantiated.
if st.session_state.pop("_parse_pending", False):
    try:
        with st.spinner("Parsing requisition"):
            parsed = parse_requisition(st.session_state["jd_text"])
        st.session_state.update(common.state_from_requisition(parsed))
        st.session_state["_parsed"] = parsed
        st.session_state.pop("_parse_error", None)
    except Exception as exc:  # noqa: BLE001  surfaced to the user below
        st.session_state["_parse_error"] = f"{type(exc).__name__}: {exc}"

ss = st.session_state
locations = common.location_options()

with st.sidebar:
    st.header("Filters")
    st.selectbox("Location", list(locations), key="f_location")
    st.radio("Location applies to", ["current", "worked", "either"], key="f_scope", horizontal=True)
    st.selectbox("Approach", [common.ANY, *APPROACHES], key="f_approach")
    st.multiselect("Sectors", tax["sectors"], key="f_sectors")
    st.multiselect("Markets covered", tax["markets"], key="f_markets")
    st.slider("Years of relevant experience", 0, common.YEARS_MAX, key="f_years")
    st.checkbox("Years is a hard filter", key="f_years_hard")
    st.multiselect("Seniority bands", list(BANDS), key="f_bands")
    st.selectbox("Degree (minimum)", [common.ANY, *DEGREES], key="f_degree")
    st.checkbox("CFA", key="f_cfa")
    st.text_input("Other certifications (comma-separated)", key="f_certs")
    st.text_input("Required skills (comma-separated)", key="f_skills_req")
    st.text_input("Preferred skills (comma-separated)", key="f_skills_pref")
    st.text_input("Languages (comma-separated)", key="f_languages")
    st.multiselect("Asset classes", tax["asset_classes"], key="f_assets")
    options = common.set_criteria(ss)
    ss["f_must"] = [f for f in ss["f_must"] if f in options]
    st.multiselect(
        "Treat as must", options, key="f_must", format_func=lambda f: common.MODE_LABELS[f],
        help="Must criteria prune candidates. The rest only score. Required skills are always must.",
    )

with st.expander("Advanced: ranking weights"):
    base = common.get_weights()
    rule_w, blend_w = {}, {}
    cols = st.columns(5)
    for i, (k, v) in enumerate(base.rule.items()):
        rule_w[k] = cols[i % 5].number_input(f"rule: {k}", 0.0, 20.0, float(v), 0.5, key=f"w_rule_{k}")
    cols = st.columns(3)
    for i, name in enumerate(("blend_rule", "blend_keyword", "blend_embedding")):
        blend_w[name] = cols[i].number_input(
            name.replace("_", " "), 0.0, 1.0, float(getattr(base, name)), 0.05, key=f"w_{name}"
        )
    cols = st.columns(4)
    adv = {
        "location_current": cols[0].number_input("location: current", 0.0, 1.0, base.location_current, 0.1),
        "location_worked": cols[1].number_input("location: worked", 0.0, 1.0, base.location_worked, 0.1),
        "location_region_factor": cols[2].number_input(
            "location: region factor", 0.0, 1.0, base.location_region_factor, 0.1
        ),
        "years_decay": cols[3].number_input("years decay (y)", 0.1, 10.0, base.years_decay, 0.5),
    }
weights = {**common.weights_to_dict(base), "rule": rule_w, **blend_w, **adv}

st.text_area(
    "Job description or keywords", key="jd_text", height=160,
    help="Always feeds the keyword and embedding scores. Parse it to pre-fill the filters.",
)
c1, c2, c3 = st.columns([1, 1, 3])
c1.button(
    "Parse requisition", on_click=lambda: ss.update(_parse_pending=True),
    disabled=not ss["jd_text"].strip(), help="One API call",
)
do_search = c2.button("Search", type="primary")
top_k = c3.number_input("Top", 1, 50, 20, label_visibility="collapsed")

if ss.get("_parse_error"):
    st.error(ss["_parse_error"])
if ss.get("_parsed") is not None:
    st.caption("Parsed requisition. Edit it with the sidebar filters.")
    st.dataframe(pd.DataFrame(common.requisition_rows(ss["_parsed"])), hide_index=True)

try:
    req = common.requisition_from_state(ss)
except ValueError as exc:
    st.error(f"Invalid requisition: {exc}")
    st.stop()

if do_search:
    ss["_last"] = {"req": common.req_cache_key(req), "weights": weights, "top": int(top_k)}
last = ss.get("_last")
if last is None:
    st.info("Set filters or paste a job description, then press Search.")
    st.stop()

results = common.run_search(last["req"], last["weights"], common.db_path(), last["top"])
searched = Requisition.model_validate_json(last["req"])
if searched.is_empty():
    st.caption("Empty requisition: every candidate matches.")
st.subheader(f"{len(results)} candidates")

reasons: dict[str, str] = {}
if st.toggle("Rerank top 10 with LLM", value=False, help="One API call"):
    key = f"{last['req']}|{[r.candidate_id for r in results[:10]]}"
    cache = ss.setdefault("_rerank", {})
    if key not in cache:
        try:
            with st.spinner("Reranking"):
                cache[key] = rerank_llm(searched, results[:10])
        except Exception as exc:  # noqa: BLE001
            st.error(f"Rerank failed: {type(exc).__name__}: {exc}")
    if key in cache:
        reasons = dict(cache[key])
        by_id = {r.candidate_id: r for r in results}
        results = [by_id[i] for i, _ in cache[key]] + results[10:]

for rank, r in enumerate(results, 1):
    with st.container(border=True):
        head, score = st.columns([4, 2])
        with head:
            st.page_link(
                "pages/2_Candidate.py", label=f"{rank}. {r.name}",
                query_params={"candidate_id": r.candidate_id},
            )
            st.caption(
                f"{r.seniority_band or 'n/a'} | {r.current_country or '?'} / {r.current_region or '?'}"
            )
            st.write(r.search_summary)
        with score:
            b = r.breakdown
            st.progress(min(max(b.final, 0.0), 1.0), text=f"final {b.final:.3f}")
            st.dataframe(
                pd.DataFrame([{"rule": b.rule, "keyword": b.keyword, "embedding": b.embedding}]),
                hide_index=True,
            )
        if r.candidate_id in reasons:
            st.markdown(f"**Rerank reason.** {reasons[r.candidate_id]}")
        common.chips(r)
        if r.keywords:
            st.caption("Keyword hits: " + ", ".join(r.keywords))
        with st.expander("Rule score breakdown"):
            st.dataframe(
                pd.DataFrame(r.breakdown.rule_terms, columns=["term", "weight", "match"]),
                hide_index=True,
            )

Overwriting app/pages/1_Search.py


#### `app/pages/2_Candidate.py`

One candidate in full: derived facets, role timeline, evidence quotes, skills, and parse metadata.

In [51]:
%%writefile app/pages/2_Candidate.py
from datetime import date

import common
import pandas as pd
import plotly.express as px
import streamlit as st

from resume_search.store.db import get_candidate

st.set_page_config(page_title="Candidate", layout="wide")
st.title("Candidate")

db = common.conn()
rows = db.execute("SELECT candidate_id, name FROM candidates ORDER BY name").fetchall()
if not rows:
    st.info("No candidates yet. Ingest resumes first.")
    st.stop()
names = {r["candidate_id"]: r["name"] for r in rows}
ids = list(names)
wanted = st.query_params.get("candidate_id")
index = ids.index(wanted) if wanted in names else 0
cid = st.selectbox("Candidate", ids, index=index, format_func=lambda i: f"{names[i]} ({i})")
st.query_params["candidate_id"] = cid

rec = get_candidate(db, cid)
ex, d, meta = rec.extracted, rec.derived, rec.meta

st.header(ex.name)
st.write(d.search_summary)
c = st.columns(4)
c[0].metric("Country / region", f"{d.current_country or '?'} / {d.current_region or '?'}")
c[1].metric("Seniority", d.seniority_band)
c[2].metric("Relevant years", f"{d.months_full_time_relevant / 12:.1f}")
c[3].metric("Internship years", f"{d.months_internship / 12:.1f}")
c = st.columns(4)
c[0].metric("Core years", f"{d.months_full_time_core / 12:.1f}")
c[1].metric("Approach", d.investment_approach)
c[2].metric("Highest degree", d.highest_degree or "n/a")
c[3].metric("CFA", (d.cfa_level or "yes") if d.has_cfa else "no")
st.caption(f"Approach evidence: {d.approach_evidence}")

st.subheader("Experience")
exp = pd.read_sql(
    "SELECT idx, employer, title, employment_type AS type, relevance, firm_type, start, end, "
    "is_current, months AS duration_months, country FROM experience WHERE candidate_id = ? "
    "ORDER BY idx",
    db, params=(cid,),
)
sectors_by_role = [", ".join(s.label for s in e.sectors) for e in ex.experience]
exp["sectors"] = sectors_by_role[: len(exp)]
st.dataframe(exp.drop(columns=["idx"]), hide_index=True)


def _date(v: str | None, end: bool = False) -> pd.Timestamp | None:
    if not v:
        return None
    if len(v) == 4:
        v = f"{v}-12-31" if end else f"{v}-01-01"
    else:
        v = f"{v}-01"
        if end:
            return pd.Timestamp(v) + pd.offsets.MonthEnd(0)
    return pd.Timestamp(v)


gantt = []
for _, r in exp.iterrows():
    start = _date(r["start"])
    finish = pd.Timestamp(date.today()) if r["is_current"] else _date(r["end"], end=True)
    if start is not None and finish is not None:
        gantt.append({
            "role": f"{r['title']} @ {r['employer']}", "start": start, "end": finish,
            "relevance": r["relevance"], "type": r["type"],
        })
if gantt:
    fig = px.timeline(
        pd.DataFrame(gantt), x_start="start", x_end="end", y="role", color="relevance",
        hover_data=["type"],
    )
    fig.update_yaxes(autorange="reversed", title=None)
    st.plotly_chart(fig)
else:
    st.caption("No role has both a start and an end date.")

st.subheader("Education")
edu = pd.read_sql(
    "SELECT institution, degree, degree_level, field, start, end, country FROM education "
    "WHERE candidate_id = ? ORDER BY idx", db, params=(cid,),
)
st.dataframe(edu, hide_index=True)

st.subheader("Skills")
skills = pd.read_sql(
    "SELECT skill, source FROM candidate_skills WHERE candidate_id = ? ORDER BY skill", db,
    params=(cid,),
)
left, right = st.columns(2)
left.markdown("**LLM** (source llm or both)")
left.write(", ".join(skills[skills["source"].isin(["llm", "both"])]["skill"]) or "none")
right.markdown("**Taxonomy matched** (source ruler or both)")
right.write(", ".join(skills[skills["source"].isin(["ruler", "both"])]["skill"]) or "none")
raw_skills = ex.skills
st.caption(
    "As written. Programming: " + (", ".join(raw_skills.programming) or "none")
    + ". Tools: " + (", ".join(raw_skills.tools) or "none")
    + ". Domain: " + (", ".join(raw_skills.domain) or "none") + "."
)

c = st.columns(2)
c[0].subheader("Certifications")
c[0].write(", ".join(ex.certifications) or "none")
c[1].subheader("Languages")
c[1].write(", ".join(f"{lang.name} ({lang.level or 'n/a'})" for lang in ex.languages) or "none")

st.subheader("Sector evidence")
if d.sectors:
    for s in d.sectors:
        st.markdown(f"**{s.label}**: \"{s.evidence}\"")
else:
    st.write("none")

st.subheader("Verification")
ver = meta.get("verification") or {}
tq = meta.get("text_quality") or {}
c = st.columns(4)
c[0].metric("Checks run", ver.get("checks_run", "n/a"))
c[1].metric("Disagreements", ver.get("disagreements", "n/a"))
c[2].metric("Regenerated", ver.get("regenerated", "n/a"))
c[3].metric("Unresolved", len(ver.get("unresolved", [])))
if ver.get("unresolved"):
    st.warning("Unresolved: " + "; ".join(str(u) for u in ver["unresolved"]))
if ver.get("details"):
    with st.expander("Disagreement details"):
        st.json(ver["details"])
st.caption(
    f"Model {meta.get('llm_model')}, prompt {meta.get('prompt_version')}, "
    f"status {meta.get('parse_status')}, extraction {meta.get('extraction_path')}, "
    f"unknown-word ratio {tq.get('unknown_ratio_after', 'n/a')}"
)

with st.expander("Raw resume text (contact block removed)"):
    row = db.execute("SELECT body FROM candidate_fts WHERE candidate_id = ?", (cid,)).fetchone()
    st.text(row["body"] if row else "No text stored.")

Overwriting app/pages/2_Candidate.py


#### `app/pages/3_Insights.py`

Distributions of region, country, approach, sector, seniority, degree, skills, and firm type.

In [52]:
%%writefile app/pages/3_Insights.py
import common
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Insights", layout="wide")
st.title("Insights")

db = common.conn()


def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, db)


n = q("SELECT COUNT(*) AS n FROM candidates")["n"][0]
if n == 0:
    st.info("No candidates yet. Ingest resumes first.")
    st.stop()
st.caption(f"{n} candidates")


def bar(df: pd.DataFrame, x: str, title: str, horizontal: bool = False):
    if df.empty:
        st.caption(f"{title}: no data")
        return
    fig = px.bar(df, x="n" if horizontal else x, y=x if horizontal else "n", title=title)
    if horizontal:
        fig.update_yaxes(autorange="reversed")
    st.plotly_chart(fig)


a, b = st.columns(2)
with a:
    bar(q("SELECT COALESCE(current_region,'?') AS region, COUNT(*) AS n FROM candidates GROUP BY 1"),
        "region", "Candidates by region")
with b:
    bar(q("SELECT COALESCE(current_country,'?') AS country, COUNT(*) AS n FROM candidates "
          "GROUP BY 1 ORDER BY n DESC"), "country", "Candidates by country")
a, b = st.columns(2)
with a:
    bar(q("SELECT investment_approach AS approach, COUNT(*) AS n FROM candidates GROUP BY 1"),
        "approach", "Investment approach")
with b:
    bar(q("SELECT seniority_band AS band, COUNT(*) AS n FROM candidates GROUP BY 1"),
        "band", "Seniority band")
a, b = st.columns(2)
with a:
    bar(q("SELECT sector, COUNT(DISTINCT candidate_id) AS n FROM candidate_sectors "
          "GROUP BY 1 ORDER BY n DESC"), "sector", "Sectors (candidates can count in several)")
with b:
    bar(q("SELECT COALESCE(highest_degree,'none') AS degree, COUNT(*) AS n FROM candidates GROUP BY 1"),
        "degree", "Highest degree")
a, b = st.columns(2)
with a:
    bar(q("SELECT skill, COUNT(DISTINCT candidate_id) AS n FROM candidate_skills "
          "GROUP BY 1 ORDER BY n DESC, skill LIMIT 20"), "skill", "Top skills", horizontal=True)
with b:
    bar(q("SELECT firm_type, COUNT(*) AS n FROM experience GROUP BY 1 ORDER BY n DESC"),
        "firm_type", "Firm types (roles)")

cross = q(
    "SELECT COALESCE(current_region,'?') AS region, investment_approach AS approach, COUNT(*) AS n "
    "FROM candidates GROUP BY 1, 2"
).pivot(index="approach", columns="region", values="n").fillna(0)
st.plotly_chart(px.imshow(cross, text_auto=True, title="Approach by region", aspect="auto"))

Overwriting app/pages/3_Insights.py


#### `app/pages/4_Ingest.py`

Multi-file upload that calls `ingest()` and shows live job status, throughput, and cost.

In [53]:
%%writefile app/pages/4_Ingest.py
import threading
from datetime import datetime
from pathlib import Path

import common
import pandas as pd
import streamlit as st

from resume_search.ingest import pipeline
from resume_search.ingest.jobs import JobStatus

st.set_page_config(page_title="Ingest", layout="wide")
st.title("Ingest")

db_path = common.db_path()
ss = st.session_state

st.markdown(
    "Job states: `queued` -> `extracting` -> `parsing` -> `deriving` -> `indexing` -> `ok`. "
    "Any running state can end in `failed`. A `queued` job ends as `skipped` when the same file "
    "and prompt version already parsed ok."
)
st.caption("States: " + ", ".join(s.value for s in JobStatus))


def _run(holder: dict, paths: list[str], force: bool, workers: int) -> None:
    try:
        holder["results"] = pipeline.ingest(paths, db_path, force=force, workers=workers)
    except Exception as exc:  # noqa: BLE001  reported on the page
        holder["error"] = f"{type(exc).__name__}: {exc}"


def start(paths: list[str], force: bool, workers: int) -> None:
    holder: dict = {"results": None, "error": None}
    holder["thread"] = threading.Thread(
        target=_run, args=(holder, paths, force, workers), daemon=True
    )
    holder["thread"].start()
    ss["_ingest"] = holder


holder = ss.get("_ingest")
running = holder is not None and holder["thread"].is_alive()

uploads = st.file_uploader("Resumes", type=["docx", "pdf"], accept_multiple_files=True)
c1, c2 = st.columns(2)
force = c1.checkbox("Force re-parse (ignore the file hash skip)")
workers = c2.slider("Workers", 1, 8, 4)
if st.button("Ingest", type="primary", disabled=running or not uploads):
    folder = common.uploads_dir()
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for f in uploads:
        p = folder / Path(f.name).name
        p.write_bytes(f.getvalue())
        paths.append(str(p))
    start(paths, force, workers)
    st.rerun()


def failed_files() -> list[str]:
    rows = common.conn().execute(
        "SELECT DISTINCT source_file FROM parse_jobs WHERE status = 'failed' AND file_sha256 NOT IN "
        "(SELECT file_sha256 FROM parse_jobs WHERE status = 'ok')"
    ).fetchall()
    found = []
    for r in rows:
        for folder in (common.uploads_dir(), Path("data")):
            if (folder / r["source_file"]).is_file():
                found.append(str(folder / r["source_file"]))
                break
    return found


c1, c2 = st.columns(2)
retry = failed_files()
if c1.button("Re-run failed", disabled=running or not retry, help=f"{len(retry)} files found on disk"):
    start(retry, True, workers)
    st.rerun()
if c2.button("Rederive all (no API)", disabled=running):
    with st.spinner("Rederiving"):
        n = pipeline.rederive(db_path)
    common.clear_caches()
    st.success(f"Rederived {n} candidates")


def jobs_frame() -> pd.DataFrame:
    return pd.read_sql(
        "SELECT job_id, source_file, status, input_tokens, output_tokens, cost_usd, latency_ms, "
        "error, queued_at, finished_at FROM parse_jobs ORDER BY job_id DESC", common.conn(),
    )


def metrics(df: pd.DataFrame) -> None:
    ok = df[df["status"] == "ok"]
    c = st.columns(4)
    c[0].metric("Parsed ok", len(ok))
    c[1].metric("Failed", int((df["status"] == "failed").sum()))
    c[2].metric("Cumulative cost (USD)", f"{df['cost_usd'].fillna(0).sum():.4f}")
    rate = "n/a"
    if len(ok):
        t0 = pd.to_datetime(ok["queued_at"]).min()
        t1 = pd.to_datetime(ok["finished_at"]).max()
        minutes = (t1 - t0).total_seconds() / 60
        if minutes > 0:
            rate = f"{len(ok) / minutes:.1f}"
    c[3].metric("Throughput (resumes/min)", rate)
    if len(ok):
        st.caption(f"Mean latency per resume: {ok['latency_ms'].mean() / 1000:.1f}s")


def panel() -> None:
    df = jobs_frame()
    metrics(df)
    st.caption(f"Updated {datetime.now():%H:%M:%S}")
    st.dataframe(df, hide_index=True)
    if holder is not None and not holder["thread"].is_alive():
        st.rerun()


st.button("Refresh")
if running:
    st.info("Ingest running. The table refreshes every 2 seconds.")
    st.fragment(run_every=2)(panel)()
else:
    if holder is not None and not holder.get("handled"):
        holder["handled"] = True
        common.clear_caches()
    if holder is not None and holder["error"]:
        st.error(holder["error"])
    elif holder is not None and holder["results"] is not None:
        st.success(
            "Last run: " + ", ".join(f"{j.source_file} {j.status.value}" for j in holder["results"])
        )
    df = jobs_frame()
    metrics(df)
    st.dataframe(df, hide_index=True)

Overwriting app/pages/4_Ingest.py


#### `app/pages/5_Evals.py`

Shows the latest parsing and ranking results and the judge-human disagreements.

In [54]:
%%writefile app/pages/5_Evals.py
import json
from pathlib import Path

import common
import pandas as pd
import streamlit as st

from resume_search.eval.results import SUITES, latest_result

st.set_page_config(page_title="Evals", layout="wide")
st.title("Evals and quality")

RESULTS = Path("evals/results")

VARIANTS = {
    "full": "Full blend", "rule_only": "Rule only", "keyword_only": "Keyword only",
    "embedding_only": "Embedding only",
}


def run_caption(path: Path, doc: dict) -> str:
    return (
        f"{path.name}. Run {doc.get('run_at')}. Task model {doc.get('task_model')}, judge "
        f"{doc.get('judge_model')}, judge prompt {doc.get('prompt_version')}. Cost "
        f"${doc.get('cost_usd', 0):.4f}, tokens {doc.get('tokens')}."
    )


def show_parsing(doc: dict) -> None:
    m = doc["metrics"]
    cols = st.columns(4)
    cols[0].metric("Candidates", m["candidates"])
    cols[1].metric("Field accuracy", f"{m['accuracy']:.0%}")
    cols[2].metric("Soft accuracy", f"{m['soft_accuracy']:.0%}")
    cols[3].metric("Hallucinated / missing", f"{m['hallucinated']} / {m['missing']}")
    st.markdown("**Per-field accuracy.** Accuracy is correct over total; soft counts partial as half.")
    st.dataframe(
        pd.DataFrame(m["per_field"]).T.rename_axis("field").reset_index(),
        hide_index=True, column_config={
            "accuracy": st.column_config.ProgressColumn("accuracy", min_value=0, max_value=1),
            "soft_accuracy": st.column_config.ProgressColumn("soft_accuracy", min_value=0, max_value=1),
        },
    )
    ag = m.get("agreement") or {}
    if ag.get("overall"):
        o = ag["overall"]
        st.markdown(
            f"**Judge versus human.** {o['n']} fields on {', '.join(ag['candidates'])} "
            f"(gold status: {', '.join(ag['gold_status'])}). Exact agreement {o['exact']:.0%}, "
            f"within one level {o['within_one_level']:.0%}."
        )
        st.dataframe(
            pd.DataFrame(
                [{"candidate": n, **{k: v for k, v in a.items() if k != "disagreements"}}
                 for n, a in ag["per_candidate"].items()]
            ),
            hide_index=True,
        )
        rows = [{"candidate": n, **d} for n, a in ag["per_candidate"].items() for d in a["disagreements"]]
        if rows:
            st.markdown("**Disagreements.**")
            st.dataframe(pd.DataFrame(rows), hide_index=True)
    else:
        st.info("No human grades found for these candidates (evals/gold/parsing_human.jsonl).")
    st.markdown("**Per candidate.** The judge's grade and reason for every field.")
    for item in sorted(doc["items"], key=lambda i: i["accuracy"]):
        label = f"{item['name']}: {item['accuracy']:.0%} correct, {item['soft_accuracy']:.0%} soft"
        with st.expander(label):
            st.dataframe(pd.DataFrame(item["grades"]), hide_index=True)


def show_ranking(doc: dict) -> None:
    m = doc["metrics"]
    items = doc["items"]
    st.markdown("**nDCG@5 and P@3 by requisition.** Same judged pool and grades for every variant.")
    rows = [
        {"requisition": i["title"], "pool": i["pool_size"], "relevant": i["relevant_in_pool"],
         **{f"{label} nDCG@5": i[f"{v}_ndcg5"] for v, label in VARIANTS.items()},
         **{f"{label} P@3": i[f"{v}_p3"] for v, label in VARIANTS.items()}}
        for i in items
    ]
    rows.append({
        "requisition": "Mean", "pool": None, "relevant": None,
        **{f"{label} nDCG@5": m["mean"][v]["ndcg5"] for v, label in VARIANTS.items()},
        **{f"{label} P@3": m["mean"][v]["p3"] for v, label in VARIANTS.items()},
    })
    st.dataframe(pd.DataFrame(rows), hide_index=True)
    st.caption(m.get("tie_break", ""))
    for i in items:
        with st.expander(f"{i['title']}: pool {i['pool_size']}, full nDCG@5 {i['full_ndcg5']:.2f}"):
            st.dataframe(
                pd.DataFrame(sorted(i["candidates"], key=lambda c: c["rank_full"])), hide_index=True
            )


SHOW = {"parsing": show_parsing, "ranking": show_ranking}

st.header("Eval results")
found = {s: latest_result(RESULTS, s) for s in SUITES}
if not any(found.values()):
    st.info(
        "No eval results yet. Run `uv run resume-search eval run --suite all`; it writes "
        "evals/results/<suite>-<timestamp>.json and this page shows the latest file per suite."
    )
for suite, hit in found.items():
    if hit is None:
        continue
    path, doc = hit
    st.subheader(suite.capitalize())
    st.caption(run_caption(path, doc))
    try:
        SHOW[suite](doc)
    except (KeyError, TypeError) as exc:
        st.error(f"{path.name} does not match the expected format: {exc!r}")
    with st.expander("Raw JSON"):
        st.json(doc)

st.header("Per-candidate parse quality")
rows = []
for r in common.conn().execute("SELECT candidate_id, name, record_json FROM candidates ORDER BY name"):
    meta = json.loads(r["record_json"])["meta"]
    ver = meta.get("verification") or {}
    tq = meta.get("text_quality") or {}
    rows.append({
        "candidate": r["name"], "parse_status": meta.get("parse_status"),
        "checks_run": ver.get("checks_run"), "disagreements": ver.get("disagreements"),
        "regenerated": ver.get("regenerated"), "unresolved": len(ver.get("unresolved", [])),
        "unresolved_items": "; ".join(str(u) for u in ver.get("unresolved", [])),
        "unknown_ratio_before": tq.get("unknown_ratio_before"),
        "unknown_ratio_after": tq.get("unknown_ratio_after"), "repairs": tq.get("repairs"),
        "chars_per_page": tq.get("chars_per_page"),
        "image_page_coverage": tq.get("max_image_page_coverage"),
        "extraction": meta.get("extraction_path"),
    })
if rows:
    df = pd.DataFrame(rows)
    st.metric("Candidates with unresolved verification items", int((df["unresolved"] > 0).sum()))
    st.dataframe(df, hide_index=True)
else:
    st.info("No candidates yet.")

Overwriting app/pages/5_Evals.py


### Evaluation requisitions

#### `evals/requisitions.jsonl`

In [55]:
%%writefile evals/requisitions.jsonl
{"id": "us-fundamental-healthcare", "title": "US fundamental healthcare analyst", "free_text": "Fundamental long/short equity fund seeks a healthcare analyst covering US pharmaceuticals, biotech and medtech. Builds financial models, writes differentiated stock ideas and has sell-side or buy-side coverage experience in the sector.", "location": {"value": "US", "mode": "preferred"}, "location_scope": "either", "approach": {"value": "Fundamental", "mode": "preferred"}, "sectors": {"value": ["Healthcare"], "mode": "must"}, "asset_classes": {"value": ["Equities"], "mode": "preferred"}, "years_min": 5, "years_max": 15, "years_mode": "preferred", "skills_preferred": ["financial modeling", "dcf"]}
{"id": "apac-systematic-equities", "title": "APAC systematic equities researcher", "free_text": "Systematic equity group is hiring a quantitative researcher for Asia-Pacific equities. Builds and backtests multi-factor models, works in Python, and has covered Greater China or other Asian markets.", "location": {"value": "APAC", "mode": "preferred"}, "location_scope": "worked", "approach": {"value": "Systematic", "mode": "preferred"}, "asset_classes": {"value": ["Equities"], "mode": "must"}, "markets_covered": {"value": ["APAC"], "mode": "preferred"}, "years_min": 1, "years_max": 8, "years_mode": "preferred", "skills_preferred": ["python", "backtesting", "machine learning"]}
{"id": "europe-credit", "title": "Europe credit analyst", "free_text": "Credit hedge fund is looking for an analyst covering European high yield and investment grade credit. Strong fundamental credit analysis, capital structure work and experience in London or another European center.", "location": {"value": "Europe", "mode": "must"}, "location_scope": "worked", "approach": {"value": "Fundamental", "mode": "preferred"}, "sectors": {"value": ["Credit"], "mode": "preferred"}, "asset_classes": {"value": ["Credit", "Fixed Income"], "mode": "preferred"}, "years_min": 4, "years_max": 12, "years_mode": "preferred", "skills_preferred": ["credit analysis", "financial modeling"]}
{"id": "india-sellside-healthcare", "title": "India sell-side healthcare analyst", "free_text": "Broker seeks a senior healthcare equity research analyst for Indian pharmaceuticals and healthcare services. Publishes ratings and price targets, has institutional investor relationships and covers listed Indian companies.", "location": {"value": "IN", "mode": "preferred"}, "location_scope": "either", "approach": {"value": "Fundamental", "mode": "preferred"}, "sectors": {"value": ["Healthcare"], "mode": "preferred"}, "markets_covered": {"value": ["IN"], "mode": "must"}, "seniority_bands": {"value": ["Senior", "Principal"], "mode": "preferred"}, "years_min": 6, "years_mode": "preferred", "skills_preferred": ["equity research", "financial modeling"]}
{"id": "junior-quant-developer", "title": "Junior quantitative developer", "free_text": "Derivatives desk needs a junior quantitative developer to build and maintain a C++ pricing library. Implements pricing models and payoffs, runs Monte Carlo and Greeks computation, and is comfortable with Python. Masters in a quantitative field.", "location": {"value": "Europe", "mode": "preferred"}, "location_scope": "either", "degree_min": {"value": "Masters", "mode": "must"}, "asset_classes": {"value": ["Derivatives"], "mode": "preferred"}, "years_min": 0, "years_max": 4, "years_mode": "preferred", "skills_preferred": ["c++", "python", "derivatives pricing"]}
{"id": "senior-generalist-ls", "title": "Senior generalist long/short equity analyst", "free_text": "Multi-sector long/short equity fund seeks a senior generalist analyst based in the US. Eight or more years of experience covering several sectors, building models and pitching long and short ideas to the portfolio manager.", "location": {"value": "US", "mode": "must"}, "location_scope": "either", "approach": {"value": "Fundamental", "mode": "preferred"}, "asset_classes": {"value": ["Equities"], "mode": "preferred"}, "seniority_bands": {"value": ["Senior", "Principal"], "mode": "preferred"}, "years_min": 8, "years_mode": "must", "skills_preferred": ["financial modeling", "dcf", "valuation"]}

Overwriting evals/requisitions.jsonl


## Install the package

The cell below installs the package you just wrote out. It prints the command by default so that a run of this notebook does not change your environment. Set `INSTALL=1` to run it.

In [56]:
import os
import subprocess

CMD = ["uv", "sync", "--extra", "dev"]   # with pip instead: pip install -e ".[dev]"
if os.environ.get("INSTALL") == "1":
    subprocess.run(CMD, check=True)
else:
    print("not installing; set INSTALL=1 to run:", " ".join(CMD))

not installing; set INSTALL=1 to run: uv sync --extra dev


## Demos

The cells below use the sample resumes in `data/`, the stored parse results in `exports/parsed/`, and the SQLite database `exports/candidates.sqlite`. None of them calls the API. If the database file is missing (it is git-ignored), the first demo cell rebuilds it from `exports/candidates.json` and the resume files, using only local code and the local embedding model.

In [57]:
import json
import os

os.environ.setdefault("TQDM_DISABLE", "1")  # keep model-loading progress bars out of the stored output
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd()
DATA = ROOT / "data"
DB = ROOT / "exports" / "candidates.sqlite"


def rebuild_db_from_export(db_path: Path = DB) -> int:
    """Rebuild the SQLite file from exports/candidates.json and the resume files.
    Uses the stored records, local code, and the local embedding model. No API call."""
    from resume_search.derive.models import CandidateRecord
    from resume_search.extract import extract
    from resume_search.search.vectors import candidate_vectors, default_embedder
    from resume_search.store.db import connect, load_taxonomy, upsert_candidate
    from resume_search.store.fts import fts_body

    records = [
        CandidateRecord.model_validate(r)
        for r in json.loads((ROOT / "exports" / "candidates.json").read_text(encoding="utf-8"))
    ]
    conn = connect(db_path)
    load_taxonomy(conn)
    for r in records:
        body = fts_body(extract(DATA / r.meta["source_file"]).text)
        upsert_candidate(conn, r, body, candidate_vectors(r, default_embedder()))
    conn.close()
    return len(records)


if not DB.exists():
    print(f"{DB.name} missing; rebuilt {rebuild_db_from_export()} candidates from exports/candidates.json")
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row
print("candidates in database:", conn.execute("SELECT COUNT(*) FROM candidates").fetchone()[0])

candidates in database: 10


### Extraction

One DOCX and one PDF: the first 20 lines of extracted text and the text-quality numbers. The PDF is Omar El-Hassan's, which has ligature glyphs.

In [58]:
from resume_search.extract import extract

for name in ["Chen Li (Alex).docx", "Omar El-Hassan 202405.pdf"]:
    e = extract(DATA / name)
    print(f"=== {e.source_file}  ({e.format.value}, {e.page_count} page(s), {len(e.text)} chars, path={e.extraction_path.value})")
    print("\n".join(e.text.splitlines()[:20]))
    print()
    print(pd.Series(vars(e.quality), name="text_quality").to_frame().T.to_string(index=False))
    print()

=== Chen Li (Alex).docx  (docx, 1 page(s), 4283 chars, path=text)
Chen Li (Alex)
Mobile: +852 65478923
Email: Alex_chen2024@gmail.com
EDUCATION

University of Hong Kong Hong Kong
Master of Science in Financial Technology Sept.2021 – Jul.2023
Related Courses: Algorithmic Trading, Quantitative Risk Management, Machine Learning in Trading and Finance
University of Hong Kong Hong Kong
Bachelor of Business Administration (Finance) Sept.2017 – Jul.2021
Related Courses: Econometrics, Statistics
WORKING EXPERIENCE

Bank of China Hong Kong
Investment Analyst June.2021 – Sep.2023
 Conduct comprehensive quantitative analysis on large-scale Asian equity datasets to provide statistical support for portfolio managers. Analyze market trends, sector rotations, and statistical anomalies, delivering actionable insights for investment decision-making and strategy optimization.
Independently developed, constructed and actively managed a multi-factor quantitative model focusing on Greater China technology 

#### Ligature repair

The PDF font encodes `ti` as `Ɵ` (U+019F), so the raw text has words such as `QuanƟtaƟve`. Repair and a fuzzy fix bring the unknown-word ratio down from 17.8% to 2.6%.

In [59]:
from resume_search.extract.pdf import extract_pdf
from resume_search.extract.repair import repair_text

raw, *_ = extract_pdf(DATA / "Omar El-Hassan 202405.pdf")
fixed, repairs = repair_text(raw)
e = extract(DATA / "Omar El-Hassan 202405.pdf")
q = e.quality
print(f"unknown-word ratio  before repair: {q.unknown_ratio_before:.1%}   after repair and fuzzy fix: {q.unknown_ratio_after:.1%}")
print(f"glyph repairs applied: {q.repairs}")
bad = [line for line in raw.splitlines() if "\u019f" in line][:3]
for line in bad:
    print("raw  :", line)
    print("fixed:", repair_text(line)[0])

unknown-word ratio  before repair: 17.8%   after repair and fuzzy fix: 2.6%
glyph repairs applied: 42
raw  : QuanƟtaƟve Developer ― BNP Paribas CIB Paris, France 
fixed: Quantitative Developer ― BNP Paribas CIB Paris, France
raw  : R&D Front Oﬃce Fixed Income team, quanƟtaƟve development in the pricing 
fixed: R&D Front Office Fixed Income team, quantitative development in the pricing
raw  : IntroducƟon of a new model (one factor Gaussian copula model) 
fixed: Introduction of a new model (one factor Gaussian copula model)


### A stored record

The parse result for Chen Li, read from `exports/parsed/`. This is what the LLM returned after validation, and the verification block that compared it with deterministic text parsing.

In [60]:
parsed = {}
for f in (ROOT / "exports" / "parsed").glob("*.json"):
    doc = json.loads(f.read_text(encoding="utf-8"))
    parsed[doc["meta"]["source_file"]] = doc
chen = parsed["Chen Li (Alex).docx"]

print("--- extracted record (LLM output, validated) ---")
print(json.dumps(chen["extracted"], indent=2, ensure_ascii=False))

--- extracted record (LLM output, validated) ---
{
  "name": "Chen Li",
  "contact": {
    "email": "Alex_chen2024@gmail.com",
    "phone": "+852 65478923",
    "location_text": null
  },
  "education": [
    {
      "institution": "University of Hong Kong",
      "degree": "Master of Science in Financial Technology",
      "degree_level": "Masters",
      "field": "Financial Technology",
      "start": "2021-09",
      "end": "2023-07",
      "country": "HK"
    },
    {
      "institution": "University of Hong Kong",
      "degree": "Bachelor of Business Administration (Finance)",
      "degree_level": "Bachelors",
      "field": "Finance",
      "start": "2017-09",
      "end": "2021-07",
      "country": "HK"
    }
  ],
  "skills": {
    "programming": [
      "Python",
      "R",
      "SQL",
      "VBA"
    ],
    "tools": [
      "Microsoft Office Suite",
      "Excel",
      "PowerPoint",
      "Word",
      "Bloomberg Terminal",
      "WIND Database",
      "GIS mapping tools"

In [61]:
v = chen["meta"]["verification"]
print("--- verification block ---")
print(f"checks run: {v['checks_run']}   disagreements: {v['disagreements']}   "
      f"regenerated: {v['regenerated']}   unresolved: {len(v['unresolved'])}")
for d in v["details"]:
    print(f"\nfield: {d['field']}")
    print("  LLM value :", json.dumps(d["llm_value"], ensure_ascii=False)[:300], "...")
    print("  text value:", d["text_value"])
print()
print(pd.DataFrame(chen["meta"]["attempts"])[["call", "model", "latency_ms", "input_tokens", "output_tokens", "cache_read_tokens", "error"]])

--- verification block ---
checks run: 21   disagreements: 1   regenerated: 1   unresolved: 0

field: skills
  LLM value : {"programming": ["Python", "R", "SQL", "VBA"], "tools": ["Microsoft Office Suite", "Excel", "PowerPoint", "Word", "Bloomberg Terminal", "WIND Database", "GIS mapping tools"], "domain": ["Algorithmic Trading", "Quantitative Risk Management", "Machine Learning in Trading and Finance", "Econometrics",  ...
  text value: ['backtesting', 'bloomberg', 'dcf', 'econometrics', 'factor-models', 'machine-learning', 'performance-attribution', 'powerpoint', 'primary-research', 'python', 'r', 'risk-management', 'sql', 'valuation', 'vba', 'wind']

           call              model  latency_ms  input_tokens  output_tokens  cache_read_tokens error
0       profile  claude-sonnet-5-5       11205          1581            451               2383  None
1    experience  claude-sonnet-5-5       17970          1549           3398               2851  None
2  regen:skills  claude-sonnet-5-5 

### The derived table

One row per candidate, from the database. Years, band, country, and CFA status are computed in code from the extracted record, never asked of the LLM.

In [62]:
derived = pd.read_sql(
    """
    SELECT name, current_country AS country, current_region AS region, seniority_band AS band,
           ROUND(months_full_time_relevant / 12.0, 1) AS yrs_relevant,
           ROUND(months_full_time_core / 12.0, 1) AS yrs_core,
           investment_approach AS approach, highest_degree AS degree, has_cfa AS cfa, parse_status
    FROM candidates ORDER BY name
    """,
    conn,
)
facets = {
    "sectors": ("candidate_sectors", "sector"),
    "assets": ("candidate_assets", "asset_class"),
    "worked": ("candidate_countries", "country"),
}
for col, (table, field) in facets.items():
    extra = " AND kind = 'worked'" if col == "worked" else ""
    s = pd.read_sql(
        f"SELECT c.name, GROUP_CONCAT(DISTINCT t.{field}) AS v FROM candidates c "
        f"JOIN {table} t USING (candidate_id) WHERE 1=1{extra} GROUP BY c.name", conn,
    ).set_index("name")["v"]
    derived[col] = derived["name"].map(s)
derived

,name,country,region,band,yrs_relevant,yrs_core,approach,degree,cfa,parse_status,sectors,assets,worked
0,Chen Li,HK,APAC,Mid,2.3,2.3,Systematic,Masters,0,ok,"Technology,Healthcare","Equities,Fixed Income",HK
1,Marcus Chen-Rodriguez,US,US,Principal,11.1,11.1,Fundamental,Bachelors,0,ok,Healthcare,"Equities,Credit",US
2,Marina Silva Costa,GB,Europe,Senior,8.2,5.2,Fundamental,MBA,0,ok,"Healthcare,Consumer,Technology,Industrials",Equities,"GB,BR"
3,Michael Rodriguez,US,US,Senior,8.8,8.8,Fundamental,Bachelors,1,ok,"Technology,Credit,Consumer","Equities,Fixed Income,Credit,Derivatives",US
4,Omar El-Hassan,FR,Europe,Mid,4.4,4.4,Systematic,Masters,0,ok,NaN,"Fixed Income,Derivatives",FR
5,Priya Nakamura,IN,APAC,Principal,12.8,12.8,Fundamental,MBA,0,ok,"Healthcare,Other,Industrials",Equities,IN
6,Ryan Patel,US,US,Senior,8.5,6.5,Fundamental,Bachelors,0,ok,"Consumer,Technology,Healthcare","Equities,Fixed Income",US
7,Vikram Shah,US,US,Principal,12.2,9.3,Fundamental,Bachelors,1,ok,Technology,Equities,US
8,Viktor Sharat,NaN,NaN,Senior,9.8,9.8,Fundamental,Masters,0,ok,"Healthcare,Other,Energy,Industrials,Consumer,Technology",Equities,NaN
9,Zara Al-Rashid,IN,APAC,Principal,11.9,11.9,Fundamental,Masters,0,ok,Healthcare,Equities,"IN,GB"


### Search

Three requisitions built by hand, with no JD parsing and no API call. Columns: `final` is the blend, `rule` the preferred-criteria score, `kw` the BM25 score, `emb` the embedding score, each on 0 to 1 within the pruned set. `+` and `-` mark chips that matched and did not. A requisition with no free text drops `kw` and `emb`. When a requisition has only must criteria, as in the third search, every survivor scores 1.0 and the tie is broken by name.

In [63]:
from resume_search.search import Criterion, Requisition, Weights, search

weights = Weights.load()
SEARCHES = {
    "Fundamental healthcare in the US": Requisition(
        free_text="fundamental healthcare equity analyst, US pharmaceuticals and biotech",
        location=Criterion[str](value="US"),
        approach=Criterion[str](value="Fundamental"),
        sectors=Criterion[list[str]](value=["Healthcare"]),
    ),
    "Systematic, 5 years or less, Python preferred": Requisition(
        free_text="systematic quantitative researcher, backtesting, factor models",
        approach=Criterion[str](value="Systematic"),
        years_max=5,
        skills_preferred=["python"],
    ),
    "Senior, CFA required": Requisition(
        seniority_bands=Criterion[list[str]](value=["Senior", "Principal"], mode="must"),
        certifications=Criterion[list[str]](value=["CFA"], mode="must"),
    ),
}
for title, req in SEARCHES.items():
    results = search(conn, req, weights, top_k=5)
    print(f"=== {title}  ({len(results)} after pruning)")
    for i, r in enumerate(results, 1):
        b = r.breakdown
        chips = "  ".join(("+" if c.status.value == "matched" else "-") + c.label for c in r.chips)
        print(f"{i}. {r.name:<22} final={b.final:.3f} rule={b.rule:.3f} kw={b.keyword:.3f} emb={b.embedding:.3f}   {chips}")
    print()

/Users/lizhezhao/Projects/case-study-millennium/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1025 12:56:55.497000 6562 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


=== Fundamental healthcare in the US  (5 after pruning)
1. Priya Nakamura         final=0.875 rule=0.750 kw=1.000 emb=1.000   -Location US  +Approach Fundamental  +Healthcare
2. Zara Al-Rashid         final=0.832 rule=0.750 kw=0.950 emb=0.890   -Location US  +Approach Fundamental  +Healthcare
3. Marcus Chen-Rodriguez  final=0.776 rule=1.000 kw=0.000 emb=0.921   +Location US  +Approach Fundamental  +Healthcare
4. Viktor Sharat          final=0.709 rule=0.750 kw=0.605 emb=0.710   -Location US  +Approach Fundamental  +Healthcare
5. Ryan Patel             final=0.678 rule=1.000 kw=0.000 emb=0.595   +Location US  +Approach Fundamental  +Healthcare

=== Systematic, 5 years or less, Python preferred  (2 after pruning)
1. Chen Li                final=0.800 rule=1.000 kw=0.000 emb=1.000   +Approach Systematic  +Years up to 5y  +python
2. Omar El-Hassan         final=0.700 rule=1.000 kw=1.000 emb=0.000   +Approach Systematic  +Years up to 5y  +python

=== Senior, CFA required  (3 after pruning)


### Insights

Two of the Insights page's charts. The tables print the same numbers, so the cell is readable where plots do not render.

In [64]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"  # JSON payload only: no 3 MB plotly.js bundle per figure


def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)


by_region = q("SELECT COALESCE(current_region, '?') AS region, investment_approach AS approach, COUNT(*) AS n "
              "FROM candidates GROUP BY 1, 2")
skills = q("SELECT skill, COUNT(DISTINCT candidate_id) AS n FROM candidate_skills "
           "GROUP BY 1 ORDER BY n DESC, skill LIMIT 12")
print("Candidates by region and approach")
print(by_region.pivot(index="region", columns="approach", values="n").fillna(0).astype(int))
print("\nTop skills (candidates)")
print(skills.to_string(index=False))
px.bar(by_region, x="region", y="n", color="approach", title="Candidates by region and approach").show()
fig = px.bar(skills, x="n", y="skill", orientation="h", title="Top skills")
fig.update_yaxes(autorange="reversed")
fig.show()

Candidates by region and approach
approach  Fundamental  Systematic
region                           
?                   1           0
APAC                2           1
Europe              1           1
US                  4           0

Top skills (candidates)
             skill  n
financial-modeling  5
         bloomberg  4
             excel  4
               dcf  3
   equity-research  3
           factset  3
  machine-learning  3
            python  3
                 r  3
               sql  3
       backtesting  2
     due-diligence  2


### Evaluation metrics

Read from the latest files in `evals/results/`. Parsing first: per-field accuracy from the Opus judge.

In [65]:
from resume_search.eval.results import latest_result

p_path, parsing = latest_result("evals/results", "parsing")
r_path, ranking = latest_result("evals/results", "ranking")
pm = parsing["metrics"]
print(f"parsing: {p_path.name}  judge={parsing['judge_model']}  task={parsing['task_model']}  "
      f"fields={pm['fields']}  accuracy={pm['accuracy']:.1%}  soft={pm['soft_accuracy']:.1%}  "
      f"hallucinated={pm['hallucinated']}  missing={pm['missing']}  cost=${parsing['cost_usd']:.2f}")
per_field = pd.DataFrame(pm["per_field"]).T[["total", "correct", "partial", "wrong", "hallucinated", "missing", "accuracy"]]
per_field.round(3)

parsing: parsing-20261025T154356Z.json  judge=claude-opus-5-5  task=claude-sonnet-5-5  fields=143  accuracy=86.7%  soft=93.4%  hallucinated=0  missing=0  cost=$0.66


,total,correct,partial,wrong,hallucinated,missing,accuracy
name,10.0,10.0,0.0,0.0,0.0,0.0,1.000
overall,143.0,124.0,19.0,0.0,0.0,0.0,0.867
contact,10.0,10.0,0.0,0.0,0.0,0.0,1.000
education,10.0,9.0,1.0,0.0,0.0,0.0,0.900
experience,10.0,8.0,2.0,0.0,0.0,0.0,0.800
skills,10.0,8.0,2.0,0.0,0.0,0.0,0.800
certifications,10.0,10.0,0.0,0.0,0.0,0.0,1.000
languages,10.0,10.0,0.0,0.0,0.0,0.0,1.000
investment_approach,10.0,9.0,1.0,0.0,0.0,0.0,0.900
search_summary,10.0,4.0,6.0,0.0,0.0,0.0,0.400


Judge against the human grades, on the three hand-graded resumes.

In [66]:
ag = pm["agreement"]
print(f"judge vs human, gold status {ag['gold_status']}: {ag['overall']['n']} fields, "
      f"exact {ag['overall']['exact']:.1%}, within one level {ag['overall']['within_one_level']:.1%}")
pd.DataFrame(ag["per_candidate"]).T[["n", "exact", "within_one_level"]]

judge vs human, gold status ['draft']: 42 fields, exact 92.9%, within one level 100.0%


,n,exact,within_one_level
Chen Li,15,1.0,1.0
Omar El-Hassan,12,0.833333,1.0
Zara Al-Rashid,15,0.933333,1.0


Ranking: nDCG@5 and P@3 per requisition for the full blend and the three single-signal ablations.

In [67]:
rm = ranking["metrics"]
print(f"ranking: {r_path.name}  judge={ranking['judge_model']}  pools={rm['pool_sizes']}")
cols = [f"{v}_{k}" for v in ["full", "rule_only", "keyword_only", "embedding_only"] for k in ["ndcg5", "p3"]]
table = pd.DataFrame(ranking["items"]).set_index("requisition_id")[["pool_size", *cols]]
table.loc["MEAN"] = [None] + [rm["mean"][c.rsplit("_", 1)[0]][c.rsplit("_", 1)[1]] for c in cols]
table.round(3)

ranking: ranking-20261025T154236Z.json  judge=claude-opus-5-5  pools={'us-fundamental-healthcare': 7, 'apac-systematic-equities': 9, 'europe-credit': 3, 'india-sellside-healthcare': 3, 'junior-quant-developer': 6, 'senior-generalist-ls': 4}


,pool_size,full_ndcg5,full_p3,rule_only_ndcg5,rule_only_p3,keyword_only_ndcg5,keyword_only_p3,embedding_only_ndcg5,embedding_only_p3
requisition_id,,,,,,,,,
us-fundamental-healthcare,7.0,0.906,1.000,0.906,1.000,0.856,1.000,0.873,1.000
apac-systematic-equities,9.0,0.907,0.333,0.924,0.333,0.950,0.667,0.907,0.333
europe-credit,3.0,1.000,0.667,0.920,0.667,1.000,0.667,0.920,0.667
india-sellside-healthcare,3.0,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
junior-quant-developer,6.0,1.000,0.333,1.000,0.333,1.000,0.333,1.000,0.333
senior-generalist-ls,4.0,1.000,1.000,1.000,1.000,1.000,1.000,0.860,1.000
MEAN,NaN,0.969,0.722,0.958,0.722,0.968,0.778,0.927,0.722


### Live LLM demos

These two cells call the Anthropic API and cost money. They run only when the environment variable `RUN_LLM` is `1`. Otherwise they print what they would do.

#### Parse one resume end to end

In [68]:
if os.environ.get("RUN_LLM") == "1":
    from resume_search.extract import extract
    from resume_search.ingest.pipeline import attempt_cost
    from resume_search.parse import ParseOk, parse_resume

    result = parse_resume(extract(DATA / "Chen Li (Alex).docx"))
    attempts = pd.DataFrame([vars(a) for a in result.attempts])
    attempts["cost_usd"] = [attempt_cost(a) for a in result.attempts]
    print("status:", result.parse_status)
    print(attempts[["call", "model", "latency_ms", "input_tokens", "output_tokens", "cache_read_tokens", "error", "cost_usd"]])
    print(f"total cost ${attempts['cost_usd'].sum():.4f}")
    if isinstance(result, ParseOk):
        v = result.verification
        print(f"verification: checks={v.checks_run} disagreements={v.disagreements} regenerated={v.regenerated} unresolved={len(v.unresolved)}")
        print(result.record.search_summary)
else:
    print("set RUN_LLM=1 to run (about $0.08)")

set RUN_LLM=1 to run (about $0.08)


#### Parse a job description into a requisition

In [69]:
if os.environ.get("RUN_LLM") == "1":
    from resume_search.ingest.pipeline import attempt_cost
    from resume_search.search import parse_requisition

    jd = json.loads((ROOT / "evals" / "requisitions.jsonl").read_text(encoding="utf-8").splitlines()[0])["free_text"]
    attempts = []
    req = parse_requisition(jd, attempts=attempts)
    print(req.model_dump_json(indent=2, exclude_none=True))
    print(f"api calls={len(attempts)}  cost=${sum(attempt_cost(a) for a in attempts):.4f}")
else:
    print("set RUN_LLM=1 to run (about $0.01)")

set RUN_LLM=1 to run (about $0.01)


## Results

Numbers come from the latest files in `evals/results/`. The judge is Claude Opus 5.5 and the task model is Claude Sonnet 5.5. The tables above are read from those files, so they match what follows.

**Parsing.** The judge graded 143 fields across the 10 resumes. 86.7% were `correct` and 93.4% were `correct` or `partial`. It found 0 hallucinated and 0 missing fields. `name`, `contact`, `certifications`, and `languages` scored 100%. `search_summary` scored 40%: the judge marks a summary `partial` when it attaches a skill or figure to the wrong role, which is a real fault in the summary but not a wrong fact about the candidate. Judging cost $0.66 for the run.

**Judge against human.** Three resumes (Chen Li, Omar El-Hassan, Zara Al-Rashid) were graded by hand. On their 42 fields the judge matched the human grade exactly 92.9% of the time and within one grade level 100% of the time. The three disagreements are one `skills`, one `investment_approach`, and one `experience` field.

**Ranking.** Six requisitions, each graded 0/1/2 per candidate by the judge. Mean nDCG@5 / P@3 for the four variants:

| Variant | nDCG@5 | P@3 |
|---|---|---|
| full blend (0.5 rule, 0.2 BM25, 0.3 embedding) | 0.969 | 0.722 |
| rule only | 0.958 | 0.722 |
| BM25 only | 0.968 | 0.778 |
| embedding only | 0.927 | 0.722 |

**Cost.** The 10 parses cost $0.7535 in total, $0.075 per resume (three calls plus any targeted regeneration), and took 28.5 seconds per resume on average.

### Caveats

- **The human gold is a draft.** The three hand-graded resumes were graded by one person and the grades are marked `draft`. Judge-human agreement of 92.9% is agreement with that draft.
- **The pools are tiny.** Each requisition prunes the 10 candidates down to between 3 and 9 before ranking. In a pool of 3, nDCG@5 is close to 1 whatever the order is.
- **The judge varies between runs.** Two parsing runs on the same data scored 85.3% and 86.7%. The judge prompt also changed between them (`pj1` to `pj2`), so the gap mixes judge variance with a prompt change, and one run per setting is not enough to separate the two.
- **The blend cannot be tuned on this data.** The four variants differ by less than one swapped pair in a small pool, so the data cannot say whether 0.5/0.2/0.3 is better than any other split. The weights come from the design, not from the eval.
- **Ties sort alphabetically.** A variant that scores many candidates equally is ranked by name inside each tie, which can help or hurt a variant by chance.

## Scalability and additional features

This section covers what the code is built to support, what would change at volume, and what is not built. The reasons behind each choice are in `docs/design.md` (sections 3.4, 6, 7, 9, 10, and 13).

### Other input formats

Only DOCX and PDF are implemented. The route for each other format:

| Tier | Formats | Route | Notes |
|---|---|---|---|
| 1 | docx, pdf | python-docx, PyMuPDF | implemented |
| 1 | txt, md, html | read the file directly | not built, a few lines each |
| 2 | doc, rtf, odt, pages | LibreOffice headless to docx or pdf, then tier 1 | adds a system dependency |
| 3 | png, jpg, scanned pdf | vision model, or Tesseract OCR | OCR is cheaper at volume and worse on layout |
| 4 | LinkedIn JSON, ATS XML | map straight into `extracted` | skips the LLM entirely |

### Vision fallback

`extract` decides per file whether the text layer is usable. It switches to the vision path when a PDF page has fewer than 400 characters, when an image covers more than half a page, or when the unknown-word ratio after repair is above 10%. The vision path renders the pages to PNG at 150 dpi and sends them as image blocks with the same schemas. The path is written but none of the 10 sample files triggers it, so it has not been exercised on real input.

### Postgres and pgvector

The DDL in `store/schema.sql` moves to Postgres with three changes. FTS5 becomes a `tsvector` column with a GIN index. The `embeddings` table becomes a `pgvector` column with an HNSW index, which replaces the in-memory brute-force cosine in `VectorIndex`. The `parse_jobs` table becomes the work queue. SQLite is enough for this demo. The first limits at volume are the in-memory vector scan and the single writer (a guess from the code, not measured).

### Queue-based ingest and the `parse_jobs` contract

`ingest()` already treats each file as an independent job. A job has a `file_sha256`, a `prompt_version`, and a status that moves only forward through `queued`, `extracting`, `parsing`, `deriving`, `indexing`, and one of `ok`, `failed`, or `skipped`. The idempotency key is `(file_sha256, prompt_version)`: a second ingest of the same bytes under the same prompt is skipped. Each file is written in one transaction, so a crash leaves either the old candidate or the new one.

To scale out, the thread pool is replaced by a queue (SQS, Cloud Tasks, or a Postgres `SKIP LOCKED` table). Producers insert `queued` rows. Stateless workers claim a row, run extract and parse, and write the result. The state machine and the idempotency key do not change.

The arithmetic matters here. The measured latency is 28.5 seconds per resume. With the 4 workers `ingest` uses, that is about 8 resumes a minute. A bulk load of 100,000 resumes at that rate takes about 8 days, so it needs about 100 concurrent workers (about 8 hours) and an API rate limit sized for them. The design called for a token-bucket rate limiter on API calls. It is not built: the code relies on 4 workers and the SDK's retries (`max_retries=4`), and a `RateLimitError` that survives the retries is recorded as that attempt's error and moves the retry ladder to its next rung.

### Re-parse and re-derive are different operations

A **prompt-version bump** changes what the LLM returns, so every resume must be parsed again and costs API money. `ingest --force` does that, and a changed `PROMPT_VERSION` makes the skip check miss. A **schema-version bump** changes only how `derived` is computed from `extracted`, so `resume-search rederive` rebuilds the derived fields, the facet tables, and the vectors from the stored `record_json` with no API call. Splitting the layers is what makes the second case free.

### Term matching beyond the synonym dictionary

Skills are matched today by the synonym dictionary in `taxonomy/skills.yaml` plus a rapidfuzz fallback, and the cross-check in `parse/verify.py` compares the LLM's skills against the same dictionary. Two further layers are designed and not built:

- **spaCy EntityRuler cross-check.** Phrase patterns compiled from the taxonomy (skills, tools, firms, sectors, certifications, locations, phone prefixes) run over the raw text at ingest. The spans would highlight evidence on the Candidate page and give a second, independent skill and country guess. The LLM keeps priority for display and disagreements are counted into `quality`.
- **Knowledge-graph query expansion.** A networkx graph built from the taxonomy YAML (synonym to canonical, skill to category, firm to firm type, approach to related terms) would expand a query at search time, so "systematic" also finds "stat arb", "factor", and "quant".

### A small classifier as a cheap pre-filter

At volume it is wasteful to send every resume to an LLM to learn its investment approach. A TF-IDF plus logistic-regression classifier, trained on the LLM's own labels for `investment_approach` and sector, could label most resumes for almost nothing and send only low-confidence ones to the LLM. A confusion matrix against the LLM labels would show where it is safe. It is not built, and 10 resumes are too few to train one.

### PII

`email`, `phone`, and `location_text` live only in their own `candidates` columns. The FTS body is the resume text with email, URL, and phone lines removed, and the embedding inputs are the `search_summary` and experience bullets. A validator rejects a `search_summary` that contains an `@` or a run of seven or more digits. Two gaps remain. The full resume text, contact block included, goes to the Anthropic API during parsing, which is a data-processing decision to confirm before real candidates are loaded. And the read-time redaction switch in the design (mask the three columns for users who should not see them) is not built.

### Cost per resume at volume

The measured cost is $0.075 per resume (10 resumes, $0.7535 in total, at the per-token prices in `ingest/pipeline.py`). That is about $75 per 1,000 resumes and about $7,500 per 100,000 at list prices. Prompt caching is already on for the system prompts. The batch API, which is billed at a discount and suits an overnight bulk load, and the small-classifier pre-filter are the two levers not yet used. Judging costs extra: $0.66 for a parsing eval and $0.17 for a ranking eval.

### What nondeterminism means for re-ingest

An LLM does not return the same JSON twice. The skip check means a re-ingest of the same file under the same prompt version does not call the model, so the stored record stays stable. Nondeterminism enters only when a re-parse is forced or the prompt version changes. The eval harness is the control for that case. Run `resume-search eval run --suite parsing` before and after the change, compare per-field accuracy, and read the judge-human agreement to see whether the judge itself moved. A prompt change that lowers a field's accuracy or adds hallucinations shows up before it reaches the index.

### Known limitations

- **The JD years range is a hard filter.** A pasted JD that says "4-8 years" prunes candidates outside it (`years_mode` defaults to `must`). A candidate with 8.5 years never appears.
- **Ties sort alphabetically.** Equal final scores are ordered by candidate name, so the order inside a tie carries no information.
- **Country is null when the LLM leaves a role's location blank.** The role then does not count toward `current_country` or `countries_worked`, and location criteria miss the candidate.
- **A completed CFA Level III does not count as a charter.** `has_cfa` is true only for a charterholder or a bare "CFA" entry, so a candidate who passed Level III and has not yet been awarded the charter is excluded by a CFA filter.
- **The eval is small.** Ten resumes, six requisitions, pools of 3 to 9, and a draft gold set. It catches regressions. It does not rank two close designs.

## Repository layout

```
case-study-millennium/
  2025_ds_case_study_resume_platform.ipynb   this notebook (generated)
  README.md                                  quick start and reference
  pyproject.toml                             uv project, Python 3.12
  docs/design.md                             architecture record
  scripts/build_notebook.py                  generates and executes this notebook
  taxonomy/                                  YAML: countries, sectors, firm types, asset classes, skills, weights
  src/resume_search/
    extract/ parse/ derive/ store/ search/ ingest/ eval/
    taxonomy.py cli.py
  app/                                       Home.py and pages/1_Search.py ... 5_Evals.py
  evals/                                     requisitions.jsonl, gold/, results/
  exports/                                   candidates.json, candidates.csv, candidates.sqlite, parsed/, text/
  data/                                      the 10 sample resumes (input only)
  tests/                                     pytest suite
```

## How to run

```
uv sync --extra dev
echo "ANTHROPIC_API_KEY=..." > .env

uv run resume-search ingest data/*.docx data/*.pdf     # parse, derive, index, export
uv run resume-search search --location US --approach Fundamental --sector Healthcare
uv run resume-search show "Chen Li"
uv run resume-search eval run --suite all              # LLM-judged evals, about $0.83
uv run resume-search eval show

uv run streamlit run app/Home.py                       # the app
RUN_LLM=1 uv run python scripts/build_notebook.py --execute   # rebuild this notebook, with the LLM demos
```

`resume-search` subcommands: `extract`, `parse`, `ingest`, `rederive`, `show`, `search`, `eval run`, `eval show`. The README lists every flag.